# Colab Session: fast-hybrid-pro-37877728627
Generated from colab-cli history log.

**Session Created**: 2026-10-09 03:06:49
- Endpoint: `m-s-kkb-use4c1-31pp2oztl480d`
- Hardware: `NONE`

*File Operation*: `upload` on `/content/production_signal_journal.csv`

In [ ]:

# ============================================================
# FAST HYBRID PRO v2
# CLEAN RESEARCH / BACKTEST NOTEBOOK
# ============================================================

!pip -q install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("✅ Setup complete")

✅ Setup complete


In [ ]:
# ============================================================
# 2) CONFIGURATION
# ============================================================

CAPITAL = 100_000
MAX_RISK_PCT = 0.02
BOARD_LOT = 100
MIN_SCORE = 3
MIN_BARS = 200
WARMUP_BARS = 200
MAX_HOLDING_DAYS = 20

# ============================================================
# PRODUCTION STRATEGY — frozen parameters
# ============================================================
PRODUCTION_MIN_SCORE = 4
PRODUCTION_WEEKLY_CONFIRM = False
PRODUCTION_ATR_MULT = 1.2
PRODUCTION_TARGET2_PCT = 0.15
PRODUCTION_MAX_HOLDING_DAYS = 20

# ============================================================
# STOCK UNIVERSE FILTER — mandatory pre-scan gate
# Source: SET listed-company classification + market data
# ============================================================
UNIVERSE_MARKETS = {"SET", "mai"}

# User requirement: market cap > 5,000 MB
UNIVERSE_MIN_MARKET_CAP_MBAHT = 5_000

# User requirement: total trading value across the latest 20 trading days > 20 MB.
# This is a TOTAL 20-day value gate, not an average-per-day gate.
UNIVERSE_MIN_TOTAL20_VALUE_MBAHT = 20

# Fundamental price gate used in the original SeedGrowthTrading universe.
UNIVERSE_MIN_PRICE_BAHT = 1
UNIVERSE_MAX_PRICE_BAHT = 80

# Research/backtest MUST use the same fundamental universe gate as production.
RESEARCH_USE_FUNDAMENTAL_UNIVERSE_GATE = True

# Exclude SET/mai sectors corresponding to:
# banks, finance/hire-purchase, insurance, property/REITs, hotels/tourism.
UNIVERSE_EXCLUDED_SECTORS = {
    "BANK",
    "FIN",
    "INSUR",
    "PROP",
    "PF&REIT",
    "TOURISM",
}

# Only ordinary common stocks.
CLASSIFICATION_SOURCE_USED = None
UNIVERSE_SECURITY_TYPE = "CS"

# The old hand-picked production universe is deliberately removed.
# Production symbols are rebuilt from the filtered SET/mai universe.
PRODUCTION_SYMBOLS = []

WATCHLIST = []

print("Universe filters:")
print(f"  Markets: {sorted(UNIVERSE_MARKETS)}")
print(f"  Market cap > {UNIVERSE_MIN_MARKET_CAP_MBAHT:,} MB")
print(f"  Total 20-day trading value > {UNIVERSE_MIN_TOTAL20_VALUE_MBAHT:,} MB")
print(f"  Price: {UNIVERSE_MIN_PRICE_BAHT}–{UNIVERSE_MAX_PRICE_BAHT} THB")
print(f"  Excluded sectors: {sorted(UNIVERSE_EXCLUDED_SECTORS)}")

@dataclass
class StrategyConfig:
    macd_fast: int = 4
    macd_slow: int = 12
    macd_signal: int = 9
    stoch_k: int = 8
    stoch_smooth: int = 3
    stoch_d: int = 3
    ema_fast: int = 5
    ema_mid: int = 10
    ema_long: int = 75
    weekly_ema_fast: int = 5
    weekly_ema_slow: int = 10
    atr_length: int = 14
    atr_stop_mult: float = 1.5
    bb_length: int = 20
    bb_mult: float = 2.0
    kc_mult: float = 1.5
    squeeze_lookback: int = 10
    volume_ma: int = 5
    volume_ratio_min: float = 1.2
    target1_pct: float = 0.07
    target2_pct: float = 0.15
    min_score: int = 3

CANONICAL = StrategyConfig(macd_fast=4, macd_slow=12, macd_signal=9)
FAST_MACD = StrategyConfig(macd_fast=6, macd_slow=13, macd_signal=5)

print("CANONICAL:", asdict(CANONICAL))
print("FAST MACD:", asdict(FAST_MACD))


Universe filters:
  Markets: ['SET', 'mai']
  Market cap > 5,000 MB
  Total 20-day trading value > 20 MB
  Price: 1–80 THB
  Excluded sectors: ['BANK', 'FIN', 'INSUR', 'PF&REIT', 'PROP', 'TOURISM']
CANONICAL: {'macd_fast': 4, 'macd_slow': 12, 'macd_signal': 9, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct': 0.15, 'min_score': 3}
FAST MACD: {'macd_fast': 6, 'macd_slow': 13, 'macd_signal': 5, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct'

In [ ]:
# ============================================================
# 3) DATA ENGINE + SET/mai UNIVERSE FILTER
# ============================================================

import io
import requests

REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"]

SET_LISTED_COMPANIES_URL = (
    "https://www.set.or.th/dat/eod/listedcompany/static/"
    "listedCompanies_en_US.xls"
)

def load_set_listed_companies():
    """Load current SET/mai ordinary-share classification.

    Primary source is SET's official listed-company file/table.  When SET's
    legacy download endpoint is unavailable from the Colab runtime, fall back
    to a daily-updated public mirror whose dataset is sourced from SET.
    """
    global CLASSIFICATION_SOURCE_USED

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Fast-Hybrid-Pro/1.0)",
        "Accept": "text/html,application/vnd.ms-excel,application/octet-stream;q=0.9,*/*;q=0.8",
    }

    def normalize_classification(raw):
        raw = raw.copy()
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = [
                " ".join(str(x).strip() for x in col if str(x).strip() and str(x) != "nan")
                for col in raw.columns
            ]
        else:
            raw.columns = [str(x).strip() for x in raw.columns]

        lower = {str(c).strip().lower(): c for c in raw.columns}
        symbol_col = lower.get("symbol")
        market_col = lower.get("market")
        sector_col = lower.get("sector")
        if not all([symbol_col, market_col, sector_col]):
            raise ValueError("classification data missing Symbol/Market/Sector columns")

        industry_col = lower.get("industry group") or lower.get("industry")
        company_col = lower.get("company name") or lower.get("company") or lower.get("name")

        out = pd.DataFrame({
            "Symbol": raw[symbol_col].astype(str).str.strip().str.upper(),
            "Market": raw[market_col].astype(str).str.strip(),
            "Industry": raw[industry_col].astype(str).str.strip().str.upper()
                if industry_col else "",
            "Sector": raw[sector_col].astype(str).str.strip().str.upper(),
            "Company": raw[company_col].astype(str).str.strip()
                if company_col else "",
        })

        # Keep ordinary public-company rows. The SET classification feed also
        # contains REITs, funds and other instruments.
        ordinary_mask = out["Company"].str.contains(
            r"PUBLIC COMPANY LIMITED|PUBLIC CO\\.?\\s*,?\\s*LTD\\.?|PCL\\.?$",
            case=False,
            regex=True,
            na=False,
        )
        out = out[ordinary_mask].copy()

        # Normalize full sector names from the SET-sourced mirror to the
        # canonical SET sector codes used by the universe gate.
        sector_name_to_code = {
            "BANKING": "BANK",
            "FINANCE & SECURITIES": "FIN",
            "INSURANCE": "INSUR",
            "PROPERTY DEVELOPMENT": "PROP",
            "PROPERTY FUND & REITS": "PF&REIT",
            "TOURISM & LEISURE": "TOURISM",
        }
        out["Sector"] = (
            out["Sector"]
            .str.upper()
            .str.strip()
            .replace(sector_name_to_code)
        )

        out = out[out["Market"].isin(UNIVERSE_MARKETS)].copy()
        out = out[out["Symbol"].ne("")].drop_duplicates("Symbol")
        if out.empty:
            raise ValueError("classification returned zero SET/mai ordinary-company rows")
        return out.reset_index(drop=True)

    # 1) Official SET legacy download.
    try:
        response = requests.get(SET_LISTED_COMPANIES_URL, headers=headers, timeout=30)
        response.raise_for_status()
        payload = response.content

        tables = []
        try:
            tables = pd.read_html(io.BytesIO(payload))
        except Exception:
            tables = []

        if tables:
            for table in tables:
                try:
                    result = normalize_classification(table)
                    CLASSIFICATION_SOURCE_USED = SET_LISTED_COMPANIES_URL
                    print(f"Classification source: {CLASSIFICATION_SOURCE_USED}")
                    return result
                except Exception:
                    continue
    except Exception:
        pass

    # 2) Current SET web page (may be client-rendered in Colab).
    try:
        html_url = "https://www.set.or.th/en/market/information/securities-list/thaiesg"
        html_response = requests.get(html_url, headers=headers, timeout=30)
        html_response.raise_for_status()
        tables = pd.read_html(io.StringIO(html_response.text))
        for table in tables:
            try:
                result = normalize_classification(table)
                CLASSIFICATION_SOURCE_USED = html_url
                print(f"Classification source: {CLASSIFICATION_SOURCE_USED}")
                return result
            except Exception:
                continue
    except Exception:
        pass

    # 3) Resilient daily-updated mirror sourced from SET.
    mirror_url = (
        "https://raw.githubusercontent.com/lumduan/thai-securities-data/"
        "main/thai_securities_all_en.json"
    )
    try:
        mirror_response = requests.get(mirror_url, headers=headers, timeout=30)
        mirror_response.raise_for_status()
        payload = mirror_response.json()
        mirror_df = pd.DataFrame(payload if isinstance(payload, list) else list(payload.values()))
        result = normalize_classification(mirror_df)
        CLASSIFICATION_SOURCE_USED = mirror_url
        print(
            "Classification source: daily SET-sourced mirror "
            f"({CLASSIFICATION_SOURCE_USED})"
        )
        return result
    except Exception as e:
        raise RuntimeError(
            "Cannot load SET listed-company classification from official SET "
            "or the SET-sourced daily mirror. "
            f"Official={SET_LISTED_COMPANIES_URL}; mirror={mirror_url}; error={e}"
        )

def normalize_ohlcv(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)
    out = out.loc[:, ~out.columns.duplicated()]
    missing = [c for c in REQUIRED_COLS if c not in out.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns: {missing}")
    out = out[REQUIRED_COLS].copy()
    out.index = pd.to_datetime(out.index)
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)
    for col in REQUIRED_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=["Open", "High", "Low", "Close"])
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out

def download_stock(symbol, period="5y"):
    raw = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    return normalize_ohlcv(raw)

def get_market_cap_baht(symbol):
    """Best-effort current market cap from Yahoo; UNKNOWN is rejected."""
    try:
        info = yf.Ticker(symbol).fast_info
        value = info.get("market_cap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    try:
        value = yf.Ticker(symbol).info.get("marketCap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    return np.nan

def build_filtered_universe():
    listed = load_set_listed_companies()

    # Hard sector exclusion uses SET's canonical sector codes.
    listed = listed[~listed["Sector"].isin(UNIVERSE_EXCLUDED_SECTORS)].copy()
    listed_meta = listed.set_index("Symbol").to_dict("index")

    symbols = listed["Symbol"].tolist()
    yf_symbols = [f"{symbol}.BK" for symbol in symbols]

    print(f"SET/mai candidates after sector exclusion: {len(listed)}")
    print("Downloading one batched 3-month OHLCV snapshot for the universe...")

    # Yahoo can silently drop tickers when hundreds are requested in one batch.
    # Bounded chunks prevent that from masquerading as a fundamental-filter failure.
    raw_by_symbol = {}
    missing_ohlcv = []
    BATCH_SIZE = 40
    for start in range(0, len(yf_symbols), BATCH_SIZE):
        batch = yf_symbols[start:start + BATCH_SIZE]
        batch_raw = yf.download(
            batch,
            period="3mo",
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=True,
            group_by="ticker",
        )
        for yf_symbol in batch:
            try:
                if isinstance(batch_raw.columns, pd.MultiIndex):
                    if yf_symbol not in batch_raw.columns.get_level_values(0):
                        missing_ohlcv.append(yf_symbol)
                        continue
                    item = batch_raw[yf_symbol].copy()
                else:
                    item = batch_raw.copy()
                item = normalize_ohlcv(item)
                if item.empty or len(item) < 20:
                    missing_ohlcv.append(yf_symbol)
                else:
                    raw_by_symbol[yf_symbol] = item
            except Exception:
                missing_ohlcv.append(yf_symbol)

    print(f"OHLCV-ready symbols: {len(raw_by_symbol)} / {len(yf_symbols)}")
    print(f"OHLCV unavailable/insufficient: {len(missing_ohlcv)}")

    # Build the full OHLCV-stage audit first. This makes it impossible for
    # later market-cap lookups to hide how many names failed liquidity/price.
    stage_rows = []
    for symbol in symbols:
        yf_symbol = f"{symbol}.BK"
        df = raw_by_symbol.get(yf_symbol, pd.DataFrame())
        if df.empty:
            continue

        recent = df.tail(20).copy()
        total20_value_baht = float((recent["Close"] * recent["Volume"]).sum())
        avg20_value_baht = total20_value_baht / len(recent)
        last_price_baht = float(recent["Close"].iloc[-1])
        meta = listed_meta[symbol]

        stage_rows.append({
            "Symbol": yf_symbol,
            "Market": meta["Market"],
            "Industry": meta["Industry"],
            "Sector": meta["Sector"],
            "Total20TradingValueMB": total20_value_baht / 1_000_000,
            "Avg20TradingValueMB": avg20_value_baht / 1_000_000,
            "LastPrice": last_price_baht,
        })

    stage = pd.DataFrame(stage_rows)
    if stage.empty:
        raise RuntimeError("No OHLCV-ready symbols; refusing to run scanner.")

    stage["PassTotal20Value"] = (
        stage["Total20TradingValueMB"] > UNIVERSE_MIN_TOTAL20_VALUE_MBAHT
    )
    stage["PassPrice"] = stage["LastPrice"].between(
        UNIVERSE_MIN_PRICE_BAHT, UNIVERSE_MAX_PRICE_BAHT, inclusive="both"
    )
    stage["PassPreCap"] = stage["PassTotal20Value"] & stage["PassPrice"]

    print("\n====================================")
    print("UNIVERSE STAGE AUDIT — BEFORE MARKET CAP")
    print("====================================")
    print(f"OHLCV-ready                  : {len(stage)}")
    print(f"Total 20D value > 20 MB     : {int(stage['PassTotal20Value'].sum())}")
    print(f"Price 1–80 pass              : {int(stage['PassPrice'].sum())}")
    print(f"Liquidity + price pass       : {int(stage['PassPreCap'].sum())}")
    print(f"Total 20D value <= 20 MB     : {int((~stage['PassTotal20Value']).sum())}")
    print("")

    # Market cap is the slower external lookup, so only run it after the
    # explicitly requested liquidity and price gates have passed.
    candidates = stage[stage["PassPreCap"]].copy()
    market_caps = []
    for yf_symbol in candidates["Symbol"]:
        market_caps.append(get_market_cap_baht(yf_symbol))
    candidates["MarketCapMB"] = np.asarray(market_caps, dtype=float) / 1_000_000

    candidates["HasMarketCap"] = np.isfinite(candidates["MarketCapMB"])
    candidates["PassMarketCap"] = (
        candidates["HasMarketCap"]
        & (candidates["MarketCapMB"] > UNIVERSE_MIN_MARKET_CAP_MBAHT)
    )
    candidates["PassUniverse"] = candidates["PassMarketCap"]

    filtered = candidates[candidates["PassUniverse"]].copy()
    filtered = filtered.sort_values(
        ["Total20TradingValueMB", "MarketCapMB"],
        ascending=[False, False],
    ).reset_index(drop=True)

    print("\n====================================")
    print("FILTERED SET/mai FUNDAMENTAL UNIVERSE")
    print("====================================")
    print(f"Rows passing liquidity + price : {len(candidates)}")
    print(f"Market-cap data available      : {int(candidates['HasMarketCap'].sum())}")
    print(f"Market cap > 5,000 MB          : {int(candidates['PassMarketCap'].sum())}")
    print(f"Final universe                  : {len(filtered)}")

    # Diagnostic bands are informational only; they do not alter production.
    print("\nLIQUIDITY DISTRIBUTION — TOTAL 20 TRADING DAYS")
    bins = [-np.inf, 20, 50, 100, 250, 500, 1000, np.inf]
    labels = ["<=20", "20–50", "50–100", "100–250", "250–500", "500–1,000", ">=1,000"]
    print(
        stage["Total20TradingValueMB"]
        .groupby(pd.cut(stage["Total20TradingValueMB"], bins=bins, labels=labels))
        .size()
        .to_string()
    )

    print("\nTOP 20 BY TOTAL 20D TRADING VALUE")
    display(
        stage.sort_values("Total20TradingValueMB", ascending=False)
        [["Symbol", "Market", "Sector", "LastPrice", "Total20TradingValueMB", "Avg20TradingValueMB"]]
        .head(20)
    )

    print("\nTOP 20 BY MARKET CAP AMONG LIQUIDITY + PRICE PASS")
    display(
        candidates.sort_values("MarketCapMB", ascending=False)
        [["Symbol", "Market", "Sector", "LastPrice", "Total20TradingValueMB", "MarketCapMB"]]
        .head(20)
    )

    display(filtered.head(100))
    return filtered, pd.concat(
        [stage, candidates[["Symbol", "MarketCapMB", "HasMarketCap", "PassMarketCap"]]],
        ignore_index=True,
        sort=False,
    ).drop_duplicates("Symbol", keep="last")


filtered_universe, universe_audit = build_filtered_universe()

PRODUCTION_SYMBOLS = filtered_universe["Symbol"].tolist()
WATCHLIST = PRODUCTION_SYMBOLS.copy()

print(f"\nProduction symbols loaded: {len(PRODUCTION_SYMBOLS)}")
print(PRODUCTION_SYMBOLS)

data = {}
errors = {}

print("\n🚀 Downloading filtered-universe market data...")

for symbol in WATCHLIST:
    try:
        df = download_stock(symbol)
        if len(df) < MIN_BARS:
            errors[symbol] = f"Only {len(df)} bars (minimum {MIN_BARS})"
        else:
            data[symbol] = df
            print(f"✅ {symbol:<12} {len(df):>5} bars")
    except Exception as e:
        errors[symbol] = str(e)
        print(f"❌ {symbol}: {e}")

print("\n====================================")
print(f"Loaded: {len(data)} / {len(WATCHLIST)}")
print("====================================")

if errors:
    print("\n⚠️ Errors / insufficient data:")
    for symbol, error in errors.items():
        print(f"{symbol}: {error}")


Classification source: daily SET-sourced mirror (https://raw.githubusercontent.com/lumduan/thai-securities-data/main/thai_securities_all_en.json)
SET/mai candidates after sector exclusion: 713


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ACAP.BK"}}}


ERROR:yfinance:
1 Failed download:


ERROR:yfinance:['ACAP.BK']: YFPricesMissingError('possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")')


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BLISS.BK"}}}


...

✅ SNNP.BK       1211 bars
✅ ONEE.BK       1194 bars


✅ MEDEZE.BK      481 bars
✅ PM.BK         1211 bars


✅ SKR.BK        1211 bars
✅ SMPC.BK       1211 bars


✅ PSGC.BK       1211 bars
✅ SCGD.BK        679 bars


✅ TFM.BK        1199 bars
✅ SPC.BK        1211 bars

Loaded: 133 / 133


In [ ]:

# ============================================================
# 4) DAILY INDICATORS
# ============================================================

def wilder_rma(series, length):
    """
    Wilder RMA
    ใช้แนวคิดเดียวกับ TradingView ta.rma()
    """

    return series.ewm(
        alpha=1 / length,
        adjust=False,
        min_periods=length
    ).mean()


def calculate_atr(df, length=14):

    prev_close = df["Close"].shift(1)

    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - prev_close).abs()
    tr3 = (df["Low"] - prev_close).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    atr = wilder_rma(
        true_range,
        length
    )

    return true_range, atr


def calculate_daily_indicators(df):

    out = df.copy()

    close = out["Close"]
    high = out["High"]
    low = out["Low"]
    volume = out["Volume"]

    # --------------------------------------------------------
    # EMA 5 / 10 / 75
    # --------------------------------------------------------

    out["EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    out["EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    out["EMA75"] = close.ewm(
        span=75,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # MACD 4 / 12 / 9
    # --------------------------------------------------------

    ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    out["MACD"] = ema_fast - ema_slow

    out["MACD_SIGNAL"] = out["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    out["MACD_HIST"] = (
        out["MACD"] - out["MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # STOCHASTIC 8 / 3 / 3
    # --------------------------------------------------------

    lowest_8 = low.rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = high.rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = highest_8 - lowest_8

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range.replace(0, np.nan)
    )

    out["STOCH_K"] = stoch_raw.rolling(
        window=3,
        min_periods=3
    ).mean()

    out["STOCH_D"] = out["STOCH_K"].rolling(
        window=3,
        min_periods=3
    ).mean()

    # --------------------------------------------------------
    # ATR 14 — WILDER
    # --------------------------------------------------------

    out["TR"], out["ATR14"] = calculate_atr(
        out,
        length=14
    )

    # --------------------------------------------------------
    # BOLLINGER BAND 20 / 2
    # --------------------------------------------------------

    bb_mid = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    bb_std = close.rolling(
        window=20,
        min_periods=20
    ).std()

    out["BB_MID"] = bb_mid

    out["BB_UPPER"] = (
        bb_mid + (2.0 * bb_std)
    )

    out["BB_LOWER"] = (
        bb_mid - (2.0 * bb_std)
    )

    # --------------------------------------------------------
    # KELTNER CHANNEL
    # --------------------------------------------------------

    out["KC_MID"] = bb_mid

    out["KC_UPPER"] = (
        bb_mid + (1.5 * out["ATR14"])
    )

    out["KC_LOWER"] = (
        bb_mid - (1.5 * out["ATR14"])
    )

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------

    out["SQUEEZE"] = (
        (out["BB_UPPER"] <= out["KC_UPPER"])
        &
        (out["BB_LOWER"] >= out["KC_LOWER"])
    )

    # --------------------------------------------------------
    # SQUEEZE RELEASE
    # --------------------------------------------------------

    out["SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    # --------------------------------------------------------
    # BARS SINCE LAST SQUEEZE
    # --------------------------------------------------------

    bars_since_squeeze = []

    last_squeeze_index = None

    for i, is_squeeze in enumerate(
        out["SQUEEZE"].fillna(False)
    ):

        if is_squeeze:

            last_squeeze_index = i
            bars_since_squeeze.append(0)

        elif last_squeeze_index is None:

            bars_since_squeeze.append(np.nan)

        else:

            bars_since_squeeze.append(
                i - last_squeeze_index
            )

    out["BARS_SINCE_SQUEEZE"] = (
        bars_since_squeeze
    )

    out["SQUEEZE_RECENT"] = (
        out["BARS_SINCE_SQUEEZE"] <= 10
    )

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------

    out["VOLUME_MA5"] = volume.rolling(
        window=5,
        min_periods=5
    ).mean()

    out["VOLUME_RATIO"] = (
        volume
        / out["VOLUME_MA5"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # CANDLE BODY
    # --------------------------------------------------------

    out["BODY"] = (
        close - out["Open"]
    ).abs()

    out["BODY_ATR_RATIO"] = (
        out["BODY"]
        / out["ATR14"].replace(0, np.nan)
    )

    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

daily_data = {}
indicator_errors = {}

print("📊 Calculating daily indicators...\n")

for symbol, df in data.items():

    try:

        result = calculate_daily_indicators(df)

        daily_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        indicator_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Daily indicator data: "
    f"{len(daily_data)} / {len(data)}"
)
print("====================================")


if indicator_errors:

    print("\n⚠️ Indicator errors:")

    for symbol, error in indicator_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# QUICK VALIDATION
# ============================================================

if daily_data:

    sample_symbol = next(iter(daily_data))

    sample = daily_data[sample_symbol]

    print(
        f"\n🔎 Quick validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "ATR14",
                "SQUEEZE",
                "SQUEEZE_RELEASE",
                "VOLUME_RATIO"
            ]
        ].tail(5)
    )

📊 Calculating daily indicators...

✅ PTT.BK        1211 bars
✅ KCE.BK        1211 bars
✅ HANA.BK       1211 bars


✅ GULF.BK        365 bars
✅ CPALL.BK      1211 bars
✅ TOP.BK        1211 bars
✅ TRUE.BK       1211 bars
✅ AOT.BK        1211 bars
✅ PTTGC.BK      1211 bars
✅ BCP.BK        1211 bars
✅ BDMS.BK       1211 bars
✅ IVL.BK        1211 bars
✅ THAI.BK       1211 bars
✅ CPF.BK        1211 bars
✅ IRPC.BK       1211 bars
✅ STECON.BK      471 bars
✅ GPSC.BK       1211 bars
✅ SPRC.BK       1211 bars
✅ BGRIM.BK      1211 bars
✅ GUNKUL.BK     1211 bars


✅ COM7.BK       1211 bars
✅ BEM.BK        1211 bars
✅ TU.BK         1211 bars
✅ CRC.BK        1211 bars


✅ STA.BK        1211 bars
✅ CBG.BK        1211 bars
✅ SCGP.BK       1211 bars
✅ RCL.BK        1211 bars
✅ HMPRO.BK      1211 bars
✅ SMT.BK        1211 bars
✅ PLANB.BK      1211 bars
✅ BJC.BK        1211 bars
✅ RATCH.BK      1211 bars
✅ CK.BK         1211 bars
✅ OR.BK         1211 bars
✅ OSP.BK        1211 bars
✅ ITC.BK         930 bars
✅ BA.BK         1211 bars
✅ BCH.BK        1211 bars
✅ PTG.BK        1211 bars
✅ EA.BK         1211 bars
✅ SPCG.BK       1211 bars


✅ GLOBAL.BK     1211 bars
✅ TFG.BK        1211 bars
✅ TASCO.BK      1211 bars
✅ PSL.BK        1211 bars


...

✅ XO.BK         1211 bars
✅ SISB.BK       1211 bars
✅ KSL.BK        1211 bars
✅ SAT.BK        1211 bars
✅ KLINIQ.BK      953 bars
✅ PCE.BK         503 bars
✅ AURA.BK        937 bars
✅ UVAN.BK       1211 bars
✅ THG.BK        1211 bars
✅ SABINA.BK     1211 bars
✅ TPIPP.BK      1211 bars
✅ CKP.BK        1211 bars
✅ RBF.BK        1211 bars
✅ VIH.BK        1211 bars
✅ NSL.BK        1211 bars
✅ SAPPE.BK      1211 bars
✅ KAMART.BK     1211 bars


✅ HTC.BK        1211 bars
✅ SAMART.BK     1211 bars
✅ DCC.BK        1211 bars
✅ KCG.BK         773 bars
✅ SGP.BK        1211 bars
✅ UTP.BK        1211 bars


✅ ACE.BK        1211 bars
✅ SAV.BK         736 bars
✅ SRICHA.BK     1211 bars
✅ DMT.BK        1211 bars
✅ AH.BK         1211 bars
✅ COCOCO.BK      744 bars
✅ LANNA.BK      1211 bars
✅ SPI.BK        1211 bars
✅ ASIAN.BK      1211 bars
✅ ILM.BK        1211 bars
✅ SUC.BK        1211 bars
✅ SNNP.BK       1211 bars


✅ ONEE.BK       1194 bars
✅ MEDEZE.BK      481 bars
✅ PM.BK         1211 bars
✅ SKR.BK        1211 bars
✅ SMPC.BK       1211 bars
✅ PSGC.BK       1211 bars
✅ SCGD.BK        679 bars
✅ TFM.BK        1199 bars
✅ SPC.BK        1211 bars

Daily indicator data: 133 / 133

🔎 Quick validation: PTT.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,ATR14,SQUEEZE,SQUEEZE_RELEASE,VOLUME_RATIO
Date,,,,,,,,,,,,,
2026-10-02,41.5,41.486885,41.746421,39.832476,-0.380120,-0.049819,-0.330301,20.000000,15.833333,0.776756,False,False,0.884640
2026-10-05,41.5,41.491257,41.701618,39.876358,-0.297412,-0.099338,-0.198074,33.333333,21.388889,0.774845,False,False,0.534009
2026-10-06,41.5,41.494171,41.664960,39.919085,-0.237119,-0.126894,-0.110225,40.000000,31.111111,0.755213,False,False,0.876104
2026-10-07,40.5,41.162781,41.453149,39.934373,-0.438071,-0.189129,-0.248942,31.794872,35.042735,0.808412,False,False,0.682990
2026-10-08,40.0,40.775187,41.188940,39.936100,-0.636211,-0.278546,-0.357666,18.461538,30.085470,0.786383,False,False,0.904417


In [ ]:

# ============================================================
# 5) WEEKLY INDICATORS
# ============================================================

def calculate_weekly_indicators(df):

    daily = df.copy()

    # --------------------------------------------------------
    # Daily -> Weekly
    #
    # Week ends on Friday.
    # --------------------------------------------------------

    weekly = daily.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    })

    weekly = weekly.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    close = weekly["Close"]

    # --------------------------------------------------------
    # WEEKLY EMA 5 / 10
    # --------------------------------------------------------

    weekly["W_EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    weekly["W_EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # WEEKLY MACD 4 / 12 / 9
    # --------------------------------------------------------

    weekly_ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    weekly_ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    weekly["W_MACD"] = (
        weekly_ema_fast
        - weekly_ema_slow
    )

    weekly["W_MACD_SIGNAL"] = (
        weekly["W_MACD"].ewm(
            span=9,
            adjust=False
        ).mean()
    )

    weekly["W_MACD_HIST"] = (
        weekly["W_MACD"]
        - weekly["W_MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # A weekly candle ending Friday becomes available
    # only from the next trading day.
    #
    # Example:
    #
    # Week ending 2026-09-25
    # -> usable from 2026-09-28
    #
    # This prevents Friday from seeing its own completed
    # weekly candle during the Friday signal.
    # --------------------------------------------------------

    weekly_features = weekly[
        [
            "W_EMA5",
            "W_EMA10",
            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]
    ].copy()

    # Move availability date to next Monday.
    #
    # W-FRI + 3 days = Monday
    #
    # If Monday is a market holiday, forward-fill will keep
    # the last completed weekly value until the next trading day.
    weekly_features.index = (
        weekly_features.index
        + pd.Timedelta(days=3)
    )

    # --------------------------------------------------------
    # Map completed weekly values to daily data
    # --------------------------------------------------------

    daily_index = daily.index

    mapped_weekly = weekly_features.reindex(
        daily_index,
        method="ffill"
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    out = daily.copy()

    for col in mapped_weekly.columns:
        out[col] = mapped_weekly[col]

    return out, weekly


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

weekly_data = {}
combined_data = {}
weekly_errors = {}

print("📅 Calculating weekly indicators...\n")

for symbol, df in daily_data.items():

    try:

        combined, weekly = calculate_weekly_indicators(df)

        combined_data[symbol] = combined
        weekly_data[symbol] = weekly

        print(
            f"✅ {symbol:<12} "
            f"Daily: {len(combined):>5} | "
            f"Weekly: {len(weekly):>4}"
        )

    except Exception as e:

        weekly_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Weekly indicator data: "
    f"{len(combined_data)} / {len(daily_data)}"
)
print("====================================")


if weekly_errors:

    print("\n⚠️ Weekly indicator errors:")

    for symbol, error in weekly_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if combined_data:

    sample_symbol = next(iter(combined_data))

    sample = combined_data[sample_symbol]

    print(
        f"\n🔎 Weekly validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST"
            ]
        ].tail(15)
    )

📅 Calculating weekly indicators...

✅ PTT.BK       Daily:  1211 | Weekly:  261
✅ KCE.BK       Daily:  1211 | Weekly:  261
✅ HANA.BK      Daily:  1211 | Weekly:  261
✅ GULF.BK      Daily:   365 | Weekly:   80
✅ CPALL.BK     Daily:  1211 | Weekly:  261
✅ TOP.BK       Daily:  1211 | Weekly:  261
✅ TRUE.BK      Daily:  1211 | Weekly:  261


✅ AOT.BK       Daily:  1211 | Weekly:  261
✅ PTTGC.BK     Daily:  1211 | Weekly:  261
✅ BCP.BK       Daily:  1211 | Weekly:  261
✅ BDMS.BK      Daily:  1211 | Weekly:  261
✅ IVL.BK       Daily:  1211 | Weekly:  261
✅ THAI.BK      Daily:  1211 | Weekly:  261
✅ CPF.BK       Daily:  1211 | Weekly:  261
✅ IRPC.BK      Daily:  1211 | Weekly:  261
✅ STECON.BK    Daily:   471 | Weekly:  102
✅ GPSC.BK      Daily:  1211 | Weekly:  261
✅ SPRC.BK      Daily:  1211 | Weekly:  261
✅ BGRIM.BK     Daily:  1211 | Weekly:  261
✅ GUNKUL.BK    Daily:  1211 | Weekly:  261
✅ COM7.BK      Daily:  1211 | Weekly:  261


✅ BEM.BK       Daily:  1211 | Weekly:  261
✅ TU.BK        Daily:  1211 | Weekly:  261
✅ CRC.BK       Daily:  1211 | Weekly:  261
✅ STA.BK       Daily:  1211 | Weekly:  261
✅ CBG.BK       Daily:  1211 | Weekly:  261
✅ SCGP.BK      Daily:  1211 | Weekly:  261
✅ RCL.BK       Daily:  1211 | Weekly:  261


✅ HMPRO.BK     Daily:  1211 | Weekly:  261
✅ SMT.BK       Daily:  1211 | Weekly:  261
✅ PLANB.BK     Daily:  1211 | Weekly:  261
✅ BJC.BK       Daily:  1211 | Weekly:  261
✅ RATCH.BK     Daily:  1211 | Weekly:  261
✅ CK.BK        Daily:  1211 | Weekly:  261
✅ OR.BK        Daily:  1211 | Weekly:  261
✅ OSP.BK       Daily:  1211 | Weekly:  261
✅ ITC.BK       Daily:   930 | Weekly:  201
✅ BA.BK        Daily:  1211 | Weekly:  261
✅ BCH.BK       Daily:  1211 | Weekly:  261
✅ PTG.BK       Daily:  1211 | Weekly:  261
✅ EA.BK        Daily:  1211 | Weekly:  261
✅ SPCG.BK      Daily:  1211 | Weekly:  261


✅ GLOBAL.BK    Daily:  1211 | Weekly:  261
✅ TFG.BK       Daily:  1211 | Weekly:  261
✅ TASCO.BK     Daily:  1211 | Weekly:  261
✅ PSL.BK       Daily:  1211 | Weekly:  261
✅ BANPU.BK     Daily:  1170 | Weekly:  253


...

✅ KAMART.BK    Daily:  1211 | Weekly:  261
✅ HTC.BK       Daily:  1211 | Weekly:  261
✅ SAMART.BK    Daily:  1211 | Weekly:  261
✅ DCC.BK       Daily:  1211 | Weekly:  261
✅ KCG.BK       Daily:   773 | Weekly:  167
✅ SGP.BK       Daily:  1211 | Weekly:  261


✅ UTP.BK       Daily:  1211 | Weekly:  261
✅ ACE.BK       Daily:  1211 | Weekly:  261
✅ SAV.BK       Daily:   736 | Weekly:  159
✅ SRICHA.BK    Daily:  1211 | Weekly:  261
✅ DMT.BK       Daily:  1211 | Weekly:  261
✅ AH.BK        Daily:  1211 | Weekly:  261
✅ COCOCO.BK    Daily:   744 | Weekly:  161
✅ LANNA.BK     Daily:  1211 | Weekly:  261
✅ SPI.BK       Daily:  1211 | Weekly:  261
✅ ASIAN.BK     Daily:  1211 | Weekly:  261
✅ ILM.BK       Daily:  1211 | Weekly:  261
✅ SUC.BK       Daily:  1211 | Weekly:  261
✅ SNNP.BK      Daily:  1211 | Weekly:  261
✅ ONEE.BK      Daily:  1194 | Weekly:  258
✅ MEDEZE.BK    Daily:   481 | Weekly:  104


✅ PM.BK        Daily:  1211 | Weekly:  261
✅ SKR.BK       Daily:  1211 | Weekly:  261
✅ SMPC.BK      Daily:  1211 | Weekly:  261
✅ PSGC.BK      Daily:  1211 | Weekly:  261
✅ SCGD.BK      Daily:   679 | Weekly:  147


✅ TFM.BK       Daily:  1199 | Weekly:  259
✅ SPC.BK       Daily:  1211 | Weekly:  261

Weekly indicator data: 133 / 133

🔎 Weekly validation: PTT.BK


Price,Close,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST
Date,,,,,,
2026-09-18,42.25,40.916977,39.839248,1.684263,1.319765,0.364498
2026-09-21,42.00,41.361318,40.277567,1.691150,1.394042,0.297108
2026-09-22,42.00,41.361318,40.277567,1.691150,1.394042,0.297108
2026-09-23,42.00,41.361318,40.277567,1.691150,1.394042,0.297108
2026-09-24,43.25,41.361318,40.277567,1.691150,1.394042,0.297108
2026-09-25,43.00,41.361318,40.277567,1.691150,1.394042,0.297108
2026-09-28,42.50,41.907545,40.772554,1.775191,1.470272,0.304920
2026-09-29,42.00,41.907545,40.772554,1.775191,1.470272,0.304920
2026-09-30,40.75,41.907545,40.772554,1.775191,1.470272,0.304920


In [ ]:

# ============================================================
# 6) BUILD FEATURE DATASETS
# ============================================================

feature_data = {}

feature_errors = {}


for symbol, df in combined_data.items():

    try:

        out = df.copy()

        # ----------------------------------------------------
        # Basic validity
        # ----------------------------------------------------

        required = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",

            "EMA5",
            "EMA10",
            "EMA75",

            "MACD",
            "MACD_SIGNAL",
            "MACD_HIST",

            "STOCH_K",
            "STOCH_D",

            "ATR14",

            "BB_UPPER",
            "BB_LOWER",

            "KC_UPPER",
            "KC_LOWER",

            "SQUEEZE",
            "SQUEEZE_RELEASE",
            "SQUEEZE_RECENT",

            "VOLUME_RATIO",

            "W_EMA5",
            "W_EMA10",

            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]

        missing = [
            col
            for col in required
            if col not in out.columns
        ]

        if missing:

            raise ValueError(
                f"Missing columns: {missing}"
            )

        # ----------------------------------------------------
        # Remove rows that cannot produce indicators
        #
        # IMPORTANT:
        # We do NOT drop rows globally before calculating
        # indicators. All indicators have already been
        # calculated using the full historical dataset.
        # ----------------------------------------------------

        out = out.dropna(
            subset=[
                "EMA75",
                "ATR14",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "VOLUME_RATIO"
            ]
        )

        # ----------------------------------------------------
        # Ensure chronological order
        # ----------------------------------------------------

        out = out.sort_index()

        # ----------------------------------------------------
        # Ensure unique dates
        # ----------------------------------------------------

        out = out[
            ~out.index.duplicated(
                keep="last"
            )
        ]

        # ----------------------------------------------------
        # Create previous-bar values
        #
        # Used later for signal confirmation.
        # ----------------------------------------------------

        out["PREV_CLOSE"] = (
            out["Close"].shift(1)
        )

        out["PREV_MACD"] = (
            out["MACD"].shift(1)
        )

        out["PREV_MACD_SIGNAL"] = (
            out["MACD_SIGNAL"].shift(1)
        )

        out["PREV_MACD_HIST"] = (
            out["MACD_HIST"].shift(1)
        )

        out["PREV_STOCH_K"] = (
            out["STOCH_K"].shift(1)
        )

        out["PREV_STOCH_D"] = (
            out["STOCH_D"].shift(1)
        )

        # ----------------------------------------------------
        # Daily trend state
        # ----------------------------------------------------

        out["TREND_BULL"] = (
            (out["Close"] > out["EMA5"])
            &
            (out["EMA5"] > out["EMA10"])
            &
            (out["EMA10"] > out["EMA75"])
        )

        # ----------------------------------------------------
        # Weekly trend state
        # ----------------------------------------------------

        out["WEEKLY_TREND_BULL"] = (
            out["W_EMA5"] > out["W_EMA10"]
        )

        # ----------------------------------------------------
        # Daily MACD bullish state
        # ----------------------------------------------------

        out["MACD_BULL"] = (
            out["MACD"] > out["MACD_SIGNAL"]
        )

        out["MACD_HIST_RISING"] = (
            out["MACD_HIST"]
            > out["PREV_MACD_HIST"]
        )

        # ----------------------------------------------------
        # Weekly MACD bullish state
        # ----------------------------------------------------

        out["WEEKLY_MACD_BULL"] = (
            out["W_MACD"] > out["W_MACD_SIGNAL"]
        )

        out["WEEKLY_MACD_HIST_RISING"] = (
            out["W_MACD_HIST"]
            > out["W_MACD_HIST"].shift(1)
        )

        # ----------------------------------------------------
        # Stochastic bullish state
        # ----------------------------------------------------

        out["STOCH_BULL"] = (
            (out["STOCH_K"] > out["STOCH_D"])
            &
            (out["STOCH_K"] < 80)
        )

        # ----------------------------------------------------
        # Volume confirmation
        # ----------------------------------------------------

        out["VOLUME_CONFIRM"] = (
            out["VOLUME_RATIO"]
            >= 1.2
        )

        # ----------------------------------------------------
        # Early-entry / squeeze state
        # ----------------------------------------------------

        out["EARLY_ENTRY_ZONE"] = (
            out["SQUEEZE"]
            |
            out["SQUEEZE_RELEASE"]
            |
            out["SQUEEZE_RECENT"]
        )

        # ----------------------------------------------------
        # Target prices
        # ----------------------------------------------------

        out["TARGET1"] = (
            out["Close"] * 1.07
        )

        out["TARGET2"] = (
            out["Close"] * 1.15
        )

        # ----------------------------------------------------
        # Final dataset
        # ----------------------------------------------------

        feature_data[symbol] = out

    except Exception as e:

        feature_errors[symbol] = str(e)


print("📦 Building feature datasets...\n")

for symbol, df in feature_data.items():

    print(
        f"✅ {symbol:<12} "
        f"{len(df):>5} usable bars"
    )


print("\n====================================")
print(
    f"Feature datasets: "
    f"{len(feature_data)} / {len(combined_data)}"
)
print("====================================")


if feature_errors:

    print("\n⚠️ Feature errors:")

    for symbol, error in feature_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if feature_data:

    sample_symbol = next(iter(feature_data))

    sample = feature_data[sample_symbol]

    print(
        f"\n🔎 Feature validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "TREND_BULL",
                "WEEKLY_TREND_BULL",
                "MACD_BULL",
                "STOCH_BULL",
                "VOLUME_CONFIRM",
                "EARLY_ENTRY_ZONE"
            ]
        ].tail(10)
    )

📦 Building feature datasets...

✅ PTT.BK        1197 usable bars
✅ KCE.BK        1198 usable bars
✅ HANA.BK       1198 usable bars
✅ GULF.BK        352 usable bars
✅ CPALL.BK      1198 usable bars
✅ TOP.BK        1196 usable bars
✅ TRUE.BK       1188 usable bars
✅ AOT.BK        1197 usable bars
✅ PTTGC.BK      1198 usable bars
✅ BCP.BK        1198 usable bars
✅ BDMS.BK       1197 usable bars
✅ IVL.BK        1198 usable bars
✅ THAI.BK        284 usable bars
✅ CPF.BK        1198 usable bars
✅ IRPC.BK       1198 usable bars
✅ STECON.BK      458 usable bars
✅ GPSC.BK       1198 usable bars
✅ SPRC.BK       1198 usable bars
✅ BGRIM.BK      1198 usable bars
✅ GUNKUL.BK     1198 usable bars
✅ COM7.BK       1198 usable bars
✅ BEM.BK        1197 usable bars
✅ TU.BK         1197 usable bars
✅ CRC.BK        1198 usable bars
✅ STA.BK        1198 usable bars
✅ CBG.BK        1198 usable bars
✅ SCGP.BK       1198 usable bars
✅ RCL.BK        1198 usable bars
✅ HMPRO.BK      1198 usable bars
✅ SMT.BK   

Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,TREND_BULL,WEEKLY_TREND_BULL,MACD_BULL,STOCH_BULL,VOLUME_CONFIRM,EARLY_ENTRY_ZONE
Date,,,,,,,,,,,,,,,,,,
2026-09-25,43.00,42.634785,42.367338,39.597054,0.439980,0.325283,0.114698,41.361318,40.277567,1.691150,1.394042,0.297108,True,True,True,True,True,False
2026-09-28,42.50,42.589856,42.391458,39.673447,0.319197,0.324065,-0.004869,41.907545,40.772554,1.775191,1.470272,0.304920,False,True,False,True,False,True
2026-09-29,42.00,42.393238,42.320284,39.734672,0.115156,0.282284,-0.167127,41.907545,40.772554,1.775191,1.470272,0.304920,False,True,False,False,False,True
2026-09-30,40.75,41.845492,42.034778,39.761392,-0.303212,0.165184,-0.468397,41.907545,40.772554,1.775191,1.470272,0.304920,False,True,False,False,True,True
2026-10-01,40.75,41.480328,41.801182,39.787408,-0.496956,0.032756,-0.529712,41.907545,40.772554,1.775191,1.470272,0.304920,False,True,False,False,False,True
2026-10-02,41.50,41.486885,41.746421,39.832476,-0.380120,-0.049819,-0.330301,41.907545,40.772554,1.775191,1.470272,0.304920,False,True,False,True,False,True
2026-10-05,41.50,41.491257,41.701618,39.876358,-0.297412,-0.099338,-0.198074,41.771697,40.904817,1.339385,1.444094,-0.104709,False,True,False,True,False,True
2026-10-06,41.50,41.494171,41.664960,39.919085,-0.237119,-0.126894,-0.110225,41.771697,40.904817,1.339385,1.444094,-0.104709,False,True,False,True,False,True
2026-10-07,40.50,41.162781,41.453149,39.934373,-0.438071,-0.189129,-0.248942,41.771697,40.904817,1.339385,1.444094,-0.104709,False,True,False,False,False,True


In [ ]:

# ============================================================
# 7) LEGACY SCORE
# ============================================================
#
# วัตถุประสงค์:
# - เก็บสูตร FAST HYBRID PRO เดิมไว้เป็น Baseline
# - ใช้เปรียบเทียบกับ Canonical Research Score ใน Cell 8
# - ยังไม่ทำ Backtest และยังไม่เปิด Position
#
# LEGACY 5-POINT SCORE
# 1. MACD 6/13/5 + Histogram Rising
# 2. Stochastic 8/3/3 : K > D และ K < 80
# 3. Trend: Close > EMA20 > EMA50
# 4. Candle Body / ATR >= 0.8
# 5. Volatility: BB breakout OR squeeze release
#
# หมายเหตุ:
# - Cell 4 ใช้ EMA5/10/75 และ MACD 4/12/9 เป็น canonical data
# - ดังนั้น Legacy MACD จะคำนวณใหม่ตรงนี้ด้วย 6/13/5
# - EMA20/EMA50 ก็จะคำนวณใหม่เพื่อรักษาสูตร Legacy เดิม
# ============================================================


def calculate_legacy_score(df):
    out = df.copy()

    # --------------------------------------------------------
    # 1) LEGACY MACD 6/13/5
    # --------------------------------------------------------
    close = out["Close"]

    legacy_ema_fast = close.ewm(
        span=6,
        adjust=False
    ).mean()

    legacy_ema_slow = close.ewm(
        span=13,
        adjust=False
    ).mean()

    out["LEGACY_MACD"] = (
        legacy_ema_fast - legacy_ema_slow
    )

    out["LEGACY_MACD_SIGNAL"] = (
        out["LEGACY_MACD"]
        .ewm(span=5, adjust=False)
        .mean()
    )

    out["LEGACY_MACD_HIST"] = (
        out["LEGACY_MACD"]
        - out["LEGACY_MACD_SIGNAL"]
    )

    out["LEGACY_MACD_HIST_PREV"] = (
        out["LEGACY_MACD_HIST"].shift(1)
    )

    out["LEGACY_MACD_BULL"] = (
        (out["LEGACY_MACD"] > out["LEGACY_MACD_SIGNAL"])
        &
        (
            out["LEGACY_MACD_HIST"]
            > out["LEGACY_MACD_HIST_PREV"]
        )
    )


    # --------------------------------------------------------
    # 2) LEGACY STOCHASTIC 8/3/3
    # --------------------------------------------------------
    lowest_8 = out["Low"].rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = out["High"].rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = (
        highest_8 - lowest_8
    ).replace(0, np.nan)

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range
    )

    out["LEGACY_STOCH_K"] = (
        stoch_raw
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_D"] = (
        out["LEGACY_STOCH_K"]
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_BULL"] = (
        (out["LEGACY_STOCH_K"] > out["LEGACY_STOCH_D"])
        &
        (out["LEGACY_STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) LEGACY TREND
    # Close > EMA20 > EMA50
    # --------------------------------------------------------
    out["LEGACY_EMA20"] = (
        close.ewm(
            span=20,
            adjust=False
        ).mean()
    )

    out["LEGACY_EMA50"] = (
        close.ewm(
            span=50,
            adjust=False
        ).mean()
    )

    out["LEGACY_TREND_BULL"] = (
        (close > out["LEGACY_EMA20"])
        &
        (out["LEGACY_EMA20"] > out["LEGACY_EMA50"])
    )


    # --------------------------------------------------------
    # 4) CANDLE BODY / ATR >= 0.8
    # --------------------------------------------------------
    #
    # ใช้ ATR14 จาก Cell 4
    # Body = abs(Close - Open)
    #
    # เงื่อนไขเดิม:
    # Body / ATR >= 0.8
    # --------------------------------------------------------

    out["LEGACY_BODY_ATR_RATIO"] = (
        (out["Close"] - out["Open"]).abs()
        / out["ATR14"].replace(0, np.nan)
    )

    out["LEGACY_CANDLE_BULL"] = (
        out["LEGACY_BODY_ATR_RATIO"] >= 0.8
    )


    # --------------------------------------------------------
    # 5) VOLATILITY
    # BB Breakout OR Squeeze Release
    # --------------------------------------------------------
    #
    # BB breakout:
    # Close > Upper BB
    #
    # Squeeze release:
    # SQUEEZE วันนี้ False
    # และเมื่อวานเป็น True
    # --------------------------------------------------------

    out["LEGACY_BB_BREAKOUT"] = (
        out["Close"] > out["BB_UPPER"]
    )

    out["LEGACY_SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    out["LEGACY_VOLATILITY_BULL"] = (
        out["LEGACY_BB_BREAKOUT"]
        |
        out["LEGACY_SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # LEGACY SCORE
    # --------------------------------------------------------

    score_components = [
        "LEGACY_MACD_BULL",
        "LEGACY_STOCH_BULL",
        "LEGACY_TREND_BULL",
        "LEGACY_CANDLE_BULL",
        "LEGACY_VOLATILITY_BULL",
    ]

    out["LEGACY_SCORE"] = (
        out[score_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # Signal classification
    # --------------------------------------------------------
    #
    # Legacy เดิมใช้ 5 คะแนนเต็ม
    #
    # >= 3 = ผ่าน baseline
    # < 3  = ไม่ผ่าน
    # --------------------------------------------------------

    out["LEGACY_SIGNAL"] = (
        out["LEGACY_SCORE"] >= 3
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

legacy_data = {}
legacy_errors = {}

print("🧮 Calculating Legacy Score...\n")

for symbol, df in feature_data.items():
    try:
        result = calculate_legacy_score(df)

        legacy_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:
        legacy_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Legacy datasets: "
    f"{len(legacy_data)} / {len(feature_data)}"
)
print("====================================")


if legacy_errors:
    print("\n⚠️ Legacy calculation errors:")

    for symbol, error in legacy_errors.items():
        print(f"{symbol}: {error}")


# ============================================================
# VALIDATION
# ============================================================

if legacy_data:

    sample_symbol = next(iter(legacy_data))
    sample = legacy_data[sample_symbol]

    print(
        f"\n🔎 Legacy validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "LEGACY_EMA20",
                "LEGACY_EMA50",

                "LEGACY_MACD",
                "LEGACY_MACD_SIGNAL",
                "LEGACY_MACD_HIST",

                "LEGACY_STOCH_K",
                "LEGACY_STOCH_D",

                "LEGACY_BODY_ATR_RATIO",

                "LEGACY_BB_BREAKOUT",
                "LEGACY_SQUEEZE_RELEASE",

                "LEGACY_MACD_BULL",
                "LEGACY_STOCH_BULL",
                "LEGACY_TREND_BULL",
                "LEGACY_CANDLE_BULL",
                "LEGACY_VOLATILITY_BULL",

                "LEGACY_SCORE",
                "LEGACY_SIGNAL",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if legacy_data:

    all_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    print("\n📊 Legacy Score Distribution")

    display(
        all_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )

    print(
        f"\nBars with Legacy Score >= 3: "
        f"{int((all_scores >= 3).sum()):,}"
    )

    print(
        f"Total usable bars: "
        f"{len(all_scores):,}"
    )

    print(
        f"Signal rate: "
        f"{(all_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Legacy Score...

✅ PTT.BK        1197 bars
✅ KCE.BK        1198 bars
✅ HANA.BK       1198 bars
✅ GULF.BK        352 bars
✅ CPALL.BK      1198 bars
✅ TOP.BK        1196 bars
✅ TRUE.BK       1188 bars
✅ AOT.BK        1197 bars
✅ PTTGC.BK      1198 bars
✅ BCP.BK        1198 bars
✅ BDMS.BK       1197 bars
✅ IVL.BK        1198 bars
✅ THAI.BK        284 bars
✅ CPF.BK        1198 bars
✅ IRPC.BK       1198 bars
✅ STECON.BK      458 bars


✅ GPSC.BK       1198 bars
✅ SPRC.BK       1198 bars
✅ BGRIM.BK      1198 bars
✅ GUNKUL.BK     1198 bars
✅ COM7.BK       1198 bars
✅ BEM.BK        1197 bars
✅ TU.BK         1197 bars


✅ CRC.BK        1198 bars
✅ STA.BK        1198 bars
✅ CBG.BK        1198 bars
✅ SCGP.BK       1198 bars
✅ RCL.BK        1198 bars
✅ HMPRO.BK      1198 bars
✅ SMT.BK        1198 bars
✅ PLANB.BK      1198 bars
✅ BJC.BK        1198 bars
✅ RATCH.BK      1198 bars
✅ CK.BK         1197 bars
✅ OR.BK         1198 bars
✅ OSP.BK        1198 bars
✅ ITC.BK         917 bars
✅ BA.BK         1198 bars
✅ BCH.BK        1198 bars
✅ PTG.BK        1198 bars
✅ EA.BK         1198 bars


✅ SPCG.BK       1198 bars
✅ GLOBAL.BK     1198 bars
✅ TFG.BK        1198 bars
✅ TASCO.BK      1198 bars
✅ PSL.BK        1197 bars
✅ BANPU.BK      1146 bars
✅ EPG.BK        1198 bars


✅ CHG.BK        1198 bars
✅ CPAXT.BK       475 bars
✅ WHAUP.BK      1198 bars
✅ DOHOME.BK     1198 bars
✅ SSP.BK        1198 bars
✅ BTS.BK        1197 bars
✅ STPI.BK       1197 bars
✅ JTS.BK        1198 bars
✅ BTG.BK         937 bars
✅ SINGER.BK     1197 bars
✅ FORTH.BK      1198 bars
✅ PRM.BK        1198 bars
✅ MRDIYT.BK      212 bars
✅ ICHI.BK       1198 bars
✅ GST.BK        1198 bars
✅ BCPG.BK       1198 bars
✅ GFPT.BK       1198 bars
✅ JMART.BK      1198 bars
✅ TOA.BK        1198 bars
✅ PR9.BK        1198 bars


...

✅ MEDEZE.BK      468 bars
✅ PM.BK         1197 bars
✅ SKR.BK        1198 bars
✅ SMPC.BK       1197 bars
✅ PSGC.BK       1198 bars
✅ SCGD.BK        666 bars
✅ TFM.BK        1186 bars
✅ SPC.BK        1198 bars

Legacy datasets: 133 / 133

🔎 Legacy validation: PTT.BK


Price,Close,LEGACY_EMA20,LEGACY_EMA50,LEGACY_MACD,LEGACY_MACD_SIGNAL,LEGACY_MACD_HIST,LEGACY_STOCH_K,LEGACY_STOCH_D,LEGACY_BODY_ATR_RATIO,LEGACY_BB_BREAKOUT,LEGACY_SQUEEZE_RELEASE,LEGACY_MACD_BULL,LEGACY_STOCH_BULL,LEGACY_TREND_BULL,LEGACY_CANDLE_BULL,LEGACY_VOLATILITY_BULL,LEGACY_SCORE,LEGACY_SIGNAL
Date,,,,,,,,,,,,,,,,,,
2026-09-25,43.00,41.897873,40.512511,0.339350,0.279223,0.060127,62.500000,45.833333,0.356684,False,False,True,True,True,False,False,3,True
2026-09-28,42.50,41.955218,40.590452,0.280996,0.279814,0.001182,66.666667,59.722222,0.692267,False,False,False,True,True,False,False,2,False
2026-09-29,42.00,41.959483,40.645728,0.162371,0.240666,-0.078295,37.500000,55.555556,0.690373,False,False,False,False,True,False,False,1,False
2026-09-30,40.75,41.844294,40.649817,-0.095455,0.128626,-0.224081,16.666667,40.277778,1.640852,False,False,False,False,False,True,False,1,False
2026-10-01,40.75,41.740076,40.653746,-0.249411,0.002613,-0.252025,10.833333,21.666667,0.000000,False,True,False,False,False,False,True,1,False
2026-10-02,41.50,41.717211,40.686932,-0.226348,-0.073707,-0.152641,20.000000,15.833333,0.643703,False,False,False,True,False,False,False,1,False
2026-10-05,41.50,41.696525,40.718817,-0.202988,-0.116801,-0.086188,33.333333,21.388889,0.000000,False,False,False,True,False,False,False,1,False
2026-10-06,41.50,41.677808,40.749452,-0.180401,-0.138001,-0.042400,40.000000,31.111111,0.331033,False,False,False,True,False,False,False,1,False
2026-10-07,40.50,41.565636,40.739670,-0.302067,-0.192690,-0.109377,31.794872,35.042735,0.618497,False,False,False,False,False,False,False,0,False



📊 Legacy Score Distribution


,bars
LEGACY_SCORE,
0,42339
1,55777
2,34675
3,11139
4,3770
5,761



Bars with Legacy Score >= 3: 15,670
Total usable bars: 148,461
Signal rate: 10.55%


In [ ]:

# ============================================================
# 8) CANONICAL RESEARCH SCORE
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# สร้างคะแนนระบบใหม่เพื่อใช้เปรียบเทียบกับ Legacy Score
#
# CANONICAL INDICATORS
#   DAY MACD      = 4 / 12 / 9
#   STOCHASTIC    = 8 / 3 / 3
#   DAY EMA       = 5 / 10 / 75
#   WEEK EMA      = 5 / 10
#   WEEK MACD     = 4 / 12 / 9
#
# RESEARCH SCORE = 5 POINTS
#
# 1) MACD
#    DAY MACD > Signal
#    AND Histogram > 0
#
# 2) STOCHASTIC
#    K > D
#    AND K < 80
#
# 3) DAY TREND
#    Close > EMA5 > EMA10 > EMA75
#
# 4) EARLY ENTRY
#    Squeeze OR Squeeze Release
#
# 5) VOLUME
#    Volume Ratio >= 1.2
#
# PASS:
#    Score >= 3
#
# IMPORTANT
# ------------------------------------------------------------
# Cell นี้เป็น Research Baseline
# ไม่ใช่ Production Weighted Score
# ============================================================


def calculate_canonical_research_score(df):

    out = df.copy()

    # --------------------------------------------------------
    # 1) DAY MACD 4/12/9
    # --------------------------------------------------------

    out["CANONICAL_MACD_BULL"] = (
        (out["MACD"] > out["MACD_SIGNAL"])
        &
        (out["MACD_HIST"] > 0)
    )


    # --------------------------------------------------------
    # 2) STOCHASTIC 8/3/3
    # --------------------------------------------------------

    out["CANONICAL_STOCH_BULL"] = (
        (out["STOCH_K"] > out["STOCH_D"])
        &
        (out["STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) DAY TREND
    #
    # Close > EMA5 > EMA10 > EMA75
    # --------------------------------------------------------

    out["CANONICAL_TREND_BULL"] = (
        (out["Close"] > out["EMA5"])
        &
        (out["EMA5"] > out["EMA10"])
        &
        (out["EMA10"] > out["EMA75"])
    )


    # --------------------------------------------------------
    # 4) EARLY ENTRY
    #
    # ใช้ Squeeze หรือ Squeeze Release
    #
    # ไม่ใช้ SQUEEZE_RECENT ในคะแนนหลัก
    # เพื่อไม่ให้ signal window กว้างเกินไป
    # --------------------------------------------------------

    out["CANONICAL_EARLY_ENTRY"] = (
        out["SQUEEZE"]
        |
        out["SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # 5) VOLUME CONFIRMATION
    # --------------------------------------------------------

    out["CANONICAL_VOLUME_CONFIRM"] = (
        out["VOLUME_RATIO"] >= 1.2
    )


    # --------------------------------------------------------
    # SCORE COMPONENTS
    # --------------------------------------------------------

    canonical_components = [
        "CANONICAL_MACD_BULL",
        "CANONICAL_STOCH_BULL",
        "CANONICAL_TREND_BULL",
        "CANONICAL_EARLY_ENTRY",
        "CANONICAL_VOLUME_CONFIRM",
    ]

    out["CANONICAL_SCORE"] = (
        out[canonical_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    out["CANONICAL_SIGNAL"] = (
        out["CANONICAL_SCORE"] >= 3
    )


    # --------------------------------------------------------
    # ADDITIONAL INFORMATION
    # --------------------------------------------------------
    #
    # Weekly confirmation ไม่ถูกนับเป็นคะแนนเพิ่ม
    # ใน Research Score
    #
    # เก็บไว้เพื่อใช้วิเคราะห์ภายหลัง
    # --------------------------------------------------------

    out["CANONICAL_WEEKLY_TREND"] = (
        out["W_EMA5"] > out["W_EMA10"]
    )

    out["CANONICAL_WEEKLY_MACD"] = (
        out["W_MACD"] > out["W_MACD_SIGNAL"]
    )

    out["CANONICAL_WEEKLY_CONFIRM"] = (
        out["CANONICAL_WEEKLY_TREND"]
        &
        out["CANONICAL_WEEKLY_MACD"]
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

canonical_data = {}
canonical_errors = {}

print("🧮 Calculating Canonical Research Score...\n")

for symbol, df in feature_data.items():

    try:

        result = calculate_canonical_research_score(df)

        canonical_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        canonical_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Canonical datasets: "
    f"{len(canonical_data)} / {len(feature_data)}"
)
print("====================================")


if canonical_errors:

    print("\n⚠️ Canonical calculation errors:")

    for symbol, error in canonical_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if canonical_data:

    sample_symbol = next(iter(canonical_data))

    sample = canonical_data[sample_symbol]

    print(
        f"\n🔎 Canonical validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",

                "EMA5",
                "EMA10",
                "EMA75",

                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",

                "STOCH_K",
                "STOCH_D",

                "W_EMA5",
                "W_EMA10",

                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",

                "VOLUME_RATIO",

                "SQUEEZE",
                "SQUEEZE_RELEASE",

                "CANONICAL_MACD_BULL",
                "CANONICAL_STOCH_BULL",
                "CANONICAL_TREND_BULL",
                "CANONICAL_EARLY_ENTRY",
                "CANONICAL_VOLUME_CONFIRM",

                "CANONICAL_SCORE",
                "CANONICAL_SIGNAL",

                "CANONICAL_WEEKLY_TREND",
                "CANONICAL_WEEKLY_MACD",
                "CANONICAL_WEEKLY_CONFIRM",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if canonical_data:

    all_canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n📊 Canonical Score Distribution")

    display(
        all_canonical_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )


    signal_count = int(
        (all_canonical_scores >= 3).sum()
    )

    total_bars = len(
        all_canonical_scores
    )

    signal_rate = (
        signal_count / total_bars * 100
        if total_bars > 0
        else 0
    )


    print(
        f"\nBars with Canonical Score >= 3: "
        f"{signal_count:,}"
    )

    print(
        f"Total usable bars: "
        f"{total_bars:,}"
    )

    print(
        f"Canonical signal rate: "
        f"{signal_rate:.2f}%"
    )


# ============================================================
# LEGACY vs CANONICAL QUICK COMPARISON
# ============================================================

if legacy_data and canonical_data:

    legacy_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n====================================")
    print("LEGACY vs CANONICAL")
    print("====================================")

    print(
        f"Legacy signal rate:     "
        f"{(legacy_scores >= 3).mean() * 100:.2f}%"
    )

    print(
        f"Canonical signal rate:  "
        f"{(canonical_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Canonical Research Score...



✅ PTT.BK        1197 bars
✅ KCE.BK        1198 bars
✅ HANA.BK       1198 bars
✅ GULF.BK        352 bars
✅ CPALL.BK      1198 bars
✅ TOP.BK        1196 bars
✅ TRUE.BK       1188 bars
✅ AOT.BK        1197 bars


✅ PTTGC.BK      1198 bars
✅ BCP.BK        1198 bars
✅ BDMS.BK       1197 bars
✅ IVL.BK        1198 bars
✅ THAI.BK        284 bars
✅ CPF.BK        1198 bars
✅ IRPC.BK       1198 bars
✅ STECON.BK      458 bars
✅ GPSC.BK       1198 bars
✅ SPRC.BK       1198 bars
✅ BGRIM.BK      1198 bars
✅ GUNKUL.BK     1198 bars
✅ COM7.BK       1198 bars
✅ BEM.BK        1197 bars
✅ TU.BK         1197 bars
✅ CRC.BK        1198 bars
✅ STA.BK        1198 bars
✅ CBG.BK        1198 bars
✅ SCGP.BK       1198 bars
✅ RCL.BK        1198 bars


✅ HMPRO.BK      1198 bars


✅ SMT.BK        1198 bars
✅ PLANB.BK      1198 bars
✅ BJC.BK        1198 bars
✅ RATCH.BK      1198 bars
✅ CK.BK         1197 bars
✅ OR.BK         1198 bars
✅ OSP.BK        1198 bars
✅ ITC.BK         917 bars
✅ BA.BK         1198 bars
✅ BCH.BK        1198 bars
✅ PTG.BK        1198 bars
✅ EA.BK         1198 bars


...

Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,VOLUME_RATIO,SQUEEZE,SQUEEZE_RELEASE,CANONICAL_MACD_BULL,CANONICAL_STOCH_BULL,CANONICAL_TREND_BULL,CANONICAL_EARLY_ENTRY,CANONICAL_VOLUME_CONFIRM,CANONICAL_SCORE,CANONICAL_SIGNAL,CANONICAL_WEEKLY_TREND,CANONICAL_WEEKLY_MACD,CANONICAL_WEEKLY_CONFIRM
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-09-25,43.00,42.634785,42.367338,39.597054,0.439980,0.325283,0.114698,62.500000,45.833333,41.361318,40.277567,1.691150,1.394042,0.297108,1.478918,False,False,True,True,True,False,True,4,True,True,True,True
2026-09-28,42.50,42.589856,42.391458,39.673447,0.319197,0.324065,-0.004869,66.666667,59.722222,41.907545,40.772554,1.775191,1.470272,0.304920,0.766200,True,False,False,True,False,True,False,2,False,True,True,True
2026-09-29,42.00,42.393238,42.320284,39.734672,0.115156,0.282284,-0.167127,37.500000,55.555556,41.907545,40.772554,1.775191,1.470272,0.304920,0.858396,True,False,False,False,False,True,False,1,False,True,True,True
2026-09-30,40.75,41.845492,42.034778,39.761392,-0.303212,0.165184,-0.468397,16.666667,40.277778,41.907545,40.772554,1.775191,1.470272,0.304920,1.326365,True,False,False,False,False,True,True,2,False,True,True,True
2026-10-01,40.75,41.480328,41.801182,39.787408,-0.496956,0.032756,-0.529712,10.833333,21.666667,41.907545,40.772554,1.775191,1.470272,0.304920,1.110644,False,True,False,False,False,True,False,1,False,True,True,True
2026-10-02,41.50,41.486885,41.746421,39.832476,-0.380120,-0.049819,-0.330301,20.000000,15.833333,41.907545,40.772554,1.775191,1.470272,0.304920,0.884640,False,False,False,True,False,False,False,1,False,True,True,True
2026-10-05,41.50,41.491257,41.701618,39.876358,-0.297412,-0.099338,-0.198074,33.333333,21.388889,41.771697,40.904817,1.339385,1.444094,-0.104709,0.534009,False,False,False,True,False,False,False,1,False,True,False,False
2026-10-06,41.50,41.494171,41.664960,39.919085,-0.237119,-0.126894,-0.110225,40.000000,31.111111,41.771697,40.904817,1.339385,1.444094,-0.104709,0.876104,False,False,False,True,False,False,False,1,False,True,False,False
2026-10-07,40.50,41.162781,41.453149,39.934373,-0.438071,-0.189129,-0.248942,31.794872,35.042735,41.771697,40.904817,1.339385,1.444094,-0.104709,0.682990,False,False,False,False,False,False,False,0,False,True,False,False



📊 Canonical Score Distribution


,bars
CANONICAL_SCORE,
0,22670
1,47929
2,47222
3,22383
4,6976
5,1281



Bars with Canonical Score >= 3: 30,640
Total usable bars: 148,461
Canonical signal rate: 20.64%



LEGACY vs CANONICAL
Legacy signal rate:     10.55%
Canonical signal rate:  20.64%


In [ ]:

# ============================================================
# 9) MONEY MANAGEMENT — FINAL
# ============================================================

INITIAL_CAPITAL = 100_000.0
MAX_RISK_PCT = 0.02
RISK_PER_TRADE = INITIAL_CAPITAL * MAX_RISK_PCT

LOT_SIZE = 100
ATR_STOP_MULT = 1.5
RECENT_LOW_LOOKBACK = 5


print("💰 Money Management Configuration")
print("------------------------------------")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} THB")
print(f"Risk / Trade    : {MAX_RISK_PCT * 100:.2f}%")
print(f"Risk Amount     : {RISK_PER_TRADE:,.2f} THB")
print(f"Lot Size        : {LOT_SIZE:,} shares")
print(f"ATR Stop Mult   : {ATR_STOP_MULT:.2f}x")
print(f"Recent Low      : {RECENT_LOW_LOOKBACK} bars")


# ============================================================
# STOP PRICE
# ============================================================

def calculate_stop_price(
    entry_price,
    atr,
    recent_low,
    atr_multiplier=ATR_STOP_MULT,
):
    if pd.isna(entry_price) or entry_price <= 0:
        return np.nan

    if pd.isna(atr) or atr <= 0:
        return np.nan

    if pd.isna(recent_low) or recent_low <= 0:
        return np.nan

    atr_stop = entry_price - atr * atr_multiplier

    # Conservative stop:
    # choose the LOWER of ATR stop and recent low
    stop_price = min(atr_stop, recent_low)

    if stop_price <= 0:
        return np.nan

    if stop_price >= entry_price:
        return np.nan

    return float(stop_price)


# ============================================================
# POSITION SIZE
# ============================================================

def calculate_position_size(
    capital,
    entry_price,
    stop_price,
    risk_pct=MAX_RISK_PCT,
    lot_size=LOT_SIZE,
):
    if pd.isna(capital) or capital <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_CAPITAL",
        }

    if pd.isna(entry_price) or entry_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_ENTRY",
        }

    if pd.isna(stop_price) or stop_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    if stop_price >= entry_price:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "STOP_NOT_BELOW_ENTRY",
        }

    max_risk = capital * risk_pct
    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": "INVALID_RISK_PER_SHARE",
        }

    # --------------------------------------------------------
    # Limit #1: risk-based position size
    # --------------------------------------------------------

    risk_lots = int(
        (max_risk / risk_per_share) // lot_size
    )

    # --------------------------------------------------------
    # Limit #2: capital-based position size
    # --------------------------------------------------------

    capital_lots = int(
        (capital / entry_price) // lot_size
    )

    # Final lots must satisfy BOTH constraints
    lots = min(risk_lots, capital_lots)

    shares = lots * lot_size

    if shares < lot_size:
        if capital_lots < 1:
            status = "INSUFFICIENT_CAPITAL_FOR_ONE_LOT"
        else:
            status = "ONE_LOT_EXCEEDS_RISK"

        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": status,
        }

    actual_risk = shares * risk_per_share
    position_value = shares * entry_price

    if capital_lots < risk_lots:
        status = "CAPITAL_LIMITED"
    else:
        status = "OK"

    return {
        "shares": int(shares),
        "lots": int(lots),
        "risk_amount": float(actual_risk),
        "risk_per_share": float(risk_per_share),
        "position_value": float(position_value),
        "status": status,
    }


# ============================================================
# COMPLETE TRADE PLAN
# ============================================================

def build_trade_plan(
    capital,
    entry_price,
    atr,
    recent_low,
    risk_pct=MAX_RISK_PCT,
    atr_multiplier=ATR_STOP_MULT,
    lot_size=LOT_SIZE,
):
    stop_price = calculate_stop_price(
        entry_price,
        atr,
        recent_low,
        atr_multiplier,
    )

    if pd.isna(stop_price):
        return {
            "entry_price": (
                float(entry_price)
                if pd.notna(entry_price)
                else np.nan
            ),
            "stop_price": np.nan,
            "shares": 0,
            "lots": 0,
            "risk_per_share": np.nan,
            "risk_amount": 0.0,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    position = calculate_position_size(
        capital,
        entry_price,
        stop_price,
        risk_pct,
        lot_size,
    )

    return {
        "entry_price": float(entry_price),
        "stop_price": float(stop_price),
        "shares": position["shares"],
        "lots": position["lots"],
        "risk_per_share": position["risk_per_share"],
        "risk_amount": position["risk_amount"],
        "position_value": position["position_value"],
        "status": position["status"],
    }


# ============================================================
# RECENT LOW
# ============================================================

for symbol, df in canonical_data.items():
    df["RECENT_LOW_5"] = (
        df["Low"]
        .rolling(
            window=RECENT_LOW_LOOKBACK,
            min_periods=RECENT_LOW_LOOKBACK,
        )
        .min()
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n====================================")
print("🧪 Money Management Validation")
print("====================================")


# ------------------------------------------------------------
# Test 1: Normal trade
# ------------------------------------------------------------

test_entry = 50.00
test_atr = 1.50
test_recent_low = 47.00

test_plan = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 1: Normal Trade")
print("------------------------------------")

for key, value in test_plan.items():
    if isinstance(value, float):
        print(f"{key:<20}: {value:,.4f}")
    else:
        print(f"{key:<20}: {value}")


# ------------------------------------------------------------
# Test 2: Stop validation
# ------------------------------------------------------------

test_entry = 10.00
test_atr = 0.50
test_recent_low = 9.80

test_plan_2 = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 2: Stop Validation")
print("------------------------------------")
print(f"Entry : {test_plan_2['entry_price']:.4f}")
print(f"Stop  : {test_plan_2['stop_price']:.4f}")

if pd.notna(test_plan_2["stop_price"]):
    print(
        "Valid : "
        f"{test_plan_2['stop_price'] < test_plan_2['entry_price']}"
    )
else:
    print("Valid : False")


# ------------------------------------------------------------
# Test 3: One lot risk check
# ------------------------------------------------------------

test_entry = 80.00
test_stop = 60.00

test_position = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 3: One Lot Risk Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position['risk_per_share']:.2f}")
print(f"Shares      : {test_position['shares']:,}")
print(f"Lots        : {test_position['lots']:,}")
print(f"Risk Amount : {test_position['risk_amount']:,.2f}")
print(f"Position    : {test_position['position_value']:,.2f}")
print(f"Status      : {test_position['status']}")


# ------------------------------------------------------------
# Test 4: Capital limit
# ------------------------------------------------------------

test_entry = 1.00
test_stop = 0.99

test_position_4 = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 4: Capital Limit Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position_4['risk_per_share']:.4f}")
print(f"Shares      : {test_position_4['shares']:,}")
print(f"Lots        : {test_position_4['lots']:,}")
print(f"Risk Amount : {test_position_4['risk_amount']:,.2f}")
print(f"Position    : {test_position_4['position_value']:,.2f}")
print(f"Status      : {test_position_4['status']}")

assert (
    test_position_4["position_value"]
    <= INITIAL_CAPITAL + 1e-9
), "Position value exceeds capital"


# ============================================================
# REAL DATA SANITY CHECK
# ============================================================

print("\n====================================")
print("🔎 Real Data Sanity Check")
print("====================================")

if canonical_data:
    sample_symbol = next(iter(canonical_data))
    sample = canonical_data[sample_symbol]

    valid_rows = (
        sample[
            ["Close", "ATR14", "RECENT_LOW_5"]
        ]
        .dropna()
        .tail(3)
    )

    print(f"\nSample: {sample_symbol}")
    display(valid_rows)


# ============================================================
# FINAL CHECKS
# ============================================================

assert INITIAL_CAPITAL > 0
assert 0 < MAX_RISK_PCT <= 0.05
assert LOT_SIZE > 0
assert ATR_STOP_MULT > 0
assert RECENT_LOW_LOOKBACK > 0

print("\n====================================")
print("✅ Money Management Cell Ready")
print("====================================")
print(f"Maximum risk per trade: {RISK_PER_TRADE:,.2f} THB")
print("Stop rule: min(Entry - ATR × 1.5, Recent Low)")
print("Position size: whole 100-share lots, rounded DOWN")
print("Position limit: Risk limit + Capital limit")
print("Target prices: calculated later from actual Entry")

💰 Money Management Configuration
------------------------------------
Initial Capital : 100,000.00 THB
Risk / Trade    : 2.00%
Risk Amount     : 2,000.00 THB
Lot Size        : 100 shares
ATR Stop Mult   : 1.50x
Recent Low      : 5 bars



🧪 Money Management Validation

Test 1: Normal Trade
------------------------------------
entry_price         : 50.0000
stop_price          : 47.0000
shares              : 600
lots                : 6
risk_per_share      : 3.0000
risk_amount         : 1,800.0000
position_value      : 30,000.0000
status              : OK

Test 2: Stop Validation
------------------------------------
Entry : 10.0000
Stop  : 9.2500
Valid : True

Test 3: One Lot Risk Check
------------------------------------
Entry       : 80.00
Stop        : 60.00
Risk / Share: 20.00
Shares      : 100
Lots        : 1
Risk Amount : 2,000.00
Position    : 8,000.00
Status      : OK

Test 4: Capital Limit Check
------------------------------------
Entry       : 1.00
Stop        : 0.99
Risk / Share: 0.0100
Shares      : 100,000
Lots        : 1,000
Risk Amount : 1,000.00
Position    : 100,000.00
Status      : CAPITAL_LIMITED

🔎 Real Data Sanity Check

Sample: PTT.BK


Price,Close,ATR14,RECENT_LOW_5
Date,,,
2026-10-06,41.5,0.755213,40.0
2026-10-07,40.5,0.808412,40.0
2026-10-08,40.0,0.786383,40.0



✅ Money Management Cell Ready
Maximum risk per trade: 2,000.00 THB
Stop rule: min(Entry - ATR × 1.5, Recent Low)
Position size: whole 100-share lots, rounded DOWN
Position limit: Risk limit + Capital limit
Target prices: calculated later from actual Entry


In [ ]:

# ============================================================
# 10) CURRENT SCANNER
# ============================================================

print("🔎 CURRENT SCANNER")
print("=" * 90)


def build_current_scan_row(symbol, df):
    if df is None or df.empty:
        return {
            "Symbol": symbol,
            "Date": pd.NaT,
            "Close": np.nan,
            "Score": np.nan,
            "Signal": False,
            "WeeklyConfirm": False,
            "MACD": np.nan,
            "MACD_Signal": np.nan,
            "MACD_Hist": np.nan,
            "EMA5": np.nan,
            "EMA10": np.nan,
            "EMA75": np.nan,
            "W_EMA5": np.nan,
            "W_EMA10": np.nan,
            "W_MACD": np.nan,
            "W_MACD_Signal": np.nan,
            "ATR14": np.nan,
            "RecentLow5": np.nan,
            "Stop": np.nan,
            "RiskPerShare": np.nan,
            "Shares": 0,
            "Lots": 0,
            "RiskAmount": 0.0,
            "PositionValue": 0.0,
            "Target1": np.nan,
            "Target2": np.nan,
            "Status": "NO_DATA",
        }

    row = df.iloc[-1]

    entry_price = row["Close"]
    atr = row["ATR14"]
    recent_low = row["RECENT_LOW_5"]

    trade_plan = build_trade_plan(
        INITIAL_CAPITAL,
        entry_price,
        atr,
        recent_low,
        atr_multiplier=PRODUCTION_ATR_MULT,
    )

    score = row.get("CANONICAL_SCORE", np.nan)
    signal = (
        symbol in PRODUCTION_SYMBOLS
        and float(row.get("CANONICAL_SCORE", 0)) >= PRODUCTION_MIN_SCORE
        and (
            not PRODUCTION_WEEKLY_CONFIRM
            or bool(row.get("CANONICAL_WEEKLY_CONFIRM", False))
        )
    )

    weekly_trend = bool(
        row.get("CANONICAL_WEEKLY_TREND", False)
    )

    weekly_macd = bool(
        row.get("CANONICAL_WEEKLY_MACD", False)
    )

    weekly_confirm = bool(
        row.get("CANONICAL_WEEKLY_CONFIRM", False)
    )

    target1 = (
        float(entry_price * 1.07)
        if pd.notna(entry_price)
        else np.nan
    )

    target2 = (
        float(entry_price * (1 + PRODUCTION_TARGET2_PCT))
        if pd.notna(entry_price)
        else np.nan
    )

    return {
        "Symbol": symbol,
        "Date": df.index[-1],
        "Close": float(entry_price),
        "Score": score,
        "Signal": signal,
        "WeeklyTrend": weekly_trend,
        "WeeklyMACD": weekly_macd,
        "WeeklyConfirm": weekly_confirm,
        "MACD": row["MACD"],
        "MACD_Signal": row["MACD_SIGNAL"],
        "MACD_Hist": row["MACD_HIST"],
        "EMA5": row["EMA5"],
        "EMA10": row["EMA10"],
        "EMA75": row["EMA75"],
        "W_EMA5": row["W_EMA5"],
        "W_EMA10": row["W_EMA10"],
        "W_MACD": row["W_MACD"],
        "W_MACD_Signal": row["W_MACD_SIGNAL"],
        "ATR14": row["ATR14"],
        "RecentLow5": row["RECENT_LOW_5"],
        "Stop": trade_plan["stop_price"],
        "RiskPerShare": trade_plan["risk_per_share"],
        "Shares": trade_plan["shares"],
        "Lots": trade_plan["lots"],
        "RiskAmount": trade_plan["risk_amount"],
        "PositionValue": trade_plan["position_value"],
        "Target1": target1,
        "Target2": target2,
        "Status": trade_plan["status"],
    }


# ============================================================
# BUILD CURRENT SCAN
# ============================================================

current_scan_rows = []

for symbol, df in canonical_data.items():
    current_scan_rows.append(
        build_current_scan_row(symbol, df)
    )

current_scan = pd.DataFrame(current_scan_rows)


# ============================================================
# SORT
# ============================================================

current_scan = current_scan.sort_values(
    by=["Signal", "Score", "WeeklyConfirm"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)


# ============================================================
# DISPLAY
# ============================================================

print("\n📊 Current Market Snapshot")
print("-" * 90)

display(
    current_scan[
        [
            "Symbol",
            "Date",
            "Close",
            "Score",
            "Signal",
            "WeeklyConfirm",
            "ATR14",
            "Stop",
            "Shares",
            "RiskAmount",
            "Target1",
            "Target2",
            "Status",
        ]
    ]
)


# ============================================================
# SIGNAL SUMMARY
# ============================================================

signal_scan = current_scan[
    current_scan["Signal"] == True
].copy()

weekly_confirmed = current_scan[
    current_scan["WeeklyConfirm"] == True
].copy()

print("\n====================================")
print("📈 CURRENT SCANNER SUMMARY")
print("====================================")

print(f"Total symbols       : {len(current_scan)}")
print(f"Production universe : {len(PRODUCTION_SYMBOLS)}")
print(f"Production min score: {PRODUCTION_MIN_SCORE}")
print(f"Production ATR mult : {PRODUCTION_ATR_MULT:.2f}x")
print(f"Production target2  : {PRODUCTION_TARGET2_PCT * 100:.1f}%")
print(f"Canonical signals   : {len(signal_scan)}")
print(f"Weekly confirmed    : {len(weekly_confirmed)}")

if len(current_scan) > 0:
    signal_rate = (
        len(signal_scan) / len(current_scan) * 100
    )
else:
    signal_rate = 0.0

print(f"Signal rate         : {signal_rate:.2f}%")


# ============================================================
# TOP SIGNALS
# ============================================================

print("\n====================================")
print("🏆 TOP CURRENT SIGNALS")
print("====================================")

if signal_scan.empty:
    print("No current Canonical signals.")
else:
    display(
        signal_scan[
            [
                "Symbol",
                "Date",
                "Close",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "Stop",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].head(10)
    )


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(current_scan) == len(canonical_data)

assert current_scan["Score"].dropna().between(0, 5).all()

assert (
    current_scan["RiskAmount"].fillna(0)
    <= RISK_PER_TRADE + 1e-9
).all()

assert (
    current_scan["PositionValue"].fillna(0)
    <= INITIAL_CAPITAL + 1e-9
).all()

print("\n====================================")
print("✅ Current Scanner Cell Ready")
print("====================================")

🔎 CURRENT SCANNER

📊 Current Market Snapshot
------------------------------------------------------------------------------------------


,Symbol,Date,Close,Score,Signal,WeeklyConfirm,ATR14,Stop,Shares,RiskAmount,Target1,Target2,Status
0,BCH.BK,2026-10-08,11.7,5,True,False,0.309921,11.000000,2800,1959.999466,12.519,13.455,OK
1,CHG.BK,2026-10-08,1.6,5,True,False,0.041161,1.510000,22200,1998.000741,1.712,1.840,OK
2,SIS.BK,2026-10-08,24.4,5,True,False,0.520629,23.700001,2800,1959.996796,26.108,28.060,OK
3,BGRIM.BK,2026-10-08,18.9,4,True,False,0.656192,18.112569,2500,1968.576726,20.223,21.735,OK
4,MGC.BK,2026-10-08,7.0,4,True,False,0.333435,6.450000,3600,1980.000687,7.490,8.050,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,UVAN.BK,2026-10-08,13.5,0,False,False,0.242276,13.209269,6800,1976.970879,14.445,15.525,OK
129,THG.BK,2026-10-08,6.7,0,False,False,0.154028,6.515166,10800,1996.202245,7.169,7.705,OK
130,VIH.BK,2026-10-08,8.3,0,False,False,0.184577,8.078508,9000,1993.428969,8.881,9.545,OK
131,SAPPE.BK,2026-10-08,30.5,0,False,False,0.676228,29.688526,2400,1947.536866,32.635,35.075,OK



📈 CURRENT SCANNER SUMMARY
Total symbols       : 133
Production universe : 133
Production min score: 4
Production ATR mult : 1.20x
Production target2  : 15.0%
Canonical signals   : 8
Weekly confirmed    : 26
Signal rate         : 6.02%

🏆 TOP CURRENT SIGNALS


,Symbol,Date,Close,Score,WeeklyConfirm,ATR14,Stop,Shares,RiskAmount,Target1,Target2
0,BCH.BK,2026-10-08,11.70,5,False,0.309921,11.000000,2800,1959.999466,12.5190,13.4550
1,CHG.BK,2026-10-08,1.60,5,False,0.041161,1.510000,22200,1998.000741,1.7120,1.8400
2,SIS.BK,2026-10-08,24.40,5,False,0.520629,23.700001,2800,1959.996796,26.1080,28.0600
3,BGRIM.BK,2026-10-08,18.90,4,False,0.656192,18.112569,2500,1968.576726,20.2230,21.7350
4,MGC.BK,2026-10-08,7.00,4,False,0.333435,6.450000,3600,1980.000687,7.4900,8.0500
5,STGT.BK,2026-10-08,10.70,4,False,0.308693,10.200000,4000,2000.000000,11.4490,12.3050
6,KCG.BK,2026-10-08,10.20,4,False,0.219531,9.850000,5700,1994.996738,10.9140,11.7300
7,SCGD.BK,2026-10-08,5.45,4,False,0.090010,5.341988,18300,1976.622801,5.8315,6.2675



✅ Current Scanner Cell Ready


In [ ]:
# ============================================================
# 10A) PRODUCTION SIGNAL JOURNAL
# ============================================================
#
# Audit-only persistence for the frozen Production scanner.
# This cell does NOT change signal logic, thresholds, or sizing rules.
# ReferenceEntryPrice is the signal-day Close used only as a planning reference.
# It is NOT an executed entry price and does not use future data.
# ============================================================

from pathlib import Path

JOURNAL_PATH = Path("/content/production_signal_journal.csv")

JOURNAL_COLUMNS = [
    "RunTimestampUTC",
    "ScanDate",
    "Symbol",
    "Score",
    "Signal",
    "WeeklyConfirm",
    "SignalClose",
    "ReferenceEntryPrice",
    "ATR14",
    "RecentLow5",
    "StopPrice",
    "Target1",
    "Target2",
    "PlannedShares",
    "PlannedLots",
    "PlannedRiskAmount",
    "PlannedPositionValue",
    "Status",
    "ProductionMinScore",
    "ProductionWeeklyConfirm",
    "ProductionATRMult",
    "ProductionTarget2Pct",
    "ProductionMaxHoldingDays",
]

journal = current_scan.copy()
journal = journal[journal["Symbol"].isin(PRODUCTION_SYMBOLS)].copy()

run_timestamp = pd.Timestamp.now(tz="UTC").isoformat()
journal["RunTimestampUTC"] = run_timestamp
journal["ScanDate"] = pd.to_datetime(journal["Date"]).dt.strftime("%Y-%m-%d")
journal["ReferenceEntryPrice"] = journal["Close"]
journal["PlannedShares"] = journal["Shares"].fillna(0).astype(int)
journal["PlannedLots"] = journal["Lots"].fillna(0).astype(int)
journal["PlannedRiskAmount"] = journal["RiskAmount"].fillna(0.0)
journal["PlannedPositionValue"] = journal["PositionValue"].fillna(0.0)

journal = journal.rename(columns={
    "Close": "SignalClose",
    "Stop": "StopPrice",
})

journal["ProductionMinScore"] = PRODUCTION_MIN_SCORE
journal["ProductionWeeklyConfirm"] = PRODUCTION_WEEKLY_CONFIRM
journal["ProductionATRMult"] = PRODUCTION_ATR_MULT
journal["ProductionTarget2Pct"] = PRODUCTION_TARGET2_PCT
journal["ProductionMaxHoldingDays"] = PRODUCTION_MAX_HOLDING_DAYS

journal = journal[JOURNAL_COLUMNS].copy()

if JOURNAL_PATH.exists():
    existing = pd.read_csv(JOURNAL_PATH)
    current_scan_date = journal["ScanDate"].max() if not journal.empty else None

    # If the universe changes during the same scan date, replace that day's
    # snapshot rather than retaining symbols from the previous universe.
    if current_scan_date is not None:
        existing = existing[
            ~(
                existing["ScanDate"].eq(current_scan_date)
                & ~existing["Symbol"].isin(PRODUCTION_SYMBOLS)
            )
        ].copy()

    journal = pd.concat([existing, journal], ignore_index=True)

journal = journal.drop_duplicates(
    subset=["ScanDate", "Symbol"],
    keep="last",
).sort_values(
    ["ScanDate", "Symbol"]
).reset_index(drop=True)

JOURNAL_PATH.parent.mkdir(parents=True, exist_ok=True)
journal.to_csv(JOURNAL_PATH, index=False)

print("\n====================================")
print("📝 PRODUCTION SIGNAL JOURNAL")
print("====================================")
print(f"Journal path      : {JOURNAL_PATH}")
print(f"Production symbols: {len(PRODUCTION_SYMBOLS)}")
print(f"Rows stored       : {len(journal)}")
print(
    f"Latest scan date  : "
    f"{journal['ScanDate'].max() if not journal.empty else 'N/A'}"
)
print(
    f"Current signals   : "
    f"{int(journal[journal['ScanDate'] == journal['ScanDate'].max()]['Signal'].sum()) if not journal.empty else 0}"
)
print("Actual execution  : not recorded until a real fill is available")

assert set(PRODUCTION_SYMBOLS) == set(
    journal[
        journal["ScanDate"] == journal["ScanDate"].max()
    ]["Symbol"]
)
assert not journal.duplicated(
    ["ScanDate", "Symbol"]
).any()

print("JOURNAL_VALID")



📝 PRODUCTION SIGNAL JOURNAL
Journal path      : /content/production_signal_journal.csv
Production symbols: 133
Rows stored       : 142
Latest scan date  : 2026-10-08
Current signals   : 8
Actual execution  : not recorded until a real fill is available
JOURNAL_VALID


In [ ]:

# ============================================================
# 11) HISTORICAL SIGNAL GENERATION
# ============================================================

print("📡 HISTORICAL SIGNAL GENERATION")
print("=" * 90)


SIGNAL_MIN_SCORE = 3


def generate_historical_signals(
    symbol,
    df,
    min_score=SIGNAL_MIN_SCORE,
):
    if df is None or df.empty:
        return pd.DataFrame()

    data = df.copy()

    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Close",
        "Open",
        "High",
        "Low",
        "ATR14",
        "RECENT_LOW_5",
        "CANONICAL_SCORE",
        "CANONICAL_SIGNAL",
        "CANONICAL_WEEKLY_TREND",
        "CANONICAL_WEEKLY_MACD",
        "CANONICAL_WEEKLY_CONFIRM",
    ]

    missing = [
        col for col in required
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Signal rows
    # --------------------------------------------------------

    signal_mask = (
        data["CANONICAL_SIGNAL"].fillna(False)
        & (data["CANONICAL_SCORE"] >= min_score)
    )

    signal_dates = data.index[signal_mask]

    rows = []

    for signal_date in signal_dates:
        signal_pos = data.index.get_loc(signal_date)

        # Need next trading day for entry
        if signal_pos + 1 >= len(data):
            continue

        entry_date = data.index[signal_pos + 1]

        signal_row = data.iloc[signal_pos]
        entry_row = data.iloc[signal_pos + 1]

        signal_close = signal_row["Close"]
        signal_atr = signal_row["ATR14"]
        recent_low = signal_row["RECENT_LOW_5"]

        # ----------------------------------------------------
        # Stop is calculated from information available
        # at SIGNAL DATE.
        # ----------------------------------------------------

        stop_price = calculate_stop_price(
            signal_close,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Entry is NEXT DAY OPEN.
        # ----------------------------------------------------

        entry_price = entry_row["Open"]

        # ----------------------------------------------------
        # If entry is invalid, skip.
        # ----------------------------------------------------

        if pd.isna(entry_price) or entry_price <= 0:
            continue

        if pd.isna(stop_price) or stop_price <= 0:
            continue

        # ----------------------------------------------------
        # Position sizing uses actual next-day entry.
        # ----------------------------------------------------

        trade_plan = build_trade_plan(
            INITIAL_CAPITAL,
            entry_price,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Targets based on ACTUAL entry.
        # ----------------------------------------------------

        target1 = entry_price * 1.07
        target2 = entry_price * 1.15

        rows.append({
            "Symbol": symbol,

            "SignalDate": signal_date,
            "EntryDate": entry_date,

            "SignalClose": float(signal_close),
            "EntryPrice": float(entry_price),

            "Score": int(signal_row["CANONICAL_SCORE"]),

            "WeeklyTrend": bool(
                signal_row["CANONICAL_WEEKLY_TREND"]
            ),

            "WeeklyMACD": bool(
                signal_row["CANONICAL_WEEKLY_MACD"]
            ),

            "WeeklyConfirm": bool(
                signal_row["CANONICAL_WEEKLY_CONFIRM"]
            ),

            "ATR14": float(signal_atr),

            "RecentLow5": float(recent_low),

            "StopPrice": float(
                trade_plan["stop_price"]
            ),

            "RiskPerShare": float(
                trade_plan["risk_per_share"]
            ),

            "Shares": int(
                trade_plan["shares"]
            ),

            "Lots": int(
                trade_plan["lots"]
            ),

            "RiskAmount": float(
                trade_plan["risk_amount"]
            ),

            "PositionValue": float(
                trade_plan["position_value"]
            ),

            "Target1": float(target1),
            "Target2": float(target2),

            "EntryOpen": float(entry_row["Open"]),
            "EntryHigh": float(entry_row["High"]),
            "EntryLow": float(entry_row["Low"]),
            "EntryClose": float(entry_row["Close"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# GENERATE SIGNALS FOR ALL SYMBOLS
# ============================================================

historical_signals_list = []

for symbol, df in canonical_data.items():

    signals = generate_historical_signals(
        symbol,
        df,
        min_score=SIGNAL_MIN_SCORE,
    )

    if not signals.empty:
        historical_signals_list.append(signals)


if historical_signals_list:
    historical_signals = pd.concat(
        historical_signals_list,
        ignore_index=True,
    )
else:
    historical_signals = pd.DataFrame()


# ============================================================
# SORT
# ============================================================

if not historical_signals.empty:
    historical_signals = historical_signals.sort_values(
        ["SignalDate", "Score"],
        ascending=[True, False],
    ).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 HISTORICAL SIGNAL SUMMARY")
print("====================================")

print(
    f"Total signals : {len(historical_signals):,}"
)

if not historical_signals.empty:

    print(
        f"Symbols       : "
        f"{historical_signals['Symbol'].nunique()}"
    )

    print(
        f"Score >= 3    : "
        f"{(historical_signals['Score'] >= 3).sum():,}"
    )

    print(
        f"Weekly confirm: "
        f"{historical_signals['WeeklyConfirm'].sum():,}"
    )

    print(
        f"Date range    : "
        f"{historical_signals['SignalDate'].min().date()} "
        f"→ "
        f"{historical_signals['SignalDate'].max().date()}"
    )


# ============================================================
# LOOKAHEAD SANITY CHECK
# ============================================================

print("\n====================================")
print("🔬 LOOKAHEAD SANITY CHECK")
print("====================================")

if not historical_signals.empty:

    invalid_entry = (
        historical_signals["EntryDate"]
        <= historical_signals["SignalDate"]
    ).sum()

    invalid_price = (
        historical_signals["EntryPrice"] <= 0
    ).sum()

    invalid_stop = (
        historical_signals["StopPrice"]
        >= historical_signals["SignalClose"]
    ).sum()

    print(
        f"EntryDate <= SignalDate : {invalid_entry}"
    )

    print(
        f"Invalid EntryPrice      : {invalid_price}"
    )

    print(
        f"Invalid Stop             : {invalid_stop}"
    )

    assert invalid_entry == 0
    assert invalid_price == 0
    assert invalid_stop == 0


# ============================================================
# SAMPLE SIGNALS
# ============================================================

print("\n====================================")
print("🔎 SAMPLE HISTORICAL SIGNALS")
print("====================================")

if historical_signals.empty:
    print("No historical signals found.")
else:
    display(
        historical_signals[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "SignalClose",
                "EntryPrice",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "StopPrice",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].tail(15)
    )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n====================================")
print("✅ Historical Signal Generation Ready")
print("====================================")

📡 HISTORICAL SIGNAL GENERATION



📊 HISTORICAL SIGNAL SUMMARY
Total signals : 30,541
Symbols       : 133
Score >= 3    : 30,541
Weekly confirm: 10,293
Date range    : 2021-11-05 → 2026-10-07

🔬 LOOKAHEAD SANITY CHECK
EntryDate <= SignalDate : 0
Invalid EntryPrice      : 0
Invalid Stop             : 0

🔎 SAMPLE HISTORICAL SIGNALS


,Symbol,SignalDate,EntryDate,SignalClose,EntryPrice,Score,WeeklyConfirm,ATR14,StopPrice,Shares,RiskAmount,Target1,Target2
30526,MC.BK,2026-10-07,2026-10-08,11.400000,11.400000,3,False,0.145730,11.181405,8700,1901.773068,12.198000,13.110000
30527,AAI.BK,2026-10-07,2026-10-08,4.280000,4.260000,3,True,0.074768,4.120000,14200,1988.004875,4.558200,4.899000
30528,XO.BK,2026-10-07,2026-10-08,19.200001,19.200001,3,False,0.573997,17.600000,1200,1920.000458,20.544001,22.080001
30529,KSL.BK,2026-10-07,2026-10-08,1.830000,1.810000,3,False,0.066135,1.640000,11700,1988.999498,1.936700,2.081500
30530,KLINIQ.BK,2026-10-07,2026-10-08,29.250000,29.250000,3,False,0.782015,27.250000,1000,2000.000000,31.297500,33.637500
30531,PCE.BK,2026-10-07,2026-10-08,2.180000,2.180000,3,False,0.048675,2.106988,27300,1993.222944,2.332600,2.507000
30532,SABINA.BK,2026-10-07,2026-10-08,15.300000,15.400000,3,False,0.169160,15.146260,6400,1623.934621,16.478000,17.710000
30533,NSL.BK,2026-10-07,2026-10-08,21.600000,21.600000,3,False,0.349926,20.500000,1800,1980.000687,23.112000,24.840000
30534,HTC.BK,2026-10-07,2026-10-08,15.100000,15.100000,3,False,0.207526,14.788712,6400,1992.247179,16.157000,17.365000
30535,SAMART.BK,2026-10-07,2026-10-08,5.600000,5.550000,3,False,0.128841,5.356738,10300,1990.600889,5.938500,6.382500



✅ Historical Signal Generation Ready


In [ ]:

# ============================================================
# 11.2) FIX HISTORICAL SIGNALS
# PRESERVE ORIGINAL SIGNAL SET
# FIX STOP FROM SIGNAL-DATE DATA
# ============================================================

print("🔧 FIXING HISTORICAL SIGNALS")
print("=" * 90)

old_signals = historical_signals.copy()

print(f"Original signals : {len(old_signals):,}")

fixed_rows = []
excluded_invalid_stop = 0

for _, old in old_signals.iterrows():

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    if symbol not in feature_data:
        continue

    df = feature_data[symbol].sort_index()

    if signal_date not in df.index:
        continue

    if entry_date not in df.index:
        continue

    # RECENT_LOW_5 is created on canonical_data in Cell 9.
    # feature_data may not carry that derived column, so rebuild it
    # here from signal-date history if it is missing.
    if "RECENT_LOW_5" not in df.columns:
        df = df.copy()
        df["RECENT_LOW_5"] = (
            df["Low"]
            .rolling(
                window=RECENT_LOW_LOOKBACK,
                min_periods=RECENT_LOW_LOOKBACK,
            )
            .min()
        )

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    signal_atr = float(signal_row["ATR14"])
    recent_low_value = signal_row["RECENT_LOW_5"]

    if not np.isfinite(recent_low_value) or recent_low_value <= 0:
        continue

    recent_low = float(recent_low_value)
    entry_price = float(entry_row["Open"])

    # --------------------------------------------------------
    # FIXED STOP FROM SIGNAL DATE
    # --------------------------------------------------------

    stop_price = calculate_stop_price(
        entry_price=signal_close,
        atr=signal_atr,
        recent_low=recent_low,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        excluded_invalid_stop += 1
        continue

    # A long trade cannot have its protective stop at or above
    # the actual next-day entry price. This can happen after a
    # gap-down between signal date and entry date. Exclude that
    # trade from the executable historical set rather than
    # manufacturing a stop or silently changing the strategy.
    if entry_price <= stop_price:
        excluded_invalid_stop += 1
        continue

    # --------------------------------------------------------
    # POSITION SIZE
    # Uses actual entry price + fixed stop
    # --------------------------------------------------------

    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        shares = 0
        lots = 0
        risk_amount = 0.0
        position_value = 0.0
        status = "INVALID_STOP"

    else:
        max_risk = INITIAL_CAPITAL * MAX_RISK_PCT

        risk_shares = int(max_risk / risk_per_share)
        capital_shares = int(INITIAL_CAPITAL / entry_price)

        shares = min(risk_shares, capital_shares)

        shares = (shares // LOT_SIZE) * LOT_SIZE
        lots = shares // LOT_SIZE

        risk_amount = risk_per_share * shares
        position_value = entry_price * shares

        status = "OK" if shares > 0 else "CAPITAL_LIMITED"

    # --------------------------------------------------------
    # TARGETS
    # --------------------------------------------------------

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    fixed_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": signal_atr,
        "RecentLow5": recent_low,
        "StopPrice": stop_price,
        "Shares": shares,
        "Lots": lots,
        "RiskAmount": risk_amount,
        "PositionValue": position_value,
        "Target1": target1,
        "Target2": target2,
        "Status": status,
        "EntryOpen": float(entry_row["Open"]),
        "EntryHigh": float(entry_row["High"]),
        "EntryLow": float(entry_row["Low"]),
        "EntryClose": float(entry_row["Close"]),
    })


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if not fixed_rows:
    raise RuntimeError(
        "No signals were rebuilt. Stop here and inspect Cell 11."
    )


historical_signals = pd.DataFrame(fixed_rows)

historical_signals = (
    historical_signals
    .sort_values(["SignalDate", "Symbol"])
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 REBUILD SUMMARY")
print("====================================")

print(f"Original signals : {len(old_signals):,}")
print(f"Fixed signals    : {len(historical_signals):,}")
print(
    f"Difference       : "
    f"{len(historical_signals) - len(old_signals):+,}"
)


# ============================================================
# VALIDATION
# ============================================================

entry_before_signal = (
    historical_signals["EntryDate"]
    < historical_signals["SignalDate"]
).sum()

invalid_entry = (
    ~np.isfinite(historical_signals["EntryPrice"])
    | (historical_signals["EntryPrice"] <= 0)
).sum()

invalid_stop = (
    ~np.isfinite(historical_signals["StopPrice"])
    | (historical_signals["StopPrice"] <= 0)
).sum()

stop_ge_entry = (
    historical_signals["StopPrice"]
    >= historical_signals["EntryPrice"]
).sum()

risk_over_limit = (
    historical_signals["RiskAmount"]
    > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9
).sum()

capital_over_limit = (
    historical_signals["PositionValue"]
    > INITIAL_CAPITAL + 1e-9
).sum()


print("\n====================================")
print("🔍 VALIDATION")
print("====================================")

print(f"Entry before Signal : {entry_before_signal:,}")
print(f"Invalid EntryPrice  : {invalid_entry:,}")
print(f"Invalid Stop        : {invalid_stop:,}")
print(f"Stop >= Entry       : {stop_ge_entry:,}")
print(f"Risk > 2%           : {risk_over_limit:,}")
print(f"Capital > 100k      : {capital_over_limit:,}")


# ============================================================
# STOP CONSISTENCY
# ============================================================

check = historical_signals.head(10).copy()

check["ExpectedStop"] = check.apply(
    lambda r: calculate_stop_price(
        r["SignalClose"],
        r["ATR14"],
        r["RecentLow5"],
    ),
    axis=1,
)

check["StopMatch"] = np.isclose(
    check["StopPrice"],
    check["ExpectedStop"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n====================================")
print("🔒 STOP CONSISTENCY")
print("====================================")

display(
    check[
        [
            "Symbol",
            "SignalDate",
            "SignalClose",
            "EntryPrice",
            "ATR14",
            "RecentLow5",
            "StopPrice",
            "ExpectedStop",
            "StopMatch",
        ]
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n====================================")

assert len(historical_signals) + excluded_invalid_stop == len(old_signals), (
    "Signal accounting mismatch during rebuild."
)
assert entry_before_signal == 0, "EntryDate is before SignalDate."
assert invalid_entry == 0, "Invalid EntryPrice found."
assert invalid_stop == 0, "Invalid StopPrice found."
assert stop_ge_entry == 0, "StopPrice >= EntryPrice found."
assert risk_over_limit == 0, "Risk exceeds configured limit."
assert capital_over_limit == 0, "Position value exceeds initial capital."
assert check["StopMatch"].all(), "Stop-price consistency check failed."

print("✅ HISTORICAL SIGNALS FIXED")
print("CELL 11.2 PASSED")
print("====================================")

🔧 FIXING HISTORICAL SIGNALS
Original signals : 30,541



📊 REBUILD SUMMARY
Original signals : 30,541
Fixed signals    : 30,431
Difference       : -110

🔍 VALIDATION
Entry before Signal : 0
Invalid EntryPrice  : 0
Invalid Stop        : 0
Stop >= Entry       : 0
Risk > 2%           : 0
Capital > 100k      : 0

🔒 STOP CONSISTENCY


,Symbol,SignalDate,SignalClose,EntryPrice,ATR14,RecentLow5,StopPrice,ExpectedStop,StopMatch
0,AH.BK,2021-11-05,21.799999,22.000000,0.618439,21.200001,20.872340,20.872340,True
1,AOT.BK,2021-11-05,65.500000,67.000000,1.111593,64.000000,63.832611,63.832611,True
2,BGRIM.BK,2021-11-05,43.750000,44.000000,0.949798,41.500000,41.500000,41.500000,True
3,CBG.BK,2021-11-05,125.000000,125.000000,2.792900,121.500000,120.810650,120.810650,True
4,CKP.BK,2021-11-05,5.000000,5.050000,0.125479,4.980000,4.811782,4.811782,True
5,CPALL.BK,2021-11-05,65.250000,65.500000,0.922350,64.000000,63.866475,63.866475,True
6,DOHOME.BK,2021-11-05,17.186701,17.186701,0.408388,16.685421,16.574119,16.574119,True
7,EA.BK,2021-11-05,66.250000,66.500000,1.375779,64.000000,64.000000,64.000000,True
8,GFPT.BK,2021-11-05,11.800000,11.800000,0.233068,11.300000,11.300000,11.300000,True
9,GPSC.BK,2021-11-05,78.500000,77.750000,1.314250,76.500000,76.500000,76.500000,True



✅ HISTORICAL SIGNALS FIXED
CELL 11.2 PASSED


In [ ]:

# ============================================================
# 12.1) SIGNAL CLUSTER ANALYSIS
# NO TRADE-ENGINE DEPENDENCY
# ============================================================

print("🔍 SIGNAL CLUSTER ANALYSIS")
print("=" * 90)

assert not historical_signals.empty, "historical_signals is empty."

analysis = historical_signals.copy()
analysis["SignalDate"] = pd.to_datetime(analysis["SignalDate"])
analysis["EntryDate"] = pd.to_datetime(analysis["EntryDate"])

analysis = analysis.sort_values(
    ["Symbol", "SignalDate"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Days since previous signal for the same symbol
# ------------------------------------------------------------

analysis["PrevSignalDate"] = (
    analysis.groupby("Symbol")["SignalDate"].shift(1)
)

analysis["DaysSincePrevSignal"] = (
    analysis["SignalDate"] - analysis["PrevSignalDate"]
).dt.days

analysis["ClusteredSignal"] = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] <= 3)
)

clustered = analysis[analysis["ClusteredSignal"]].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_signals = len(analysis)
clustered_count = len(clustered)
cluster_rate = clustered_count / total_signals * 100

print("\n====================================")
print("📊 SIGNAL CLUSTER SUMMARY")
print("====================================")
print(f"Total signals     : {total_signals:,}")
print(f"Clustered signals : {clustered_count:,}")
print(f"Cluster rate      : {cluster_rate:.2f}%")

# ------------------------------------------------------------
# Symbol-level summary
# ------------------------------------------------------------

symbol_cluster = (
    analysis.groupby("Symbol")
    .agg(
        Signals=("SignalDate", "count"),
        Clustered=("ClusteredSignal", "sum"),
    )
)

symbol_cluster["ClusterRate"] = (
    symbol_cluster["Clustered"]
    / symbol_cluster["Signals"]
    * 100
)

symbol_cluster = symbol_cluster.sort_values(
    ["Signals", "ClusterRate"],
    ascending=[False, False],
)

print("\n====================================")
print("📈 SIGNALS BY SYMBOL")
print("====================================")
display(symbol_cluster)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

same_symbol_order_ok = (
    analysis.groupby("Symbol")["SignalDate"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)

invalid_gap = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] < 0)
).sum()

assert same_symbol_order_ok, "Signal dates are not chronological by symbol."
assert invalid_gap == 0, "Negative signal-date gap found."
assert clustered_count <= total_signals

print("\n====================================")
print("🔎 RECENT CLUSTERED SIGNALS")
print("====================================")

if clustered.empty:
    print("No clustered signals found.")
else:
    display(
        clustered[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "Score",
                "WeeklyConfirm",
                "DaysSincePrevSignal",
            ]
        ].tail(30)
    )

print("\n====================================")
print("✅ CELL 12.1 PASSED")
print("====================================")

🔍 SIGNAL CLUSTER ANALYSIS

📊 SIGNAL CLUSTER SUMMARY
Total signals     : 30,431
Clustered signals : 16,786
Cluster rate      : 55.16%

📈 SIGNALS BY SYMBOL


,Signals,Clustered,ClusterRate
Symbol,,,
UVAN.BK,352,223,63.352273
TTW.BK,352,217,61.647727
DMT.BK,334,202,60.479042
HTC.BK,333,200,60.060060
SPCG.BK,321,197,61.370717
...,...,...,...
CPAXT.BK,80,44,55.000000
NEO.BK,74,37,50.000000
MRDIYT.BK,64,40,62.500000



🔎 RECENT CLUSTERED SIGNALS


,Symbol,SignalDate,EntryDate,Score,WeeklyConfirm,DaysSincePrevSignal
30377,XO.BK,2025-06-26,2025-06-27,4,False,1.0
30378,XO.BK,2025-06-27,2025-06-30,3,False,1.0
30381,XO.BK,2025-07-18,2025-07-21,3,True,1.0
30387,XO.BK,2025-11-28,2025-12-01,3,False,1.0
30389,XO.BK,2026-01-06,2026-01-07,3,False,1.0
30391,XO.BK,2026-01-21,2026-01-22,3,False,1.0
30393,XO.BK,2026-02-10,2026-02-11,3,False,1.0
30394,XO.BK,2026-02-11,2026-02-12,4,False,1.0
30397,XO.BK,2026-03-17,2026-03-18,3,False,1.0
30399,XO.BK,2026-03-31,2026-04-01,3,False,1.0



✅ CELL 12.1 PASSED


In [ ]:
# ============================================================
# CELL 12.2 — NON-OVERLAPPING BACKTEST
# ============================================================

print("=" * 90)
print("🔬 NON-OVERLAPPING HISTORICAL BACKTEST")
print("=" * 90)

# ------------------------------------------------------------
# 1) Conservative trade simulator
# ------------------------------------------------------------

def simulate_trade_fixed_stop(signal, price_data):
    """
    Simulate one long trade with:
      - next-day entry
      - conservative gap/stop handling
      - target 1 tracking
      - target 2 exit
      - hard MAX_HOLDING_DAYS time exit
      - MAE/MFE measured only while the trade is actually open

    MAX_HOLDING_DAYS is interpreted as calendar days from EntryDate.
    On the maximum-holding-date bar, STOP/TARGET rules have priority;
    otherwise the position is closed at that day's Close.
    """

    entry_date = pd.Timestamp(signal["EntryDate"])
    entry_price = float(signal["EntryPrice"])
    stop_price = float(signal["StopPrice"])
    shares = int(signal["Shares"])

    if entry_price <= 0 or stop_price <= 0 or shares <= 0:
        return None

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    future = price_data.loc[
        price_data.index >= entry_date
    ].copy()

    if future.empty:
        return None

    max_hold_date = entry_date + pd.Timedelta(
        days=MAX_HOLDING_DAYS
    )

    future = future.loc[
        future.index <= max_hold_date
    ].copy()

    if future.empty:
        return None

    target1_hit = False
    exit_date = pd.NaT
    exit_price = np.nan
    exit_reason = None

    for current_date, bar in future.iterrows():

        day_open = float(bar["Open"])
        day_high = float(bar["High"])
        day_low = float(bar["Low"])
        day_close = float(bar["Close"])

        # ----------------------------------------------------
        # Gap handling
        # ----------------------------------------------------
        # If market opens below stop, actual exit is OPEN.
        # This is conservative and avoids assuming execution
        # at a better price than the market provided.
        if day_open <= stop_price:
            exit_date = current_date
            exit_price = day_open
            exit_reason = "STOP_GAP"
            break

        # ----------------------------------------------------
        # Normal intraday stop
        # ----------------------------------------------------
        if day_low <= stop_price:
            exit_date = current_date
            exit_price = stop_price
            exit_reason = "STOP"
            break

        # ----------------------------------------------------
        # Target 2
        # ----------------------------------------------------
        if day_open >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = day_open
            exit_reason = "TARGET2_GAP"
            break

        if day_high >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = target2
            exit_reason = "TARGET2"
            break

        # ----------------------------------------------------
        # Target 1 — tracking only
        # ----------------------------------------------------
        if day_high >= target1:
            target1_hit = True

        # ----------------------------------------------------
        # Hard time exit
        # ----------------------------------------------------
        # STOP/TARGET checks above always have priority on this
        # date. If neither occurs, exit at the closing price.
        if current_date >= max_hold_date:
            exit_date = current_date
            exit_price = day_close
            exit_reason = "MAX_HOLD"
            break

    # --------------------------------------------------------
    # End of data
    # --------------------------------------------------------

    if pd.isna(exit_date):
        # This can only happen when the available dataset ends
        # before MAX_HOLDING_DAYS. Close at the final available bar.
        exit_date = future.index[-1]
        exit_price = float(future.iloc[-1]["Close"])
        exit_reason = "EOD"

    holding_days = (
        pd.Timestamp(exit_date) - entry_date
    ).days

    if holding_days > MAX_HOLDING_DAYS:
        raise AssertionError(
            f"MAX_HOLDING_DAYS violated: "
            f"{holding_days} > {MAX_HOLDING_DAYS}"
        )

    pnl = (exit_price - entry_price) * shares
    return_pct = (
        (exit_price - entry_price)
        / entry_price
        * 100
    )

    # IMPORTANT:
    # MAE/MFE must be calculated only over the period in which
    # the position was actually open. Using all future bars would
    # introduce look-ahead leakage into the diagnostics.
    held_future = price_data.loc[
        (price_data.index >= entry_date)
        & (price_data.index <= exit_date)
    ].copy()

    min_low = float(held_future["Low"].min())
    max_high = float(held_future["High"].max())

    mae_pct = (
        (min_low - entry_price)
        / entry_price
        * 100
    )

    mfe_pct = (
        (max_high - entry_price)
        / entry_price
        * 100
    )

    return {
        "Symbol": signal["Symbol"],
        "SignalDate": signal["SignalDate"],
        "EntryDate": entry_date,
        "EntryPrice": entry_price,
        "StopPrice": stop_price,
        "Shares": shares,
        "Target1": target1,
        "Target2": target2,
        "Target1Hit": target1_hit,
        "ExitDate": exit_date,
        "ExitPrice": exit_price,
        "ExitReason": exit_reason,
        "HoldingDays": holding_days,
        "PnL": pnl,
        "ReturnPct": return_pct,
        "MAE_Pct": mae_pct,
        "MFE_Pct": mfe_pct,
        "Score": signal["Score"],
        "WeeklyConfirm": signal["WeeklyConfirm"],
    }


# ------------------------------------------------------------
# 2) Backtest one symbol without overlapping trades
# ------------------------------------------------------------

def backtest_one_symbol_nonoverlap(symbol):

    signals = historical_signals[
        historical_signals["Symbol"] == symbol
    ].copy()

    signals = signals.sort_values(
        ["EntryDate", "SignalDate"]
    ).reset_index(drop=True)

    if symbol not in feature_data:
        return pd.DataFrame()

    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT
    skipped_overlap = 0

    for _, signal in signals.iterrows():

        entry_date = pd.Timestamp(signal["EntryDate"])

        # ----------------------------------------------------
        # Prevent overlapping trades
        # ----------------------------------------------------
        if not pd.isna(previous_exit_date):
            if entry_date <= previous_exit_date:
                skipped_overlap += 1
                continue

        trade = simulate_trade_fixed_stop(
            signal,
            price_data
        )

        if trade is None:
            continue

        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    result = pd.DataFrame(trades)

    return result


# ------------------------------------------------------------
# 3) Test AU.BK first
# ------------------------------------------------------------

TEST_SYMBOL = WATCHLIST[0] if WATCHLIST else None
if TEST_SYMBOL is None:
    raise RuntimeError("No symbols passed the canonical fundamental universe gate.")

au_trades = backtest_one_symbol_nonoverlap(TEST_SYMBOL)

print("\n====================================")
print(f"📊 {TEST_SYMBOL} BACKTEST")
print("====================================")

print(
    f"Signals available : "
    f"{len(historical_signals[historical_signals['Symbol'] == TEST_SYMBOL]):,}"
)

print(f"Trades executed   : {len(au_trades):,}")

if not au_trades.empty:

    print(
        f"Total P&L         : "
        f"{au_trades['PnL'].sum():,.2f} THB"
    )

    print(
        f"Average Return    : "
        f"{au_trades['ReturnPct'].mean():.2f}%"
    )

    print(
        f"Median Return     : "
        f"{au_trades['ReturnPct'].median():.2f}%"
    )

    print(
        f"Win Rate          : "
        f"{(au_trades['ReturnPct'] > 0).mean() * 100:.2f}%"
    )

    print("\n====================================")
    print("🔎 TRADE SAMPLE")
    print("====================================")

    display(
        au_trades[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "ExitDate",
                "ExitPrice",
                "ExitReason",
                "HoldingDays",
                "Shares",
                "PnL",
                "ReturnPct",
                "Target1Hit",
            ]
        ].head(15)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    invalid_exit = (
        au_trades["ExitDate"] < au_trades["EntryDate"]
    ).sum()

    invalid_return = (
        ~pd.to_numeric(
            au_trades["ReturnPct"],
            errors="coerce"
        ).notna()
    ).sum()

    invalid_holding = (
        au_trades["HoldingDays"] > MAX_HOLDING_DAYS
    ).sum()

    print("\n====================================")
    print("🛡️ BACKTEST VALIDATION")
    print("====================================")
    print(f"Exit before Entry : {invalid_exit}")
    print(f"Invalid Return    : {invalid_return}")
    print(f"Over Max Holding  : {invalid_holding}")

    assert invalid_exit == 0
    assert invalid_return == 0
    assert invalid_holding == 0

    print("\n====================================")
    print("✅ CELL 12.2 PASSED")
    print("====================================")
    print(
        "Non-overlapping backtest is valid for the test symbol."
    )

else:
    print(
        "\n⚠️ No executable trades found for",
        TEST_SYMBOL
    )



🔬 NON-OVERLAPPING HISTORICAL BACKTEST



📊 PTT.BK BACKTEST
Signals available : 259
Trades executed   : 63
Total P&L         : 11.22 THB
Average Return    : 0.08%
Median Return     : -0.67%
Win Rate          : 42.86%

🔎 TRADE SAMPLE


,Symbol,SignalDate,EntryDate,EntryPrice,StopPrice,ExitDate,ExitPrice,ExitReason,HoldingDays,Shares,PnL,ReturnPct,Target1Hit
0,PTT.BK,2021-11-08,2021-11-09,38.25,37.000000,2021-11-22,37.000000,STOP,13,1600,-2000.000000,-3.267974,False
1,PTT.BK,2021-11-24,2021-11-25,38.00,36.681335,2021-11-29,36.681335,STOP,4,1500,-1977.997013,-3.470170,False
2,PTT.BK,2021-12-30,2022-01-04,38.25,37.030710,2022-01-24,38.500000,MAX_HOLD,20,1600,400.000000,0.653595,False
3,PTT.BK,2022-01-27,2022-01-28,38.75,37.609107,2022-02-17,40.000000,MAX_HOLD,20,1700,2125.000000,3.225806,False
4,PTT.BK,2022-03-02,2022-03-03,40.00,38.250000,2022-03-07,38.250000,STOP,4,1100,-1925.000000,-4.375000,False
5,PTT.BK,2022-03-14,2022-03-15,38.75,37.250000,2022-04-04,38.750000,MAX_HOLD,20,1300,0.000000,0.000000,False
6,PTT.BK,2022-04-28,2022-04-29,37.25,36.311631,2022-05-12,36.311631,STOP,13,2100,-1970.575455,-2.519112,False
7,PTT.BK,2022-05-17,2022-05-18,37.00,36.031952,2022-06-07,38.000000,MAX_HOLD,20,2000,2000.000000,2.702703,False
8,PTT.BK,2022-06-28,2022-06-29,34.75,32.750000,2022-07-19,34.250000,MAX_HOLD,20,1000,-500.000000,-1.438849,False
9,PTT.BK,2022-07-19,2022-07-20,34.25,33.000000,2022-08-09,36.250000,MAX_HOLD,20,1600,3200.000000,5.839416,True



🛡️ BACKTEST VALIDATION
Exit before Entry : 0
Invalid Return    : 0
Over Max Holding  : 0

✅ CELL 12.2 PASSED
Non-overlapping backtest is valid for the test symbol.


In [ ]:

# ============================================================
# CELL 12.3 — TRADE ENGINE DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🔎 TRADE ENGINE DIAGNOSTICS — TEST SYMBOL")
print("=" * 90)

if au_trades.empty:
    print("⚠️ No trades available.")
else:

    df = au_trades.copy()

    # --------------------------------------------------------
    # 1) Exit distribution
    # --------------------------------------------------------

    exit_counts = (
        df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    exit_counts["Pct"] = (
        exit_counts["Trades"] / len(df) * 100
    )

    print("\n====================================")
    print("📊 EXIT DISTRIBUTION")
    print("====================================")

    display(exit_counts)

    # --------------------------------------------------------
    # 2) Return distribution
    # --------------------------------------------------------

    print("\n====================================")
    print("📈 RETURN DISTRIBUTION")
    print("====================================")

    print(f"Best trade        : {df['ReturnPct'].max():.2f}%")
    print(f"Worst trade       : {df['ReturnPct'].min():.2f}%")
    print(f"Average           : {df['ReturnPct'].mean():.2f}%")
    print(f"Median            : {df['ReturnPct'].median():.2f}%")
    print(f"Win rate          : {(df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Loss rate         : {(df['ReturnPct'] < 0).mean() * 100:.2f}%")
    print(f"Flat              : {(df['ReturnPct'] == 0).mean() * 100:.2f}%")

    # --------------------------------------------------------
    # 3) Risk characteristics
    # --------------------------------------------------------

    df["RiskPct"] = (
        (df["EntryPrice"] - df["StopPrice"])
        / df["EntryPrice"]
        * 100
    )

    print("\n====================================")
    print("🛡️ STOP / RISK DISTRIBUTION")
    print("====================================")

    print(f"Average Stop Risk : {df['RiskPct'].mean():.2f}%")
    print(f"Median Stop Risk  : {df['RiskPct'].median():.2f}%")
    print(f"Minimum Stop Risk : {df['RiskPct'].min():.2f}%")
    print(f"Maximum Stop Risk : {df['RiskPct'].max():.2f}%")

    # --------------------------------------------------------
    # 4) Target1 reached but eventually lost
    # --------------------------------------------------------

    target1_then_loss = df[
        (df["Target1Hit"] == True) &
        (df["ReturnPct"] < 0)
    ].copy()

    print("\n====================================")
    print("⚠️ TARGET1 HIT → EVENTUAL LOSS")
    print("====================================")

    print(
        f"Count: {len(target1_then_loss):,} "
        f"({len(target1_then_loss) / len(df) * 100:.2f}%)"
    )

    if not target1_then_loss.empty:
        display(
            target1_then_loss[
                [
                    "SignalDate",
                    "EntryDate",
                    "EntryPrice",
                    "StopPrice",
                    "ExitDate",
                    "ExitPrice",
                    "ExitReason",
                    "ReturnPct",
                    "PnL",
                ]
            ].sort_values("ReturnPct").head(10)
        )

    # --------------------------------------------------------
    # 5) Large stop-distance trades
    # --------------------------------------------------------

    print("\n====================================")
    print("⚠️ LARGEST STOP DISTANCES")
    print("====================================")

    display(
        df[
            [
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "RiskPct",
                "ExitReason",
                "ReturnPct",
                "PnL",
            ]
        ]
        .sort_values("RiskPct", ascending=False)
        .head(10)
    )

    # --------------------------------------------------------
    # 6) Trade duration
    # --------------------------------------------------------

    df["HoldingDays"] = (
        pd.to_datetime(df["ExitDate"])
        - pd.to_datetime(df["EntryDate"])
    ).dt.days

    print("\n====================================")
    print("⏱️ HOLDING PERIOD")
    print("====================================")

    print(f"Average days     : {df['HoldingDays'].mean():.1f}")
    print(f"Median days      : {df['HoldingDays'].median():.1f}")
    print(f"Maximum days     : {df['HoldingDays'].max():.0f}")
    print(f"MAX_HOLDING_DAYS : {MAX_HOLDING_DAYS}")

    over_limit = int(
        (df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    print(f"Over max holding : {over_limit}")

    # --------------------------------------------------------
    # 7) Basic assertions
    # --------------------------------------------------------

    assert (df["EntryDate"] <= df["ExitDate"]).all()

    assert (df["HoldingDays"] >= 0).all()
    assert (df["HoldingDays"] <= MAX_HOLDING_DAYS).all()
    assert over_limit == 0
    assert df["RiskPct"].notna().all()
    assert df["ReturnPct"].notna().all()

    print("\n====================================")
    print("✅ CELL 12.3 PASSED")
    print("====================================")
    print("Trade-engine diagnostics completed.")


🔎 TRADE ENGINE DIAGNOSTICS — TEST SYMBOL

📊 EXIT DISTRIBUTION


,ExitReason,Trades,Pct
0,MAX_HOLD,32,50.793651
1,STOP,22,34.920635
2,EOD,7,11.111111
3,STOP_GAP,2,3.174603



📈 RETURN DISTRIBUTION
Best trade        : 7.03%
Worst trade       : -5.76%
Average           : 0.08%
Median            : -0.67%
Win rate          : 42.86%
Loss rate         : 50.79%
Flat              : 6.35%

🛡️ STOP / RISK DISTRIBUTION
Average Stop Risk : 3.40%
Median Stop Risk  : 3.27%
Minimum Stop Risk : 1.39%
Maximum Stop Risk : 6.33%

⚠️ TARGET1 HIT → EVENTUAL LOSS
Count: 0 (0.00%)

⚠️ LARGEST STOP DISTANCES


,SignalDate,EntryDate,EntryPrice,StopPrice,RiskPct,ExitReason,ReturnPct,PnL
60,2026-08-11,2026-08-13,39.50,37.00,6.329114,MAX_HOLD,3.797468,1200.0
17,2023-06-08,2023-06-09,31.75,29.75,6.299213,MAX_HOLD,3.149606,1000.0
18,2023-06-29,2023-06-30,33.00,31.00,6.060606,MAX_HOLD,3.030303,1000.0
31,2024-08-09,2024-08-13,33.25,31.25,6.015038,MAX_HOLD,0.751880,250.0
32,2024-09-06,2024-09-09,34.75,32.75,5.755396,STOP,-5.755396,-2000.0
8,2022-06-28,2022-06-29,34.75,32.75,5.755396,MAX_HOLD,-1.438849,-500.0
59,2026-07-03,2026-07-06,37.00,35.00,5.405405,EOD,6.756757,2500.0
13,2022-12-27,2022-12-28,32.50,30.75,5.384615,MAX_HOLD,3.076923,1100.0
21,2023-10-11,2023-10-12,33.25,31.50,5.263158,MAX_HOLD,-0.751880,-275.0
43,2025-06-13,2025-06-16,31.00,29.50,4.838710,EOD,-2.419355,-975.0



⏱️ HOLDING PERIOD
Average days     : 14.9
Median days      : 20.0
Maximum days     : 20
MAX_HOLDING_DAYS : 20
Over max holding : 0

✅ CELL 12.3 PASSED
Trade-engine diagnostics completed.


In [ ]:
# ============================================================
# CELL 12.4 — ALL-SYMBOL NON-OVERLAPPING BACKTEST
# Research validation across the complete watchlist
# ============================================================

print("=" * 90)
print("📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST")
print("=" * 90)

all_trades = []
symbol_summary = []

for symbol in WATCHLIST:
    trades = backtest_one_symbol_nonoverlap(symbol)

    signal_count = int(
        (historical_signals["Symbol"] == symbol).sum()
    )

    trade_count = len(trades)

    if not trades.empty:
        all_trades.append(trades)

        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": trade_count,
            "PnL": float(trades["PnL"].sum()),
            "AvgReturnPct": float(trades["ReturnPct"].mean()),
            "WinRatePct": float((trades["ReturnPct"] > 0).mean() * 100),
            "MaxHoldingDays": int(trades["HoldingDays"].max()),
        })
    else:
        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": 0,
            "PnL": 0.0,
            "AvgReturnPct": np.nan,
            "WinRatePct": np.nan,
            "MaxHoldingDays": 0,
        })

if all_trades:
    all_trades_df = (
        pd.concat(all_trades, ignore_index=True)
        .sort_values(["EntryDate", "Symbol"])
        .reset_index(drop=True)
    )
else:
    all_trades_df = pd.DataFrame()

symbol_summary_df = pd.DataFrame(symbol_summary)

print("\n====================================")
print("📈 PORTFOLIO-LEVEL SUMMARY")
print("====================================")

print(f"Symbols tested     : {len(WATCHLIST)}")
print(f"Total trades       : {len(all_trades_df):,}")

if not all_trades_df.empty:
    print(f"Total P&L          : {all_trades_df['PnL'].sum():,.2f} THB")
    print(f"Average return     : {all_trades_df['ReturnPct'].mean():.2f}%")
    print(f"Median return      : {all_trades_df['ReturnPct'].median():.2f}%")
    print(f"Win rate           : {(all_trades_df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Max holding days   : {all_trades_df['HoldingDays'].max():.0f}")

    over_limit = int(
        (all_trades_df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    invalid_dates = int(
        (all_trades_df["ExitDate"] < all_trades_df["EntryDate"]).sum()
    )

    invalid_returns = int(
        (~pd.to_numeric(
            all_trades_df["ReturnPct"],
            errors="coerce"
        ).notna()).sum()
    )

    print(f"Over max holding   : {over_limit}")
    print(f"Invalid exit dates : {invalid_dates}")
    print(f"Invalid returns    : {invalid_returns}")

    print("\n====================================")
    print("🚪 EXIT REASONS")
    print("====================================")
    display(
        all_trades_df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    print("\n====================================")
    print("🏆 BY SYMBOL")
    print("====================================")
    display(
        symbol_summary_df
        .sort_values(["PnL", "WinRatePct"], ascending=[False, False])
        .reset_index(drop=True)
    )

    assert over_limit == 0
    assert invalid_dates == 0
    assert invalid_returns == 0
    assert all_trades_df["HoldingDays"].ge(0).all()
    assert all_trades_df["HoldingDays"].le(MAX_HOLDING_DAYS).all()

    print("\n====================================")
    print("✅ CELL 12.4 PASSED")
    print("====================================")
    print("All-symbol non-overlapping backtest passed validation.")
else:
    print("⚠️ No executable trades found across the watchlist.")


📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST



📈 PORTFOLIO-LEVEL SUMMARY
Symbols tested     : 133
Total trades       : 8,594
Total P&L          : -757,102.60 THB
Average return     : -0.13%
Median return      : -1.88%
Win rate           : 36.43%
Max holding days   : 20
Over max holding   : 0
Invalid exit dates : 0
Invalid returns    : 0

🚪 EXIT REASONS


,ExitReason,Trades
0,STOP,3541
1,MAX_HOLD,2936
2,EOD,1019
3,TARGET2,677
4,STOP_GAP,360
5,TARGET2_GAP,61



🏆 BY SYMBOL


,Symbol,Signals,Trades,PnL,AvgReturnPct,WinRatePct,MaxHoldingDays
0,SRICHA.BK,264,73,53849.536444,2.979650,43.835616,20
1,UVAN.BK,352,73,50021.879136,0.615894,39.726027,20
2,SAPPE.BK,273,68,45214.565105,1.855842,48.529412,20
3,MGC.BK,180,51,36864.912933,2.107067,50.980392,20
4,JTS.BK,234,75,33250.836042,0.041816,44.000000,20
...,...,...,...,...,...,...,...
128,CHG.BK,253,74,-42179.090855,-1.149497,31.081081,20
129,TPIPP.BK,272,67,-42327.632969,-0.780682,23.880597,20
130,DCC.BK,285,74,-42333.768853,-0.989691,32.432432,20
131,BTS.BK,240,71,-44528.756955,-1.706861,19.718310,20



✅ CELL 12.4 PASSED
All-symbol non-overlapping backtest passed validation.


In [ ]:
# ============================================================
# CELL 12.32 — PRODUCTION READINESS AUDIT
# ============================================================
print("=" * 90)
print("PRODUCTION READINESS AUDIT — NO STRATEGY CHANGE")
print("=" * 90)

assert "canonical_data" in globals()
assert "PRODUCTION_SYMBOLS" in globals()
assert "PRODUCTION_MIN_SCORE" in globals()
assert "PRODUCTION_WEEKLY_CONFIRM" in globals()
assert "PRODUCTION_ATR_MULT" in globals()
assert "PRODUCTION_TARGET2_PCT" in globals()
assert "PRODUCTION_MAX_HOLDING_DAYS" in globals()

audit_rows = []
for sym in PRODUCTION_SYMBOLS:
    df = canonical_data.get(sym)
    if df is None or df.empty:
        audit_rows.append({
            "Symbol": sym, "Bars": 0, "FirstDate": None, "LastDate": None,
            "MissingOHLCV": True, "EMA75Ready": False, "ATR14Ready": False,
            "Status": "MISSING_DATA",
        })
        continue

    required = ["Open", "High", "Low", "Close", "Volume"]
    missing = bool(df[required].isna().any().any()) if all(c in df.columns for c in required) else True
    bars = len(df)
    ema75_ready = "EMA75" in df.columns and df["EMA75"].notna().sum() >= 1
    atr14_ready = "ATR14" in df.columns and df["ATR14"].notna().sum() >= 1

    audit_rows.append({
        "Symbol": sym,
        "Bars": bars,
        "FirstDate": pd.Timestamp(df.index.min()).date(),
        "LastDate": pd.Timestamp(df.index.max()).date(),
        "MissingOHLCV": missing,
        "EMA75Ready": ema75_ready,
        "ATR14Ready": atr14_ready,
        "Status": "OK" if bars >= 200 and not missing and ema75_ready and atr14_ready else "CHECK",
    })

production_data_audit = pd.DataFrame(audit_rows)
display(production_data_audit)

print("\nPRODUCTION CONFIG")
print("Symbols:", ", ".join(PRODUCTION_SYMBOLS))
print(f"Min score: {PRODUCTION_MIN_SCORE}")
print(f"Weekly confirmation: {PRODUCTION_WEEKLY_CONFIRM}")
print(f"ATR multiplier: {PRODUCTION_ATR_MULT}")
print(f"Target2: {PRODUCTION_TARGET2_PCT:.2%}")
print(f"Max holding days: {PRODUCTION_MAX_HOLDING_DAYS}")

bad = production_data_audit[production_data_audit["Status"] != "OK"]

print("\nDATA READINESS")
print(f"Production symbols checked: {len(production_data_audit)}")
print(f"Ready: {len(production_data_audit) - len(bad)}")
print(f"Needs attention: {len(bad)}")

if bad.empty:
    print("PRODUCTION READINESS: PASS")
else:
    print("PRODUCTION READINESS: CHECK_REQUIRED")

print("PRODUCTION CONFIGURATION UNCHANGED")
print("PRODUCTION READINESS AUDIT COMPLETE")



PRODUCTION READINESS AUDIT — NO STRATEGY CHANGE


,Symbol,Bars,FirstDate,LastDate,MissingOHLCV,EMA75Ready,ATR14Ready,Status
0,PTT.BK,1197,2021-11-01,2026-10-08,False,True,True,OK
1,KCE.BK,1198,2021-11-01,2026-10-08,False,True,True,OK
2,HANA.BK,1198,2021-11-01,2026-10-08,False,True,True,OK
3,GULF.BK,352,2025-04-25,2026-10-08,False,True,True,OK
4,CPALL.BK,1198,2021-11-01,2026-10-08,False,True,True,OK
...,...,...,...,...,...,...,...,...
128,SMPC.BK,1197,2021-11-01,2026-10-08,False,True,True,OK
129,PSGC.BK,1198,2021-11-01,2026-10-08,False,True,True,OK
130,SCGD.BK,666,2024-01-11,2026-10-08,False,True,True,OK
131,TFM.BK,1186,2021-11-17,2026-10-08,False,True,True,OK



PRODUCTION CONFIG
Symbols: PTT.BK, KCE.BK, HANA.BK, GULF.BK, CPALL.BK, TOP.BK, TRUE.BK, AOT.BK, PTTGC.BK, BCP.BK, BDMS.BK, IVL.BK, THAI.BK, CPF.BK, IRPC.BK, STECON.BK, GPSC.BK, SPRC.BK, BGRIM.BK, GUNKUL.BK, COM7.BK, BEM.BK, TU.BK, CRC.BK, STA.BK, CBG.BK, SCGP.BK, RCL.BK, HMPRO.BK, SMT.BK, PLANB.BK, BJC.BK, RATCH.BK, CK.BK, OR.BK, OSP.BK, ITC.BK, BA.BK, BCH.BK, PTG.BK, EA.BK, SPCG.BK, GLOBAL.BK, TFG.BK, TASCO.BK, PSL.BK, BANPU.BK, EPG.BK, CHG.BK, CPAXT.BK, WHAUP.BK, DOHOME.BK, SSP.BK, BTS.BK, STPI.BK, JTS.BK, BTG.BK, SINGER.BK, FORTH.BK, PRM.BK, MRDIYT.BK, ICHI.BK, GST.BK, BCPG.BK, GFPT.BK, JMART.BK, TOA.BK, PR9.BK, TVO.BK, MGC.BK, TTW.BK, MEGA.BK, TTA.BK, NER.BK, SJWD.BK, STGT.BK, MOSHI.BK, SKY.BK, DITTO.BK, PSP.BK, TKN.BK, BBGI.BK, NEO.BK, MC.BK, SIS.BK, AAI.BK, MAJOR.BK, M.BK, SYNEX.BK, XO.BK, SISB.BK, KSL.BK, SAT.BK, KLINIQ.BK, PCE.BK, AURA.BK, UVAN.BK, THG.BK, SABINA.BK, TPIPP.BK, CKP.BK, RBF.BK, VIH.BK, NSL.BK, SAPPE.BK, KAMART.BK, HTC.BK, SAMART.BK, DCC.BK, KCG.BK, SGP.BK, UTP.B

In [ ]:
# ============================================================
# CELL 12.33 — FROZEN PRODUCTION BASELINE BACKTEST
# Fundamental universe + frozen production parameters
# NO strategy optimization / NO friction yet
# ============================================================

print("=" * 90)
print("📌 FROZEN PRODUCTION BASELINE BACKTEST")
print("=" * 90)

assert RESEARCH_USE_FUNDAMENTAL_UNIVERSE_GATE is True
assert len(PRODUCTION_SYMBOLS) == 133
assert PRODUCTION_MIN_SCORE == 4
assert PRODUCTION_WEEKLY_CONFIRM is False
assert PRODUCTION_ATR_MULT == 1.2
assert PRODUCTION_TARGET2_PCT == 0.15
assert PRODUCTION_MAX_HOLDING_DAYS == 20

# ------------------------------------------------------------
# Rebuild executable production signals from the already-fixed
# canonical signal dates, using the FROZEN production score/stop.
# This deliberately does not modify historical research signals.
# ------------------------------------------------------------
production_signal_rows = []

for _, old in historical_signals.iterrows():
    if old["Symbol"] not in PRODUCTION_SYMBOLS:
        continue
    if int(old["Score"]) < PRODUCTION_MIN_SCORE:
        continue
    if PRODUCTION_WEEKLY_CONFIRM and not bool(old["WeeklyConfirm"]):
        continue

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    df = feature_data.get(symbol)
    if df is None or signal_date not in df.index or entry_date not in df.index:
        continue

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    atr = float(signal_row["ATR14"])
    recent_low = float(signal_row.get("RECENT_LOW_5", np.nan))
    if not np.isfinite(recent_low):
        recent_low = float(df.loc[:signal_date, "Low"].tail(RECENT_LOW_LOOKBACK).min())
    entry_price = float(entry_row["Open"])

    stop_price = calculate_stop_price(
        signal_close,
        atr,
        recent_low,
        atr_multiplier=PRODUCTION_ATR_MULT,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        continue
    if entry_price <= stop_price or entry_price <= 0:
        continue

    position = calculate_position_size(
        INITIAL_CAPITAL,
        entry_price,
        stop_price,
        risk_pct=MAX_RISK_PCT,
        lot_size=LOT_SIZE,
    )

    if position["shares"] <= 0:
        continue

    production_signal_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": atr,
        "RecentLow5": recent_low,
        "StopPrice": float(stop_price),
        "Shares": int(position["shares"]),
        "Lots": int(position["lots"]),
        "RiskAmount": float(position["risk_amount"]),
        "PositionValue": float(position["position_value"]),
        "Target1": entry_price * 1.07,
        "Target2": entry_price * (1 + PRODUCTION_TARGET2_PCT),
    })

production_signals = pd.DataFrame(production_signal_rows)
if not production_signals.empty:
    production_signals = production_signals.sort_values(
        ["SignalDate", "Symbol"]
    ).reset_index(drop=True)

print(f"Production signals (Score >= {PRODUCTION_MIN_SCORE}): {len(production_signals):,}")
print(f"Production symbols: {production_signals['Symbol'].nunique() if not production_signals.empty else 0} / {len(PRODUCTION_SYMBOLS)}")

# ------------------------------------------------------------
# Frozen production backtest. The existing simulator is used
# unchanged: next-day open, conservative gap/stop handling,
# target2, and hard 20-day exit.
# ------------------------------------------------------------

def backtest_production_symbol(symbol):
    signals = production_signals[production_signals["Symbol"] == symbol].copy()
    signals = signals.sort_values(["EntryDate", "SignalDate"]).reset_index(drop=True)
    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT

    for _, signal in signals.iterrows():
        entry_date = pd.Timestamp(signal["EntryDate"])
        if not pd.isna(previous_exit_date) and entry_date <= previous_exit_date:
            continue

        trade = simulate_trade_fixed_stop(signal, price_data)
        if trade is None:
            continue
        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    return pd.DataFrame(trades)

production_trades = []
production_symbol_summary = []

for symbol in PRODUCTION_SYMBOLS:
    trades = backtest_production_symbol(symbol)
    if not trades.empty:
        production_trades.append(trades)
        production_symbol_summary.append({
            "Symbol": symbol,
            "Signals": int((production_signals["Symbol"] == symbol).sum()),
            "Trades": len(trades),
            "PnL": float(trades["PnL"].sum()),
            "AvgReturnPct": float(trades["ReturnPct"].mean()),
            "WinRatePct": float((trades["ReturnPct"] > 0).mean() * 100),
        })
    else:
        production_symbol_summary.append({
            "Symbol": symbol,
            "Signals": int((production_signals["Symbol"] == symbol).sum()),
            "Trades": 0,
            "PnL": 0.0,
            "AvgReturnPct": np.nan,
            "WinRatePct": np.nan,
        })

if production_trades:
    production_trades_df = pd.concat(production_trades, ignore_index=True).sort_values(
        ["EntryDate", "Symbol"]
    ).reset_index(drop=True)
else:
    production_trades_df = pd.DataFrame()

production_symbol_summary_df = pd.DataFrame(production_symbol_summary)

print("\n====================================")
print("📈 FROZEN PRODUCTION BASELINE SUMMARY")
print("====================================")
print(f"Universe            : {len(PRODUCTION_SYMBOLS)} symbols")
print(f"Signals              : {len(production_signals):,}")
print(f"Trades               : {len(production_trades_df):,}")

if not production_trades_df.empty:
    print(f"Total P&L            : {production_trades_df['PnL'].sum():,.2f} THB")
    print(f"Average return       : {production_trades_df['ReturnPct'].mean():.2f}%")
    print(f"Median return        : {production_trades_df['ReturnPct'].median():.2f}%")
    print(f"Win rate             : {(production_trades_df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Max holding days     : {production_trades_df['HoldingDays'].max():.0f}")

    over_limit = int((production_trades_df["HoldingDays"] > PRODUCTION_MAX_HOLDING_DAYS).sum())
    invalid_dates = int((production_trades_df["ExitDate"] < production_trades_df["EntryDate"]).sum())
    invalid_returns = int((~pd.to_numeric(production_trades_df["ReturnPct"], errors="coerce").notna()).sum())
    risk_over_limit = int((production_signals["RiskAmount"] > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9).sum())

    print(f"Over max holding     : {over_limit}")
    print(f"Invalid exit dates   : {invalid_dates}")
    print(f"Invalid returns      : {invalid_returns}")
    print(f"Risk > 2%            : {risk_over_limit}")

    print("\n====================================")
    print("🚪 PRODUCTION EXIT REASONS")
    print("====================================")
    display(production_trades_df["ExitReason"].value_counts().rename_axis("ExitReason").reset_index(name="Trades"))

    print("\n====================================")
    print("🏆 PRODUCTION BY SYMBOL")
    print("====================================")
    display(production_symbol_summary_df.sort_values("PnL", ascending=False).reset_index(drop=True).head(30))

    assert over_limit == 0
    assert invalid_dates == 0
    assert invalid_returns == 0
    assert risk_over_limit == 0
    assert production_trades_df["HoldingDays"].ge(0).all()
    assert production_trades_df["HoldingDays"].le(PRODUCTION_MAX_HOLDING_DAYS).all()

print("\n====================================")
print("✅ CELL 12.33 PASSED")
print("Frozen production baseline completed without changing strategy parameters.")
print("====================================")


📌 FROZEN PRODUCTION BASELINE BACKTEST


Production signals (Score >= 4): 8,185
Production symbols: 133 / 133



📈 FROZEN PRODUCTION BASELINE SUMMARY
Universe            : 133 symbols
Signals              : 8,185
Trades               : 3,767
Total P&L            : -274,732.03 THB
Average return       : -0.08%
Median return        : -1.80%
Win rate             : 35.63%
Max holding days     : 20
Over max holding     : 0
Invalid exit dates   : 0
Invalid returns      : 0
Risk > 2%            : 0

🚪 PRODUCTION EXIT REASONS


,ExitReason,Trades
0,STOP,1583
1,MAX_HOLD,1262
2,EOD,427
3,TARGET2,280
4,STOP_GAP,190
5,TARGET2_GAP,25



🏆 PRODUCTION BY SYMBOL


,Symbol,Signals,Trades,PnL,AvgReturnPct,WinRatePct
0,SRICHA.BK,72,30,50355.760815,4.077694,53.333333
1,TKN.BK,69,33,32925.079186,0.978639,45.454545
2,SKY.BK,84,38,32314.709658,1.391469,39.473684
3,TOP.BK,75,31,32180.302122,3.020036,48.387097
4,MGC.BK,36,21,31750.916446,4.567201,52.380952
5,UVAN.BK,102,41,30226.660165,1.536980,48.780488
6,GUNKUL.BK,56,27,29004.707759,3.208640,51.851852
7,SAPPE.BK,94,37,27023.406835,2.408822,51.351351
8,PSP.BK,51,22,24893.949267,2.198461,31.818182
9,JTS.BK,44,24,24654.586899,2.051623,50.000000



✅ CELL 12.33 PASSED
Frozen production baseline completed without changing strategy parameters.


In [ ]:
# ============================================================
# CELL 12.34 — TRUE GLOBAL PORTFOLIO FRICTION / GAP STRESS
# Frozen production parameters; no optimization
# ============================================================

print("=" * 90)
print("🌐 TRUE GLOBAL PORTFOLIO — FRICTION / SLIPPAGE / GAP STRESS")
print("=" * 90)

assert len(PRODUCTION_SYMBOLS) == 133
assert PRODUCTION_MIN_SCORE == 4
assert PRODUCTION_WEEKLY_CONFIRM is False
assert PRODUCTION_ATR_MULT == 1.2
assert PRODUCTION_TARGET2_PCT == 0.15
assert PRODUCTION_MAX_HOLDING_DAYS == 20
GLOBAL_INITIAL_CAPITAL = 100_000.0
GLOBAL_RISK_PCT = 0.02
GLOBAL_MAX_POSITIONS = 5

# Build one deterministic executable trade candidate per production signal.
# Exit timing comes only from the unchanged trade simulator.
candidate_rows = []

for _, sig in production_signals.sort_values(["EntryDate", "Score", "Symbol"]).iterrows():
    skeleton = sig.copy()
    skeleton["Shares"] = 1
    skeleton["PositionValue"] = float(sig["EntryPrice"])
    skeleton["RiskAmount"] = float(sig["EntryPrice"] - sig["StopPrice"])
    trade = simulate_trade_fixed_stop(skeleton, feature_data[sig["Symbol"]])
    if trade is None:
        continue
    row = sig.to_dict()
    row.update({
        "ExitDate": trade["ExitDate"],
        "ExitPrice": float(trade["ExitPrice"]),
        "ExitReason": trade["ExitReason"],
        "HoldingDays": int(trade["HoldingDays"]),
        "Target1Hit": bool(trade["Target1Hit"]),
    })
    candidate_rows.append(row)

global_candidates = pd.DataFrame(candidate_rows)
assert not global_candidates.empty
global_candidates = global_candidates.sort_values(
    ["EntryDate", "Score", "Symbol"]
).reset_index(drop=True)

print(f"Executable production candidates : {len(global_candidates):,}")
print(f"Candidate symbols                 : {global_candidates['Symbol'].nunique()} / {len(PRODUCTION_SYMBOLS)}")

def run_true_global_portfolio(cost_rate=0.0, slippage_rate=0.0, gap_loss_rate=0.0):
    cash = GLOBAL_INITIAL_CAPITAL
    open_positions = []
    closed = []
    skipped_capacity = 0
    skipped_symbol_overlap = 0
    skipped_capital = 0

    # Events are entry/exit dates. Exits on an entry date are processed first.
    event_dates = sorted(set(global_candidates["EntryDate"]) | set(global_candidates["ExitDate"]))

    candidates_by_date = {
        d: g.sort_values(["Score", "Symbol"], ascending=[False, True])
        for d, g in global_candidates.groupby("EntryDate")
    }

    for current_date in event_dates:
        current_date = pd.Timestamp(current_date)

        # Close positions whose scheduled exit is today or earlier.
        still_open = []
        for pos in open_positions:
            if pd.Timestamp(pos["ExitDate"]) <= current_date:
                raw_exit = float(pos["ExitPrice"])
                shocked_exit = raw_exit
                if pos["ExitReason"] == "STOP_GAP":
                    shocked_exit *= (1 - gap_loss_rate)

                net_exit_price = shocked_exit * (1 - slippage_rate) * (1 - cost_rate)
                proceeds = pos["Shares"] * net_exit_price
                pnl = proceeds - pos["EntryCash"]

                cash += proceeds
                closed.append({
                    **pos,
                    "GrossExitPrice": raw_exit,
                    "StressExitPrice": shocked_exit,
                    "NetExitPrice": net_exit_price,
                    "Proceeds": proceeds,
                    "PnL": pnl,
                })
            else:
                still_open.append(pos)
        open_positions = still_open

        # New entries: exits first, then best scores first.
        for _, sig in candidates_by_date.get(current_date, pd.DataFrame()).iterrows():
            symbol = sig["Symbol"]

            if any(p["Symbol"] == symbol for p in open_positions):
                skipped_symbol_overlap += 1
                continue

            if len(open_positions) >= GLOBAL_MAX_POSITIONS:
                skipped_capacity += 1
                continue

            entry_price = float(sig["EntryPrice"])
            stop_price = float(sig["StopPrice"])
            if entry_price <= stop_price:
                continue

            position = calculate_position_size(
                cash,
                entry_price,
                stop_price,
                risk_pct=GLOBAL_RISK_PCT,
                lot_size=LOT_SIZE,
            )

            shares = int(position["shares"])
            if shares <= 0:
                skipped_capital += 1
                continue

            net_entry_price = entry_price * (1 + slippage_rate) * (1 + cost_rate)
            entry_cash = shares * net_entry_price

            if entry_cash > cash + 1e-9:
                skipped_capital += 1
                continue

            cash -= entry_cash

            open_positions.append({
                "Symbol": symbol,
                "SignalDate": pd.Timestamp(sig["SignalDate"]),
                "EntryDate": current_date,
                "EntryPrice": entry_price,
                "StopPrice": stop_price,
                "Shares": shares,
                "Score": int(sig["Score"]),
                "ExitDate": pd.Timestamp(sig["ExitDate"]),
                "ExitPrice": float(sig["ExitPrice"]),
                "ExitReason": sig["ExitReason"],
                "EntryCash": entry_cash,
                "HoldingDays": int((pd.Timestamp(sig["ExitDate"]) - current_date).days),
            })

    # Everything should have exited by the final event date.
    for pos in open_positions:
        raw_exit = float(pos["ExitPrice"])
        shocked_exit = raw_exit * (1 - gap_loss_rate) if pos["ExitReason"] == "STOP_GAP" else raw_exit
        net_exit_price = shocked_exit * (1 - slippage_rate) * (1 - cost_rate)
        proceeds = pos["Shares"] * net_exit_price
        cash += proceeds
        closed.append({
            **pos,
            "GrossExitPrice": raw_exit,
            "StressExitPrice": shocked_exit,
            "NetExitPrice": net_exit_price,
            "Proceeds": proceeds,
            "PnL": proceeds - pos["EntryCash"],
        })

    trades = pd.DataFrame(closed)
    if trades.empty:
        return {
            "FinalEquity": cash,
            "NetPnL": cash - GLOBAL_INITIAL_CAPITAL,
            "Trades": 0,
            "WinRatePct": np.nan,
            "ProfitFactor": np.nan,
            "MaxDrawdownPct": 0.0,
            "SkippedCapacity": skipped_capacity,
            "SkippedSymbolOverlap": skipped_symbol_overlap,
            "SkippedCapital": skipped_capital,
            "TradesDF": trades,
        }

    trades = trades.sort_values(["ExitDate", "EntryDate", "Symbol"]).reset_index(drop=True)
    trades["Equity"] = GLOBAL_INITIAL_CAPITAL + trades["PnL"].cumsum()
    running_peak = trades["Equity"].cummax()
    trades["DrawdownPct"] = (trades["Equity"] / running_peak - 1) * 100

    wins = trades["PnL"] > 0
    gross_profit = float(trades.loc[wins, "PnL"].sum())
    gross_loss = float(-trades.loc[~wins, "PnL"].sum())
    pf = gross_profit / gross_loss if gross_loss > 0 else np.inf

    return {
        "FinalEquity": float(cash),
        "NetPnL": float(cash - GLOBAL_INITIAL_CAPITAL),
        "Trades": len(trades),
        "WinRatePct": float(wins.mean() * 100),
        "ProfitFactor": float(pf),
        "MaxDrawdownPct": float(trades["DrawdownPct"].min()),
        "SkippedCapacity": int(skipped_capacity),
        "SkippedSymbolOverlap": int(skipped_symbol_overlap),
        "SkippedCapital": int(skipped_capital),
        "TradesDF": trades,
    }

scenarios = [
    ("GROSS_BASELINE", 0.00, 0.00, 0.00),
    ("REALISTIC", 0.0015, 0.0010, 0.0025),
    ("MEDIUM_STRESS", 0.0020, 0.0025, 0.0050),
    ("WORST_STRESS", 0.0025, 0.0050, 0.0100),
    ("GAP_ONLY", 0.0000, 0.0000, 0.0050),
    ("SLIPPAGE_ONLY", 0.0000, 0.0025, 0.0000),
]

stress_rows = []
stress_runs = {}

for name, cost, slip, gap in scenarios:
    result = run_true_global_portfolio(cost, slip, gap)
    stress_runs[name] = result
    stress_rows.append({
        "Scenario": name,
        "CostPerSidePct": cost * 100,
        "SlippagePerSidePct": slip * 100,
        "GapShockPct": gap * 100,
        "Trades": result["Trades"],
        "FinalEquity": result["FinalEquity"],
        "NetPnL": result["NetPnL"],
        "ReturnPct": result["NetPnL"] / GLOBAL_INITIAL_CAPITAL * 100,
        "WinRatePct": result["WinRatePct"],
        "ProfitFactor": result["ProfitFactor"],
        "MaxDrawdownPct": result["MaxDrawdownPct"],
        "SkippedCapacity": result["SkippedCapacity"],
        "SkippedSymbolOverlap": result["SkippedSymbolOverlap"],
        "SkippedCapital": result["SkippedCapital"],
    })

stress_results = pd.DataFrame(stress_rows)

print("\n====================================")
print("📊 TRUE GLOBAL PORTFOLIO STRESS RESULTS")
print("====================================")
display(stress_results)

# Integrity checks: same signal universe and fixed portfolio constraints.
assert stress_results["Trades"].ge(0).all()
assert stress_results["FinalEquity"].ge(0).all()
assert (stress_results["Trades"] <= len(global_candidates)).all()

for name, result in stress_runs.items():
    t = result["TradesDF"]
    if not t.empty:
        assert (t["HoldingDays"] <= PRODUCTION_MAX_HOLDING_DAYS).all()
        assert (t["Shares"] > 0).all()
        assert (t["PnL"].notna()).all()

# Report realistic survival verdict without changing production.
realistic = stress_results.loc[stress_results["Scenario"] == "REALISTIC"].iloc[0]
gross = stress_results.loc[stress_results["Scenario"] == "GROSS_BASELINE"].iloc[0]

print("\n====================================")
print("🔎 REALISTIC-FRICTION VERDICT")
print("====================================")
print(f"Gross P&L       : {gross['NetPnL']:,.2f} THB")
print(f"Realistic P&L   : {realistic['NetPnL']:,.2f} THB")
print(f"Realistic PF    : {realistic['ProfitFactor']:.3f}")
print(f"Realistic MaxDD : {realistic['MaxDrawdownPct']:.2f}%")

if realistic["NetPnL"] > 0 and realistic["ProfitFactor"] > 1:
    print("REALISTIC_FRICTION_SURVIVES")
else:
    print("REALISTIC_FRICTION_DOES_NOT_SURVIVE")

print("\n====================================")
print("✅ CELL 12.34 PASSED")
print("True Global Portfolio stress validation completed; production parameters unchanged.")
print("====================================")


🌐 TRUE GLOBAL PORTFOLIO — FRICTION / SLIPPAGE / GAP STRESS


Executable production candidates : 8,185
Candidate symbols                 : 133 / 133



📊 TRUE GLOBAL PORTFOLIO STRESS RESULTS


,Scenario,CostPerSidePct,SlippagePerSidePct,GapShockPct,Trades,FinalEquity,NetPnL,ReturnPct,WinRatePct,ProfitFactor,MaxDrawdownPct,SkippedCapacity,SkippedSymbolOverlap,SkippedCapital
0,GROSS_BASELINE,0.00,0.00,0.00,730,39225.321106,-60774.678894,-60.774679,31.780822,0.698074,-67.179290,6325,820,310
1,REALISTIC,0.15,0.10,0.25,739,14265.603400,-85734.396600,-85.734397,29.093369,0.487559,-86.027065,5801,840,805
2,MEDIUM_STRESS,0.20,0.25,0.50,724,9416.566531,-90583.433469,-90.583433,27.624309,0.434758,-90.755399,5619,831,1011
3,WORST_STRESS,0.25,0.50,1.00,729,4367.945007,-95632.054993,-95.632055,25.788752,0.361092,-95.696971,5317,830,1309
4,GAP_ONLY,0.00,0.00,0.50,723,32413.379640,-67586.620360,-67.586620,31.120332,0.641828,-71.342858,6224,820,418
5,SLIPPAGE_ONLY,0.00,0.25,0.00,729,14788.672997,-85211.327003,-85.211327,29.766804,0.498649,-85.514726,6072,832,552



🔎 REALISTIC-FRICTION VERDICT
Gross P&L       : -60,774.68 THB
Realistic P&L   : -85,734.40 THB
Realistic PF    : 0.488
Realistic MaxDD : -86.03%
REALISTIC_FRICTION_DOES_NOT_SURVIVE

✅ CELL 12.34 PASSED
True Global Portfolio stress validation completed; production parameters unchanged.


In [ ]:
# ============================================================
# CELL 13.0 — CAUSAL EARLY-FAILURE RISK MODEL
# Fresh research branch; Production remains frozen
# ============================================================

print("=" * 90)
print("🧪 CAUSAL EARLY-FAILURE RISK MODEL — FRESH RESEARCH")
print("=" * 90)

# Research contract:
# - signal-date-only features
# - early failure = losing trade exiting within 5 days
# - train-only learning
# - validation-selected probability threshold
# - unseen chronological OOS
# - realistic friction: 0.15% cost + 0.10% slippage + 0.25% gap
# - promotion gate: >=2/3 folds, PF > 1 every fold, and better than BASE
# - Production parameters are never modified here.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RESEARCH_COST = 0.0015
RESEARCH_SLIPPAGE = 0.0010
RESEARCH_GAP = 0.0025
EARLY_DAYS = 5
THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

# ------------------------------------------------------------
# Build a causal feature table. Every feature is available no
# later than SignalDate; labels use future trade outcomes only.
# IMPORTANT: use canonical_data for canonical boolean score components.
# ------------------------------------------------------------
feature_rows = []

for _, sig in global_candidates.iterrows():
    symbol = sig["Symbol"]
    signal_date = pd.Timestamp(sig["SignalDate"])
    # Canonical flags are created in Cell 8 on canonical_data; feature_data
    # intentionally contains only the raw/derived indicator columns.
    df = canonical_data[symbol]
    if signal_date not in df.index:
        continue

    row = df.loc[signal_date]
    signal_close = float(sig["SignalClose"])
    entry_price = float(sig["EntryPrice"])
    atr = float(sig["ATR14"])
    stop = float(sig["StopPrice"])

    feature_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": pd.Timestamp(sig["EntryDate"]),
        "ExitDate": pd.Timestamp(sig["ExitDate"]),
        "ExitReason": sig["ExitReason"],
        "ExitPrice": float(sig["ExitPrice"]),
        "EntryPrice": entry_price,
        "StopPrice": stop,
        "Score": int(sig["Score"]),
        "WeeklyConfirm": int(bool(sig["WeeklyConfirm"])),
        "VolumeRatio": float(row["VOLUME_RATIO"]),
        "ATRPct": atr / signal_close if signal_close > 0 else np.nan,
        "StopRiskPct": (entry_price - stop) / entry_price if entry_price > 0 else np.nan,
        "EntryGapPct": entry_price / signal_close - 1 if signal_close > 0 else np.nan,
        "DistEMA75Pct": signal_close / float(row["EMA75"]) - 1 if float(row["EMA75"]) > 0 else np.nan,
        "MACDBull": int(bool((row["MACD"] > row["MACD_SIGNAL"]) and (row["MACD_HIST"] > 0))),
        "StochBull": int(bool((row["STOCH_K"] > row["STOCH_D"]) and (row["STOCH_K"] < 80))),
        "TrendBull": int(bool((row["Close"] > row["EMA5"]) and (row["EMA5"] > row["EMA10"]) and (row["EMA10"] > row["EMA75"]))),
        "EarlyEntry": int(bool(row["SQUEEZE"] or row["SQUEEZE_RELEASE"])),
        "VolumeConfirm": int(bool(row["VOLUME_RATIO"] >= 1.2)),
    })

model_df = pd.DataFrame(feature_rows).dropna().reset_index(drop=True)
model_df["HoldingDays"] = (model_df["ExitDate"] - model_df["EntryDate"]).dt.days
model_df["ReturnPct"] = (model_df["ExitPrice"] - model_df["EntryPrice"]) / model_df["EntryPrice"] * 100
model_df["EarlyFailure"] = (
    (model_df["HoldingDays"] <= EARLY_DAYS)
    & (model_df["ReturnPct"] < 0)
).astype(int)

FEATURES = [
    "Score", "WeeklyConfirm", "VolumeRatio", "ATRPct", "StopRiskPct",
    "EntryGapPct", "DistEMA75Pct", "MACDBull", "StochBull",
    "TrendBull", "EarlyEntry", "VolumeConfirm",
]

assert model_df[FEATURES].notna().all().all()
assert set(model_df["EarlyFailure"].unique()).issubset({0, 1})

print(f"Model rows          : {len(model_df):,}")
print(f"Early failures      : {int(model_df['EarlyFailure'].sum()):,}")
print(f"Early-failure rate  : {model_df['EarlyFailure'].mean() * 100:.2f}%")

# ------------------------------------------------------------
# Parameterized global portfolio runner for research subsets.
# ------------------------------------------------------------
def run_global_research(candidates):
    candidates = candidates.sort_values(["EntryDate", "Score", "Symbol"]).reset_index(drop=True)
    cash = GLOBAL_INITIAL_CAPITAL
    open_positions = []
    closed = []

    if candidates.empty:
        return pd.DataFrame(), cash

    by_date = {
        d: g.sort_values(["Score", "Symbol"], ascending=[False, True])
        for d, g in candidates.groupby("EntryDate")
    }
    event_dates = sorted(set(candidates["EntryDate"]) | set(candidates["ExitDate"]))

    for current_date in event_dates:
        current_date = pd.Timestamp(current_date)

        still_open = []
        for pos in open_positions:
            if pd.Timestamp(pos["ExitDate"]) <= current_date:
                raw_exit = float(pos["ExitPrice"])
                shocked = raw_exit * (1 - RESEARCH_GAP) if pos["ExitReason"] == "STOP_GAP" else raw_exit
                net_exit = shocked * (1 - RESEARCH_SLIPPAGE) * (1 - RESEARCH_COST)
                proceeds = pos["Shares"] * net_exit
                pnl = proceeds - pos["EntryCash"]
                cash += proceeds
                closed.append({
                    **pos,
                    "NetExitPrice": net_exit,
                    "Proceeds": proceeds,
                    "PnL": pnl,
                })
            else:
                still_open.append(pos)
        open_positions = still_open

        for _, sig in by_date.get(current_date, pd.DataFrame()).iterrows():
            symbol = sig["Symbol"]
            if any(p["Symbol"] == symbol for p in open_positions):
                continue
            if len(open_positions) >= GLOBAL_MAX_POSITIONS:
                continue

            entry_price = float(sig["EntryPrice"])
            stop_price = float(sig["StopPrice"])
            pos_size = calculate_position_size(
                cash, entry_price, stop_price,
                risk_pct=GLOBAL_RISK_PCT, lot_size=LOT_SIZE,
            )
            shares = int(pos_size["shares"])
            if shares <= 0:
                continue

            net_entry = entry_price * (1 + RESEARCH_SLIPPAGE) * (1 + RESEARCH_COST)
            entry_cash = shares * net_entry
            if entry_cash > cash + 1e-9:
                continue

            cash -= entry_cash
            open_positions.append({
                "Symbol": symbol,
                "SignalDate": pd.Timestamp(sig["SignalDate"]),
                "EntryDate": current_date,
                "EntryPrice": entry_price,
                "StopPrice": stop_price,
                "Shares": shares,
                "Score": int(sig["Score"]),
                "ExitDate": pd.Timestamp(sig["ExitDate"]),
                "ExitPrice": float(sig["ExitPrice"]),
                "ExitReason": sig["ExitReason"],
                "EntryCash": entry_cash,
                "HoldingDays": int((pd.Timestamp(sig["ExitDate"]) - current_date).days),
            })

    for pos in open_positions:
        raw_exit = float(pos["ExitPrice"])
        shocked = raw_exit * (1 - RESEARCH_GAP) if pos["ExitReason"] == "STOP_GAP" else raw_exit
        net_exit = shocked * (1 - RESEARCH_SLIPPAGE) * (1 - RESEARCH_COST)
        proceeds = pos["Shares"] * net_exit
        cash += proceeds
        closed.append({
            **pos,
            "NetExitPrice": net_exit,
            "Proceeds": proceeds,
            "PnL": proceeds - pos["EntryCash"],
        })

    trades = pd.DataFrame(closed)
    return trades, float(cash)

def portfolio_metrics(trades):
    if trades.empty:
        return {"PnL": -GLOBAL_INITIAL_CAPITAL, "PF": 0.0, "Trades": 0, "WinRate": 0.0}
    gp = float(trades.loc[trades["PnL"] > 0, "PnL"].sum())
    gl = float(-trades.loc[trades["PnL"] < 0, "PnL"].sum())
    pf = gp / gl if gl > 0 else np.inf
    return {
        "PnL": float(trades["PnL"].sum()),
        "PF": float(pf),
        "Trades": int(len(trades)),
        "WinRate": float((trades["PnL"] > 0).mean() * 100),
    }

# ------------------------------------------------------------
# Three chronological nested OOS folds.
# Blocks 0..6 are used so every test period is unseen.
# ------------------------------------------------------------
dates = np.array(sorted(model_df["SignalDate"].dt.normalize().unique()))
blocks = np.array_split(dates, 7)

fold_specs = [
    (0, 2, 3, 4),
    (0, 3, 4, 5),
    (0, 4, 5, 6),
]

fold_results = []

for fold_no, (tr0, tr1, val, test) in enumerate(fold_specs, 1):
    train_dates = np.concatenate(blocks[tr0:tr1 + 1])
    val_dates = blocks[val]
    test_dates = blocks[test]

    train = model_df[model_df["SignalDate"].dt.normalize().isin(train_dates)].copy()
    val_df = model_df[model_df["SignalDate"].dt.normalize().isin(val_dates)].copy()
    test_df = model_df[model_df["SignalDate"].dt.normalize().isin(test_dates)].copy()

    # Train-only model.
    model = Pipeline([
        ("scale", StandardScaler()),
        ("logit", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ])
    model.fit(train[FEATURES], train["EarlyFailure"])

    val_df["RiskProb"] = model.predict_proba(val_df[FEATURES])[:, 1]
    test_df["RiskProb"] = model.predict_proba(test_df[FEATURES])[:, 1]

    # Threshold is selected on validation only.
    threshold_scores = []
    for threshold in THRESHOLDS:
        accepted_val = val_df[val_df["RiskProb"] <= threshold]
        vt, _ = run_global_research(accepted_val)
        vm = portfolio_metrics(vt)
        threshold_scores.append((threshold, vm["PnL"], vm["PF"], vm["Trades"]))

    threshold_scores.sort(key=lambda x: (x[1], x[2], -x[3]), reverse=True)
    selected_threshold = threshold_scores[0][0]

    # Frozen BASE on exactly the same unseen test period.
    base_test = test_df.copy()
    bt, _ = run_global_research(base_test)
    base_m = portfolio_metrics(bt)

    selected_test = test_df[test_df["RiskProb"] <= selected_threshold].copy()
    st, _ = run_global_research(selected_test)
    selected_m = portfolio_metrics(st)

    fold_results.append({
        "Fold": fold_no,
        "TrainRows": len(train),
        "ValRows": len(val_df),
        "TestRows": len(test_df),
        "Threshold": selected_threshold,
        "BasePnL": base_m["PnL"],
        "SelectedPnL": selected_m["PnL"],
        "DeltaPnL": selected_m["PnL"] - base_m["PnL"],
        "BasePF": base_m["PF"],
        "SelectedPF": selected_m["PF"],
        "SelectedTrades": selected_m["Trades"],
        "PositiveDelta": selected_m["PnL"] > base_m["PnL"],
        "PFgt1": selected_m["PF"] > 1.0,
    })

research_folds = pd.DataFrame(fold_results)

print("\n====================================")
print("📊 NESTED OOS — CAUSAL EARLY-FAILURE MODEL")
print("====================================")
display(research_folds)

passed_folds = int((research_folds["PositiveDelta"] & research_folds["PFgt1"]).sum())
print(f"Promotion gate: {passed_folds}/3 folds satisfy BOTH improvement and PF > 1.")

if passed_folds >= 2:
    print("EARLY_FAILURE_MODEL_GATE_PASS")
else:
    print("EARLY_FAILURE_MODEL_GATE_FAIL")

# Production is never changed by this research cell.
print("PRODUCTION_UNCHANGED")
print("CELL 13.0 PASSED")


🧪 CAUSAL EARLY-FAILURE RISK MODEL — FRESH RESEARCH


Model rows          : 8,185
Early failures      : 1,636
Early-failure rate  : 19.99%



📊 NESTED OOS — CAUSAL EARLY-FAILURE MODEL


,Fold,TrainRows,ValRows,TestRows,Threshold,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,PositiveDelta,PFgt1
0,1,3204,1112,788,0.6,-35584.089242,-40433.573136,-4849.483893,0.384778,0.250098,103,False,False
1,2,4316,788,1253,0.5,-10359.139806,-8117.236812,2241.902995,0.709959,0.737275,77,True,False
2,3,5104,1253,1828,0.6,-6916.194752,-14207.509217,-7291.314465,0.828951,0.684898,92,False,False


Promotion gate: 0/3 folds satisfy BOTH improvement and PF > 1.
EARLY_FAILURE_MODEL_GATE_FAIL
PRODUCTION_UNCHANGED
CELL 13.0 PASSED


In [ ]:
# ============================================================
# CELL 13.1 — CAUSAL ENTRY-QUALITY FILTER GRID
# Fresh research on the corrected 133-stock universe
# Production remains frozen
# ============================================================

print("=" * 90)
print("🧪 CAUSAL ENTRY-QUALITY FILTER GRID — FRESH RESEARCH")
print("=" * 90)

# Research contract:
# - only signal-date features are used to define a filter
# - future trade P&L is NEVER used to choose a filter
# - chronological nested OOS: train/validation/test
# - validation chooses one filter; test is unseen
# - realistic friction is already enforced by run_global_research()
# - minimum selected trades prevents tiny-sample winners
# - production parameters are never modified

assert len(PRODUCTION_SYMBOLS) == 133
assert "model_df" in globals()
assert "run_global_research" in globals()
assert "portfolio_metrics" in globals()

GRID_SCORE = [4, 5, 6, 7]
GRID_VOLUME = [0.0, 1.2, 1.5, 2.0]
GRID_GAP = [0.00, 0.01, 0.02, 0.03]
GRID_STOP_RISK = [np.inf, 0.08, 0.06, 0.04]
MIN_VALIDATION_TRADES = 25

base_research = model_df.copy()

# All filter variables are known at SignalDate.
# We deliberately do not include EarlyFailure, ReturnPct, ExitDate, or ExitPrice
# in the filter definition.
def apply_entry_filter(frame, min_score, min_volume, max_gap, max_stop_risk):
    return frame[
        (frame["Score"] >= min_score)
        & (frame["VolumeRatio"] >= min_volume)
        & (frame["EntryGapPct"] <= max_gap)
        & (frame["StopRiskPct"] <= max_stop_risk)
    ].copy()

filter_specs = []
for min_score in GRID_SCORE:
    for min_volume in GRID_VOLUME:
        for max_gap in GRID_GAP:
            for max_stop_risk in GRID_STOP_RISK:
                filter_specs.append({
                    "MinScore": min_score,
                    "MinVolumeRatio": min_volume,
                    "MaxEntryGapPct": max_gap,
                    "MaxStopRiskPct": max_stop_risk,
                })

print(f"Candidate filters: {len(filter_specs)}")
print(f"Validation minimum trades: {MIN_VALIDATION_TRADES}")

dates = np.array(sorted(base_research["SignalDate"].dt.normalize().unique()))
blocks = np.array_split(dates, 7)

fold_specs = [
    (0, 2, 3, 4),
    (0, 3, 4, 5),
    (0, 4, 5, 6),
]

grid_fold_results = []
grid_selected_specs = []

for fold_no, (tr0, tr1, val_block, test_block) in enumerate(fold_specs, 1):
    train_dates = np.concatenate(blocks[tr0:tr1 + 1])
    val_dates = blocks[val_block]
    test_dates = blocks[test_block]

    train = base_research[
        base_research["SignalDate"].dt.normalize().isin(train_dates)
    ].copy()
    val_df = base_research[
        base_research["SignalDate"].dt.normalize().isin(val_dates)
    ].copy()
    test_df = base_research[
        base_research["SignalDate"].dt.normalize().isin(test_dates)
    ].copy()

    # Training is used only as an information boundary here. The filter itself
    # is selected exclusively from validation performance; no test observation
    # can influence selection.
    assert train["SignalDate"].max() < val_df["SignalDate"].min()
    assert val_df["SignalDate"].max() < test_df["SignalDate"].min()

    val_scores = []

    for spec in filter_specs:
        accepted_val = apply_entry_filter(
            val_df,
            spec["MinScore"],
            spec["MinVolumeRatio"],
            spec["MaxEntryGapPct"],
            spec["MaxStopRiskPct"],
        )

        vt, _ = run_global_research(accepted_val)
        vm = portfolio_metrics(vt)

        # Prefer profitable validation samples with PF > 1. If none exist,
        # maximize P&L subject to a minimum sample size. This rule is fixed
        # before seeing any test results.
        val_scores.append({
            **spec,
            "ValPnL": vm["PnL"],
            "ValPF": vm["PF"],
            "ValTrades": vm["Trades"],
            "ValWinRate": vm["WinRate"],
        })

    val_table = pd.DataFrame(val_scores)
    eligible = val_table[val_table["ValTrades"] >= MIN_VALIDATION_TRADES].copy()
    eligible_pf = eligible[eligible["ValPF"] > 1.0].copy()

    if not eligible.empty:
        ranking_pool = eligible_pf if not eligible_pf.empty else eligible
        ranking_pool = ranking_pool.sort_values(
            ["ValPnL", "ValPF", "ValTrades"],
            ascending=[False, False, False],
        )
        selected = ranking_pool.iloc[0].to_dict()
    else:
        # No candidate with adequate validation sample: use frozen base.
        selected = {
            "MinScore": 4,
            "MinVolumeRatio": 0.0,
            "MaxEntryGapPct": 0.03,
            "MaxStopRiskPct": np.inf,
            "ValPnL": np.nan,
            "ValPF": np.nan,
            "ValTrades": 0,
            "ValWinRate": np.nan,
        }

    selected_spec = {
        "MinScore": int(selected["MinScore"]),
        "MinVolumeRatio": float(selected["MinVolumeRatio"]),
        "MaxEntryGapPct": float(selected["MaxEntryGapPct"]),
        "MaxStopRiskPct": float(selected["MaxStopRiskPct"]),
    }
    grid_selected_specs.append(selected_spec)

    # Frozen base and selected filter are both evaluated on the identical,
    # completely unseen test period.
    base_test = test_df.copy()
    bt, _ = run_global_research(base_test)
    base_m = portfolio_metrics(bt)

    selected_test = apply_entry_filter(test_df, **{
        "min_score": selected_spec["MinScore"],
        "min_volume": selected_spec["MinVolumeRatio"],
        "max_gap": selected_spec["MaxEntryGapPct"],
        "max_stop_risk": selected_spec["MaxStopRiskPct"],
    })
    st, _ = run_global_research(selected_test)
    selected_m = portfolio_metrics(st)

    grid_fold_results.append({
        "Fold": fold_no,
        "TrainRows": len(train),
        "ValRows": len(val_df),
        "TestRows": len(test_df),
        "SelectedMinScore": selected_spec["MinScore"],
        "SelectedMinVolumeRatio": selected_spec["MinVolumeRatio"],
        "SelectedMaxEntryGapPct": selected_spec["MaxEntryGapPct"] * 100,
        "SelectedMaxStopRiskPct": (
            selected_spec["MaxStopRiskPct"] * 100
            if np.isfinite(selected_spec["MaxStopRiskPct"]) else np.inf
        ),
        "ValPnL": float(selected["ValPnL"]) if pd.notna(selected["ValPnL"]) else np.nan,
        "ValPF": float(selected["ValPF"]) if pd.notna(selected["ValPF"]) else np.nan,
        "ValTrades": int(selected["ValTrades"]),
        "BasePnL": base_m["PnL"],
        "SelectedPnL": selected_m["PnL"],
        "DeltaPnL": selected_m["PnL"] - base_m["PnL"],
        "BasePF": base_m["PF"],
        "SelectedPF": selected_m["PF"],
        "SelectedTrades": selected_m["Trades"],
        "SelectedWinRate": selected_m["WinRate"],
        "PositiveDelta": selected_m["PnL"] > base_m["PnL"],
        "PFgt1": selected_m["PF"] > 1.0,
    })

research_entry_grid_folds = pd.DataFrame(grid_fold_results)

print("\n====================================")
print("📊 NESTED OOS — ENTRY-QUALITY FILTER GRID")
print("====================================")
display(research_entry_grid_folds)

passed = int(
    (research_entry_grid_folds["PositiveDelta"] & research_entry_grid_folds["PFgt1"]).sum()
)
print(f"Promotion gate: {passed}/3 folds satisfy BOTH improvement and PF > 1.")

if passed >= 2:
    print("ENTRY_QUALITY_GRID_GATE_PASS")
else:
    print("ENTRY_QUALITY_GRID_GATE_FAIL")

print("\nSELECTED VALIDATION FILTERS")
display(pd.DataFrame(grid_selected_specs))

# Integrity: the selected filter must never have been chosen using test outcomes.
assert len(research_entry_grid_folds) == 3
assert all(
    spec["MinScore"] >= 4
    and spec["MinVolumeRatio"] >= 0
    and spec["MaxEntryGapPct"] >= 0
    and spec["MaxStopRiskPct"] > 0
    for spec in grid_selected_specs
)

# Production remains frozen regardless of research outcome.
print("\nPRODUCTION_UNCHANGED")
print("CELL 13.1 PASSED")



🧪 CAUSAL ENTRY-QUALITY FILTER GRID — FRESH RESEARCH
Candidate filters: 256
Validation minimum trades: 25



📊 NESTED OOS — ENTRY-QUALITY FILTER GRID


,Fold,TrainRows,ValRows,TestRows,SelectedMinScore,SelectedMinVolumeRatio,SelectedMaxEntryGapPct,SelectedMaxStopRiskPct,ValPnL,ValPF,ValTrades,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,SelectedWinRate,PositiveDelta,PFgt1
0,1,3204,1112,788,5,2.0,3.0,inf,3261.274043,1.140759,42,-35584.089242,-11182.726542,24401.362701,0.384778,0.401634,19,10.526316,True,False
1,2,4316,788,1253,4,2.0,0.0,4.0,-8432.394002,0.685446,41,-10359.139806,-12174.225881,-1815.086075,0.709959,0.498416,47,27.659574,False,False
2,3,5104,1253,1828,4,1.5,3.0,inf,11390.982676,1.376080,87,-6916.194752,2127.676253,9043.871005,0.828951,1.059052,96,39.583333,True,True


Promotion gate: 1/3 folds satisfy BOTH improvement and PF > 1.
ENTRY_QUALITY_GRID_GATE_FAIL

SELECTED VALIDATION FILTERS


,MinScore,MinVolumeRatio,MaxEntryGapPct,MaxStopRiskPct
0,5,2.0,0.03,inf
1,4,2.0,0.00,0.04
2,4,1.5,0.03,inf



PRODUCTION_UNCHANGED
CELL 13.1 PASSED


In [ ]:
# ============================================================
# CELL 13.2 — CAUSAL PORTFOLIO PRIORITY / CAPACITY RESEARCH
# Fresh research: improve global portfolio selection without changing production
# ============================================================

print("=" * 90)
print("🧪 CAUSAL PORTFOLIO PRIORITY / CAPACITY RESEARCH")
print("=" * 90)

# The 133-stock universe generates many simultaneous candidates while the
# True Global Portfolio has a hard capacity limit. This cell tests whether
# choosing the strongest signal-date candidates improves the portfolio.
# Only information known on SignalDate is used for ranking.
# No exit/P&L/outcome field is used for selection.

assert len(PRODUCTION_SYMBOLS) == 133
assert "model_df" in globals()
assert "run_global_research" in globals()
assert "portfolio_metrics" in globals()

RANK_METHODS = [
    "SCORE_DESC",
    "SCORE_DESC_VOLUME_DESC",
    "SCORE_DESC_STOP_RISK_ASC",
    "SCORE_DESC_GAP_ASC",
    "SCORE_DESC_VOLUME_DESC_STOP_RISK_ASC",
]
TOP_N = [1, 2, 3, 5, 8, 10]
MIN_VALIDATION_TRADES = 25

candidate = model_df.copy()
candidate["SignalDate"] = pd.to_datetime(candidate["SignalDate"]).dt.normalize()

# Signal-date-only ranking. Lower gap and lower stop risk are preferable;
# higher score and higher volume confirmation are preferable.
def rank_candidates(frame, method, top_n):
    x = frame.copy()
    if method == "SCORE_DESC":
        sort_cols, asc = ["Score", "Symbol"], [False, True]
    elif method == "SCORE_DESC_VOLUME_DESC":
        sort_cols, asc = ["Score", "VolumeRatio", "Symbol"], [False, False, True]
    elif method == "SCORE_DESC_STOP_RISK_ASC":
        sort_cols, asc = ["Score", "StopRiskPct", "Symbol"], [False, True, True]
    elif method == "SCORE_DESC_GAP_ASC":
        sort_cols, asc = ["Score", "EntryGapPct", "Symbol"], [False, True, True]
    elif method == "SCORE_DESC_VOLUME_DESC_STOP_RISK_ASC":
        sort_cols, asc = ["Score", "VolumeRatio", "StopRiskPct", "EntryGapPct", "Symbol"], [False, False, True, True, True]
    else:
        raise ValueError(method)
    x = x.sort_values(["SignalDate"] + sort_cols, ascending=[True] + asc, kind="mergesort")
    return x.groupby("SignalDate", sort=False, group_keys=False).head(top_n).copy()

rank_specs = [(m, n) for m in RANK_METHODS for n in TOP_N]
print(f"Candidate ranking variants: {len(rank_specs)}")
print(f"Validation minimum trades: {MIN_VALIDATION_TRADES}")

# Seven chronological blocks; selection occurs on validation only.
dates = np.array(sorted(candidate["SignalDate"].unique()))
blocks = np.array_split(dates, 7)
fold_specs = [(0, 2, 3, 4), (0, 3, 4, 5), (0, 4, 5, 6)]

fold_results = []
selected_specs = []

for fold_no, (tr0, tr1, val_block, test_block) in enumerate(fold_specs, 1):
    train_dates = np.concatenate(blocks[tr0:tr1 + 1])
    val_dates = blocks[val_block]
    test_dates = blocks[test_block]
    train = candidate[candidate["SignalDate"].isin(train_dates)].copy()
    val_df = candidate[candidate["SignalDate"].isin(val_dates)].copy()
    test_df = candidate[candidate["SignalDate"].isin(test_dates)].copy()

    assert train["SignalDate"].max() < val_df["SignalDate"].min()
    assert val_df["SignalDate"].max() < test_df["SignalDate"].min()

    scores = []
    for method, top_n in rank_specs:
        selected_val = rank_candidates(val_df, method, top_n)
        vt, _ = run_global_research(selected_val)
        vm = portfolio_metrics(vt)
        scores.append({
            "RankMethod": method,
            "TopN": top_n,
            "ValPnL": vm["PnL"],
            "ValPF": vm["PF"],
            "ValTrades": vm["Trades"],
            "ValWinRate": vm["WinRate"],
        })

    score_table = pd.DataFrame(scores)
    eligible = score_table[score_table["ValTrades"] >= MIN_VALIDATION_TRADES].copy()
    profitable_pf = eligible[eligible["ValPF"] > 1.0].copy()

    if not eligible.empty:
        pool = profitable_pf if not profitable_pf.empty else eligible
        pool = pool.sort_values(["ValPnL", "ValPF", "ValTrades"], ascending=[False, False, False])
        selected = pool.iloc[0].to_dict()
    else:
        selected = {"RankMethod": "SCORE_DESC", "TopN": 1, "ValPnL": np.nan, "ValPF": np.nan, "ValTrades": 0, "ValWinRate": np.nan}

    method = str(selected["RankMethod"])
    top_n = int(selected["TopN"])
    selected_specs.append({"Fold": fold_no, "RankMethod": method, "TopN": top_n})

    # Identical unseen test period: frozen base vs selected ranking.
    bt, _ = run_global_research(test_df)
    base_m = portfolio_metrics(bt)
    selected_test = rank_candidates(test_df, method, top_n)
    st, _ = run_global_research(selected_test)
    selected_m = portfolio_metrics(st)

    fold_results.append({
        "Fold": fold_no,
        "TrainRows": len(train),
        "ValRows": len(val_df),
        "TestRows": len(test_df),
        "RankMethod": method,
        "TopN": top_n,
        "ValPnL": float(selected["ValPnL"]) if pd.notna(selected["ValPnL"]) else np.nan,
        "ValPF": float(selected["ValPF"]) if pd.notna(selected["ValPF"]) else np.nan,
        "ValTrades": int(selected["ValTrades"]),
        "BasePnL": base_m["PnL"],
        "SelectedPnL": selected_m["PnL"],
        "DeltaPnL": selected_m["PnL"] - base_m["PnL"],
        "BasePF": base_m["PF"],
        "SelectedPF": selected_m["PF"],
        "SelectedTrades": selected_m["Trades"],
        "SelectedWinRate": selected_m["WinRate"],
        "PositiveDelta": selected_m["PnL"] > base_m["PnL"],
        "PFgt1": selected_m["PF"] > 1.0,
    })

research_portfolio_priority = pd.DataFrame(fold_results)
print("\n====================================")
print("📊 NESTED OOS — PORTFOLIO PRIORITY / CAPACITY")
print("====================================")
display(research_portfolio_priority)

passed = int((research_portfolio_priority["PositiveDelta"] & research_portfolio_priority["PFgt1"]).sum())
print(f"Promotion gate: {passed}/3 folds satisfy BOTH improvement and PF > 1.")
if passed >= 2:
    print("PORTFOLIO_PRIORITY_GATE_PASS")
else:
    print("PORTFOLIO_PRIORITY_GATE_FAIL")

print("\nSELECTED VALIDATION RANKING RULES")
display(pd.DataFrame(selected_specs))

# Integrity checks: chronological separation and causal-only ranking.
assert len(research_portfolio_priority) == 3
assert all(spec["TopN"] in TOP_N for spec in selected_specs)
assert all(spec["RankMethod"] in RANK_METHODS for spec in selected_specs)

# Production remains frozen regardless of research outcome.
print("\nPRODUCTION_UNCHANGED")
print("CELL 13.2 PASSED")



🧪 CAUSAL PORTFOLIO PRIORITY / CAPACITY RESEARCH
Candidate ranking variants: 30
Validation minimum trades: 25



📊 NESTED OOS — PORTFOLIO PRIORITY / CAPACITY


,Fold,TrainRows,ValRows,TestRows,RankMethod,TopN,ValPnL,ValPF,ValTrades,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,SelectedWinRate,PositiveDelta,PFgt1
0,1,3204,1112,788,SCORE_DESC_VOLUME_DESC,2,-16435.677453,0.669030,111,-35584.089242,-31401.910246,4182.178997,0.384778,0.394415,105,20.952381,True,False
1,2,4316,788,1253,SCORE_DESC_VOLUME_DESC,1,-18726.383517,0.587992,90,-10359.139806,-17318.077063,-6958.937256,0.709959,0.550429,87,39.080460,False,False
2,3,5104,1253,1828,SCORE_DESC_STOP_RISK_ASC,1,1100.645028,1.033605,81,-6916.194752,-21617.452723,-14701.257971,0.828951,0.453585,84,33.333333,False,False


Promotion gate: 0/3 folds satisfy BOTH improvement and PF > 1.
PORTFOLIO_PRIORITY_GATE_FAIL

SELECTED VALIDATION RANKING RULES


,Fold,RankMethod,TopN
0,1,SCORE_DESC_VOLUME_DESC,2
1,2,SCORE_DESC_VOLUME_DESC,1
2,3,SCORE_DESC_STOP_RISK_ASC,1



PRODUCTION_UNCHANGED
CELL 13.2 PASSED


In [ ]:
# ============================================================
# CELL 13.3 — CAUSAL EXIT / HOLDING POLICY NESTED OOS
# Fresh research on corrected 133-stock universe; production frozen
# ============================================================

print("=" * 90)
print("🧪 CAUSAL EXIT / HOLDING POLICY NESTED OOS")
print("=" * 90)

# Research only: stop logic remains the frozen production stop.
# We vary Target2 and maximum holding period because these are exit
# decisions. Selection is validation-only; test periods remain unseen.

assert len(PRODUCTION_SYMBOLS) == 133
assert "model_df" in globals()
assert "run_global_research" in globals()
assert "portfolio_metrics" in globals()

EXIT_GRID = [
    {"Target2Pct": 0.10, "MaxHoldDays": 10},
    {"Target2Pct": 0.10, "MaxHoldDays": 15},
    {"Target2Pct": 0.10, "MaxHoldDays": 20},
    {"Target2Pct": 0.10, "MaxHoldDays": 30},
    {"Target2Pct": 0.15, "MaxHoldDays": 10},
    {"Target2Pct": 0.15, "MaxHoldDays": 15},
    {"Target2Pct": 0.15, "MaxHoldDays": 20},
    {"Target2Pct": 0.15, "MaxHoldDays": 30},
    {"Target2Pct": 0.20, "MaxHoldDays": 10},
    {"Target2Pct": 0.20, "MaxHoldDays": 15},
    {"Target2Pct": 0.20, "MaxHoldDays": 20},
    {"Target2Pct": 0.20, "MaxHoldDays": 30},
]
MIN_VALIDATION_TRADES = 25

def run_exit_policy(candidates, target2_pct, max_hold_days):
    """True Global Portfolio runner with frozen stop/entry sizing and causal exit policy."""
    if candidates.empty:
        return pd.DataFrame(), float(GLOBAL_INITIAL_CAPITAL)
    cash = float(GLOBAL_INITIAL_CAPITAL)
    open_positions, closed = [], []
    by_date = {d:g.sort_values(["Score","Symbol"], ascending=[False,True]) for d,g in candidates.groupby("EntryDate")}
    event_dates = sorted(set(candidates["EntryDate"]))
    for current_date in event_dates:
        current_date = pd.Timestamp(current_date)
        still=[]
        for pos in open_positions:
            if pd.Timestamp(pos["ExitDate"]) <= current_date:
                raw_exit=float(pos["ExitPrice"])
                shocked=raw_exit*(1-RESEARCH_GAP) if pos["ExitReason"]=="STOP_GAP" else raw_exit
                net_exit=shocked*(1-RESEARCH_SLIPPAGE)*(1-RESEARCH_COST)
                proceeds=pos["Shares"]*net_exit
                cash += proceeds
                closed.append({**pos,"NetExitPrice":net_exit,"Proceeds":proceeds,"PnL":proceeds-pos["EntryCash"]})
            else: still.append(pos)
        open_positions=still
        for _, sig in by_date.get(current_date, pd.DataFrame()).iterrows():
            if any(p["Symbol"]==sig["Symbol"] for p in open_positions) or len(open_positions)>=GLOBAL_MAX_POSITIONS: continue
            entry_price=float(sig["EntryPrice"]); stop_price=float(sig["StopPrice"])
            size=calculate_position_size(cash,entry_price,stop_price,risk_pct=GLOBAL_RISK_PCT,lot_size=LOT_SIZE)
            if size["shares"]<=0: continue
            net_entry=entry_price*(1+RESEARCH_SLIPPAGE)*(1+RESEARCH_COST)
            entry_cash=int(size["shares"])*net_entry
            if entry_cash>cash+1e-9: continue
            df=feature_data.get(sig["Symbol"])
            if df is None or current_date not in df.index: continue
            future=df.loc[df.index>=current_date].copy()
            max_date=current_date+pd.Timedelta(days=int(max_hold_days))
            future=future.loc[future.index<=max_date]
            if future.empty: continue
            target2=entry_price*(1+float(target2_pct))
            exit_date=pd.NaT; exit_price=np.nan; exit_reason=None
            for d,bar in future.iterrows():
                op=float(bar["Open"]); hi=float(bar["High"]); lo=float(bar["Low"]); cl=float(bar["Close"])
                if op<=stop_price: exit_date=d; exit_price=op; exit_reason="STOP_GAP"; break
                if lo<=stop_price: exit_date=d; exit_price=stop_price; exit_reason="STOP"; break
                if op>=target2: exit_date=d; exit_price=op; exit_reason="TARGET2_GAP"; break
                if hi>=target2: exit_date=d; exit_price=target2; exit_reason="TARGET2"; break
                if d>=max_date: exit_date=d; exit_price=cl; exit_reason="MAX_HOLD"; break
            if pd.isna(exit_date): exit_date=future.index[-1]; exit_price=float(future.iloc[-1]["Close"]); exit_reason="EOD"
            open_positions.append({"Symbol":sig["Symbol"],"SignalDate":pd.Timestamp(sig["SignalDate"]),"EntryDate":current_date,"EntryPrice":entry_price,"StopPrice":stop_price,"Shares":int(size["shares"]),"Score":int(sig["Score"]),"ExitDate":pd.Timestamp(exit_date),"ExitPrice":float(exit_price),"ExitReason":exit_reason,"EntryCash":float(entry_cash),"HoldingDays":int((pd.Timestamp(exit_date)-current_date).days)})
            cash-=entry_cash
    # Realize remaining positions using their already determined causal exits.
    for pos in open_positions:
        raw_exit=float(pos["ExitPrice"]); shocked=raw_exit*(1-RESEARCH_GAP) if pos["ExitReason"]=="STOP_GAP" else raw_exit
        net_exit=shocked*(1-RESEARCH_SLIPPAGE)*(1-RESEARCH_COST); proceeds=pos["Shares"]*net_exit; cash+=proceeds
        closed.append({**pos,"NetExitPrice":net_exit,"Proceeds":proceeds,"PnL":proceeds-pos["EntryCash"]})
    return pd.DataFrame(closed), float(cash)

dates=np.array(sorted(pd.to_datetime(model_df["SignalDate"]).dt.normalize().unique()))
blocks=np.array_split(dates,7)
fold_specs=[(0,2,3,4),(0,3,4,5),(0,4,5,6)]
fold_results=[]; selected_specs=[]

for fold_no,(tr0,tr1,val_block,test_block) in enumerate(fold_specs,1):
    train_dates=np.concatenate(blocks[tr0:tr1+1]); val_dates=blocks[val_block]; test_dates=blocks[test_block]
    train=model_df[pd.to_datetime(model_df["SignalDate"]).dt.normalize().isin(train_dates)].copy()
    val_df=model_df[pd.to_datetime(model_df["SignalDate"]).dt.normalize().isin(val_dates)].copy()
    test_df=model_df[pd.to_datetime(model_df["SignalDate"]).dt.normalize().isin(test_dates)].copy()
    assert train["SignalDate"].max()<val_df["SignalDate"].min() and val_df["SignalDate"].max()<test_df["SignalDate"].min()
    scores=[]
    for spec in EXIT_GRID:
        vt,_=run_exit_policy(val_df,spec["Target2Pct"],spec["MaxHoldDays"]); vm=portfolio_metrics(vt)
        scores.append({**spec,"ValPnL":vm["PnL"],"ValPF":vm["PF"],"ValTrades":vm["Trades"]})
    eligible=pd.DataFrame(scores); eligible=eligible[eligible["ValTrades"]>=MIN_VALIDATION_TRADES]
    if eligible.empty: selected={**EXIT_GRID[0],"ValPnL":np.nan,"ValPF":np.nan,"ValTrades":0}
    else:
        profitable=eligible[eligible["ValPF"]>1.0]; pool=profitable if not profitable.empty else eligible
        selected=pool.sort_values(["ValPnL","ValPF","ValTrades"],ascending=[False,False,False]).iloc[0].to_dict()
    selected_specs.append({"Fold":fold_no,"Target2Pct":float(selected["Target2Pct"]),"MaxHoldDays":int(selected["MaxHoldDays"])})
    bt,_=run_exit_policy(test_df,PRODUCTION_TARGET2_PCT,PRODUCTION_MAX_HOLDING_DAYS); base_m=portfolio_metrics(bt)
    st,_=run_exit_policy(test_df,float(selected["Target2Pct"]),int(selected["MaxHoldDays"])); selected_m=portfolio_metrics(st)
    fold_results.append({"Fold":fold_no,"TrainRows":len(train),"ValRows":len(val_df),"TestRows":len(test_df),"SelectedTarget2Pct":float(selected["Target2Pct"]),"SelectedMaxHoldDays":int(selected["MaxHoldDays"]),"ValPnL":float(selected["ValPnL"]) if pd.notna(selected["ValPnL"]) else np.nan,"ValPF":float(selected["ValPF"]) if pd.notna(selected["ValPF"]) else np.nan,"ValTrades":int(selected["ValTrades"]),"BasePnL":base_m["PnL"],"SelectedPnL":selected_m["PnL"],"DeltaPnL":selected_m["PnL"]-base_m["PnL"],"BasePF":base_m["PF"],"SelectedPF":selected_m["PF"],"SelectedTrades":selected_m["Trades"],"PositiveDelta":selected_m["PnL"]>base_m["PnL"],"PFgt1":selected_m["PF"]>1.0})

research_exit=pd.DataFrame(fold_results)
display(research_exit)
passed=int((research_exit["PositiveDelta"]&research_exit["PFgt1"]).sum())
print(f"Promotion gate: {passed}/3 folds satisfy BOTH improvement and PF > 1.")
print("EXIT_POLICY_GATE_PASS" if passed>=2 else "EXIT_POLICY_GATE_FAIL")
print("\nSELECTED EXIT POLICIES")
display(pd.DataFrame(selected_specs))
print("\nPRODUCTION_UNCHANGED")
print("CELL 13.3 PASSED")


🧪 CAUSAL EXIT / HOLDING POLICY NESTED OOS


,Fold,TrainRows,ValRows,TestRows,SelectedTarget2Pct,SelectedMaxHoldDays,ValPnL,ValPF,ValTrades,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,PositiveDelta,PFgt1
0,1,3204,1112,788,0.20,20,-27845.658844,0.485171,111,-35584.089242,-36027.865833,-443.776591,0.384778,0.382029,120,False,False
1,2,4316,788,1253,0.20,30,-23153.997476,0.517794,87,-10359.139806,793.516436,11152.656242,0.709959,1.035395,70,True,True
2,3,5104,1253,1828,0.15,30,1040.150213,1.036702,73,-6916.194752,4305.392984,11221.587736,0.828951,1.178276,70,True,True


Promotion gate: 2/3 folds satisfy BOTH improvement and PF > 1.
EXIT_POLICY_GATE_PASS

SELECTED EXIT POLICIES


,Fold,Target2Pct,MaxHoldDays
0,1,0.20,20
1,2,0.20,30
2,3,0.15,30



PRODUCTION_UNCHANGED
CELL 13.3 PASSED


In [ ]:
# ============================================================
# CELL 13.4 — CAUSAL MARKET-REGIME / BREADTH FILTER NESTED OOS
# Fresh research branch; production remains frozen.
# ============================================================

print("=" * 90)
print("🧪 CAUSAL MARKET-REGIME / BREADTH FILTER NESTED OOS")
print("=" * 90)

assert len(PRODUCTION_SYMBOLS) == 133
assert "model_df" in globals()
assert "run_global_research" in globals()
assert "portfolio_metrics" in globals()
assert "feature_data" in globals()

# Causal equal-weight regime from the same 133-stock universe.
# Breadth75 = fraction of symbols with Close > EMA75.
# Return20  = equal-weight mean of each symbol's 20-session return.
regime_rows = []
for symbol in PRODUCTION_SYMBOLS:
    df = feature_data.get(symbol)
    if df is None or df.empty or "Close" not in df.columns:
        continue
    x = df.copy()
    close = pd.to_numeric(x["Close"], errors="coerce")
    ema75 = close.ewm(span=75, adjust=False, min_periods=75).mean()
    ret20 = close.pct_change(20)
    tmp = pd.DataFrame({
        "Date": pd.to_datetime(x.index).normalize(),
        "AboveEMA75": (close > ema75).astype(float).values,
        "Return20": ret20.values,
    }).dropna()
    if not tmp.empty:
        regime_rows.append(tmp)

assert regime_rows, "No causal regime observations could be constructed."

regime_long = pd.concat(regime_rows, ignore_index=True)
regime = (
    regime_long.groupby("Date", as_index=False)
    .agg(Breadth75=("AboveEMA75", "mean"), Return20=("Return20", "mean"))
    .sort_values("Date")
    .reset_index(drop=True)
)

research_candidates = model_df.copy()
research_candidates["EntryDate"] = pd.to_datetime(research_candidates["EntryDate"]).dt.normalize()
research_candidates = research_candidates.merge(regime, left_on="EntryDate", right_on="Date", how="inner")
research_candidates.drop(columns=["Date"], inplace=True)

assert len(research_candidates) > 0
assert research_candidates["Breadth75"].notna().all()
assert research_candidates["Return20"].notna().all()

BREADTH_GRID = [0.00, 0.35, 0.45, 0.55, 0.65]
RETURN20_GRID = [-np.inf, -0.05, 0.00, 0.05, 0.10]
REGIME_GRID = [{"MinBreadth75": b, "MinReturn20": r} for b in BREADTH_GRID for r in RETURN20_GRID]
MIN_VALIDATION_TRADES = 25

dates = np.array(sorted(pd.to_datetime(research_candidates["SignalDate"]).dt.normalize().unique()))
blocks = np.array_split(dates, 7)
fold_specs = [(0, 2, 3, 4), (0, 3, 4, 5), (0, 4, 5, 6)]

fold_results = []
selected_specs = []

for fold_no, (tr0, tr1, val_block, test_block) in enumerate(fold_specs, 1):
    train_dates = np.concatenate(blocks[tr0:tr1 + 1])
    val_dates = blocks[val_block]
    test_dates = blocks[test_block]

    train = research_candidates[
        pd.to_datetime(research_candidates["SignalDate"]).dt.normalize().isin(train_dates)
    ].copy()
    val_df = research_candidates[
        pd.to_datetime(research_candidates["SignalDate"]).dt.normalize().isin(val_dates)
    ].copy()
    test_df = research_candidates[
        pd.to_datetime(research_candidates["SignalDate"]).dt.normalize().isin(test_dates)
    ].copy()

    assert train["SignalDate"].max() < val_df["SignalDate"].min()
    assert val_df["SignalDate"].max() < test_df["SignalDate"].min()

    scores = []
    for spec in REGIME_GRID:
        val_filtered = val_df[
            (val_df["Breadth75"] >= spec["MinBreadth75"]) &
            (val_df["Return20"] >= spec["MinReturn20"])
        ].copy()
        vt, _ = run_global_research(val_filtered)
        vm = portfolio_metrics(vt)
        scores.append({
            **spec,
            "ValPnL": vm["PnL"],
            "ValPF": vm["PF"],
            "ValTrades": vm["Trades"],
        })

    eligible = pd.DataFrame(scores)
    eligible = eligible[eligible["ValTrades"] >= MIN_VALIDATION_TRADES].copy()

    if eligible.empty:
        selected = {"MinBreadth75": 0.00, "MinReturn20": -np.inf, "ValPnL": np.nan, "ValPF": np.nan, "ValTrades": 0}
    else:
        profitable = eligible[eligible["ValPF"] > 1.0]
        pool = profitable if not profitable.empty else eligible
        selected = pool.sort_values(["ValPnL", "ValPF", "ValTrades"], ascending=[False, False, False]).iloc[0].to_dict()

    selected_specs.append({
        "Fold": fold_no,
        "MinBreadth75": float(selected["MinBreadth75"]),
        "MinReturn20": float(selected["MinReturn20"]),
    })

    base_test, _ = run_global_research(test_df)
    base_m = portfolio_metrics(base_test)

    selected_test_df = test_df[
        (test_df["Breadth75"] >= float(selected["MinBreadth75"])) &
        (test_df["Return20"] >= float(selected["MinReturn20"]))
    ].copy()
    selected_test, _ = run_global_research(selected_test_df)
    selected_m = portfolio_metrics(selected_test)

    fold_results.append({
        "Fold": fold_no,
        "TrainRows": len(train),
        "ValRows": len(val_df),
        "TestRows": len(test_df),
        "SelectedMinBreadth75": float(selected["MinBreadth75"]),
        "SelectedMinReturn20": float(selected["MinReturn20"]),
        "ValPnL": float(selected["ValPnL"]) if pd.notna(selected["ValPnL"]) else np.nan,
        "ValPF": float(selected["ValPF"]) if pd.notna(selected["ValPF"]) else np.nan,
        "ValTrades": int(selected["ValTrades"]),
        "BasePnL": base_m["PnL"],
        "SelectedPnL": selected_m["PnL"],
        "DeltaPnL": selected_m["PnL"] - base_m["PnL"],
        "BasePF": base_m["PF"],
        "SelectedPF": selected_m["PF"],
        "SelectedTrades": selected_m["Trades"],
        "PositiveDelta": selected_m["PnL"] > base_m["PnL"],
        "PFgt1": selected_m["PF"] > 1.0,
    })

research_regime = pd.DataFrame(fold_results)
display(research_regime)

passed = int((research_regime["PositiveDelta"] & research_regime["PFgt1"]).sum())
print(f"Promotion gate: {passed}/3 folds satisfy BOTH improvement and PF > 1.")
print("REGIME_FILTER_GATE_PASS" if passed >= 2 else "REGIME_FILTER_GATE_FAIL")

print("\nSELECTED REGIME FILTERS")
display(pd.DataFrame(selected_specs))

print("\nREGIME DATA AUDIT")
print(f"Regime observations: {len(regime):,}")
print(f"Candidate rows retained after causal merge: {len(research_candidates):,}")
print(f"Breadth75 range: {regime['Breadth75'].min():.3f}–{regime['Breadth75'].max():.3f}")
print(f"Return20 range: {regime['Return20'].min():.3%}–{regime['Return20'].max():.3%}")

print("\nPRODUCTION_UNCHANGED")
print("CELL 13.4 PASSED")


🧪 CAUSAL MARKET-REGIME / BREADTH FILTER NESTED OOS


,Fold,TrainRows,ValRows,TestRows,SelectedMinBreadth75,SelectedMinReturn20,ValPnL,ValPF,ValTrades,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,PositiveDelta,PFgt1
0,1,3151,1031,778,0.45,-inf,-3671.479476,0.835505,55,-33377.446667,-1731.167066,31646.279601,0.456800,0.851338,19,True,False
1,2,4182,778,1269,0.35,-inf,-5004.533655,0.710026,30,-9078.574537,-1913.904606,7164.669931,0.766443,0.921361,59,True,False
2,3,4960,1269,1800,0.00,0.0,12904.676253,1.867866,44,4108.486635,-10860.056326,-14968.542961,1.125394,0.690369,83,False,False


Promotion gate: 0/3 folds satisfy BOTH improvement and PF > 1.
REGIME_FILTER_GATE_FAIL

SELECTED REGIME FILTERS


,Fold,MinBreadth75,MinReturn20
0,1,0.45,-inf
1,2,0.35,-inf
2,3,0.00,0.0



REGIME DATA AUDIT
Regime observations: 1,178
Candidate rows retained after causal merge: 8,029
Breadth75 range: 0.000–0.955
Return20 range: -11.596%–13.675%

PRODUCTION_UNCHANGED
CELL 13.4 PASSED


In [ ]:
# ============================================================
# CELL 13.5 — CAUSAL STOP-MULTIPLIER / EXIT SURFACE NESTED OOS
# Fresh research: stop distance + target + holding period
# Production remains frozen.
# ============================================================

print("=" * 90)
print("🧪 CAUSAL STOP / EXIT SURFACE NESTED OOS")
print("=" * 90)

assert len(PRODUCTION_SYMBOLS) == 133
assert "model_df" in globals()
assert "run_global_research" in globals()
assert "portfolio_metrics" in globals()
assert "feature_data" in globals()
assert "canonical_data" in globals()

THAI_TZ = "Asia/Bangkok"

# IMPORTANT:
# Yahoo daily OHLCV indexes are normally exchange-calendar labels when naive.
# A naive daily label must NOT be interpreted as UTC and shifted by +7 hours.
# Only timezone-aware timestamps are converted to Bangkok.  We then compare
# all daily research data by the resulting Bangkok calendar date.
def thai_trading_day(value):
    ts = pd.Timestamp(value)
    if pd.isna(ts):
        return pd.NaT
    if ts.tzinfo is None:
        return ts.normalize()
    return ts.tz_convert(THAI_TZ).tz_localize(None).normalize()

def thai_trading_frame(df):
    out = df.copy()
    idx = pd.to_datetime(out.index, errors="coerce")
    valid = ~idx.isna()
    out = out.loc[valid].copy()
    idx = idx[valid]
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_convert(THAI_TZ).tz_localize(None)
    else:
        idx = idx.normalize()
    out.index = idx
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out

required_cols = {"SignalDate", "EntryDate", "EntryPrice", "Score", "Symbol"}
assert required_cols.issubset(model_df.columns), sorted(required_cols - set(model_df.columns))

# Preserve the complete canonical trade row, including ExitDate/ExitPrice/
# ExitReason/StopPrice. The frozen-base comparator requires these fields.
surface_base = model_df.copy()
surface_base["SignalDate"] = surface_base["SignalDate"].map(thai_trading_day)
surface_base["EntryDate"] = surface_base["EntryDate"].map(thai_trading_day)
surface_base["ExitDate"] = surface_base["ExitDate"].map(thai_trading_day)
surface_base = surface_base.dropna(
    subset=["SignalDate", "EntryDate", "ExitDate", "Symbol"]
)

surface_rows = []
lookup_misses = 0
invalid_rows = 0
missing_symbols = set()
canonical_cache = {}

for symbol in surface_base["Symbol"].dropna().astype(str).unique():
    df = canonical_data.get(symbol)
    if df is None or df.empty:
        missing_symbols.add(symbol)
        continue
    local = thai_trading_frame(df)
    if "RECENT_LOW_5" not in local.columns:
        raise AssertionError(f"{symbol}: RECENT_LOW_5 missing from canonical_data")
    canonical_cache[symbol] = local

for _, rec in surface_base.iterrows():
    symbol = str(rec["Symbol"])
    signal_date = rec["SignalDate"]
    local = canonical_cache.get(symbol)

    if local is None:
        lookup_misses += 1
        continue

    # Daily bars are keyed by Bangkok trading calendar date, never by an
    # exact timezone-bearing timestamp. This prevents UTC/Bangkok mismatches.
    if signal_date not in local.index:
        lookup_misses += 1
        continue

    row = local.loc[signal_date]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[-1]

    atr = pd.to_numeric(pd.Series([row.get("ATR14", np.nan)]), errors="coerce").iloc[0]
    recent_low = pd.to_numeric(pd.Series([row.get("RECENT_LOW_5", np.nan)]), errors="coerce").iloc[0]
    signal_close = pd.to_numeric(pd.Series([row.get("Close", np.nan)]), errors="coerce").iloc[0]

    if not all(np.isfinite(float(v)) for v in [atr, recent_low, signal_close]):
        invalid_rows += 1
        continue

    surface_rows.append({
        **rec.to_dict(),
        "SignalDate": signal_date,
        "EntryDate": rec["EntryDate"],
        "ExitDate": thai_trading_day(rec["ExitDate"]),
        "EntryPrice": float(rec["EntryPrice"]),
        "Score": int(rec["Score"]),
        "Symbol": symbol,
        "SignalClose": float(signal_close),
        "ATR14": float(atr),
        "RecentLow5": float(recent_low),
    })

surface_model_df = pd.DataFrame(surface_rows)
print(f"Signal-date lookup misses: {lookup_misses:,}")
print(f"Signal-date invalid stop-input rows: {invalid_rows:,}")
print(f"Symbols missing from canonical_data: {len(missing_symbols):,}")
print(f"Valid causal signal-date rows: {len(surface_model_df):,}")

assert not surface_model_df.empty, "No valid causal signal-date stop inputs."
assert len(surface_model_df) >= int(len(model_df) * 0.95), (
    "Too many causal stop-input rows were lost during date/symbol alignment."
)

STOP_GRID = [0.80, 1.00, 1.20, 1.50, 1.80, 2.00]
TARGET_GRID = [0.10, 0.15, 0.20]
HOLD_GRID = [10, 20, 30]
MIN_VALIDATION_TRADES = 25

def run_stop_exit_policy(candidates, atr_mult, target2_pct, max_hold_days):
    if candidates.empty:
        return pd.DataFrame(), float(GLOBAL_INITIAL_CAPITAL)

    cash = float(GLOBAL_INITIAL_CAPITAL)
    open_positions, closed = [], []

    candidates = candidates.copy()
    candidates["SignalDate"] = candidates["SignalDate"].map(thai_trading_day)
    candidates["EntryDate"] = candidates["EntryDate"].map(thai_trading_day)

    by_date = {
        d: g.sort_values(["Score", "Symbol"], ascending=[False, True])
        for d, g in candidates.groupby("EntryDate")
    }
    event_dates = sorted(set(candidates["EntryDate"]))

    for current_date in event_dates:
        current_date = pd.Timestamp(current_date)

        still = []
        for pos in open_positions:
            if pd.Timestamp(pos["ExitDate"]) <= current_date:
                raw_exit = float(pos["ExitPrice"])
                shocked = raw_exit * (1 - RESEARCH_GAP) if pos["ExitReason"] == "STOP_GAP" else raw_exit
                net_exit = shocked * (1 - RESEARCH_SLIPPAGE) * (1 - RESEARCH_COST)
                proceeds = pos["Shares"] * net_exit
                cash += proceeds
                closed.append({
                    **pos,
                    "NetExitPrice": net_exit,
                    "Proceeds": proceeds,
                    "PnL": proceeds - pos["EntryCash"],
                })
            else:
                still.append(pos)
        open_positions = still

        for _, sig in by_date.get(current_date, pd.DataFrame()).iterrows():
            if any(p["Symbol"] == sig["Symbol"] for p in open_positions):
                continue
            if len(open_positions) >= GLOBAL_MAX_POSITIONS:
                continue

            signal_close = float(sig["SignalClose"])
            atr = float(sig["ATR14"])
            recent_low = float(sig["RecentLow5"])
            entry_price = float(sig["EntryPrice"])

            stop_price = calculate_stop_price(
                entry_price=signal_close,
                atr=atr,
                recent_low=recent_low,
                atr_multiplier=float(atr_mult),
            )
            if not np.isfinite(stop_price) or stop_price <= 0 or entry_price <= stop_price:
                continue

            size = calculate_position_size(
                cash,
                entry_price,
                stop_price,
                risk_pct=GLOBAL_RISK_PCT,
                lot_size=LOT_SIZE,
            )
            if size["shares"] <= 0:
                continue

            net_entry = entry_price * (1 + RESEARCH_SLIPPAGE) * (1 + RESEARCH_COST)
            entry_cash = int(size["shares"]) * net_entry
            if entry_cash > cash + 1e-9:
                continue

            df = feature_data.get(sig["Symbol"])
            if df is None or df.empty:
                continue
            df = thai_trading_frame(df)

            if current_date not in df.index:
                continue

            future = df.loc[df.index >= current_date].copy()
            max_date = current_date + pd.Timedelta(days=int(max_hold_days))
            future = future.loc[future.index <= max_date]
            if future.empty:
                continue

            target2 = entry_price * (1 + float(target2_pct))
            exit_date = pd.NaT
            exit_price = np.nan
            exit_reason = None

            for d, bar in future.iterrows():
                op = float(bar["Open"])
                hi = float(bar["High"])
                lo = float(bar["Low"])
                cl = float(bar["Close"])

                if op <= stop_price:
                    exit_date, exit_price, exit_reason = d, op, "STOP_GAP"
                    break
                if lo <= stop_price:
                    exit_date, exit_price, exit_reason = d, stop_price, "STOP"
                    break
                if op >= target2:
                    exit_date, exit_price, exit_reason = d, op, "TARGET2_GAP"
                    break
                if hi >= target2:
                    exit_date, exit_price, exit_reason = d, target2, "TARGET2"
                    break
                if d >= max_date:
                    exit_date, exit_price, exit_reason = d, cl, "MAX_HOLD"
                    break

            if pd.isna(exit_date):
                exit_date = future.index[-1]
                exit_price = float(future.iloc[-1]["Close"])
                exit_reason = "EOD"

            open_positions.append({
                "Symbol": sig["Symbol"],
                "SignalDate": pd.Timestamp(sig["SignalDate"]),
                "EntryDate": current_date,
                "EntryPrice": entry_price,
                "StopPrice": stop_price,
                "Shares": int(size["shares"]),
                "Score": int(sig["Score"]),
                "ExitDate": pd.Timestamp(exit_date),
                "ExitPrice": float(exit_price),
                "ExitReason": exit_reason,
                "EntryCash": float(entry_cash),
                "HoldingDays": int((pd.Timestamp(exit_date) - current_date).days),
                "ATRMul": float(atr_mult),
                "Target2Pct": float(target2_pct),
                "MaxHoldDays": int(max_hold_days),
            })
            cash -= entry_cash

    for pos in open_positions:
        raw_exit = float(pos["ExitPrice"])
        shocked = raw_exit * (1 - RESEARCH_GAP) if pos["ExitReason"] == "STOP_GAP" else raw_exit
        net_exit = shocked * (1 - RESEARCH_SLIPPAGE) * (1 - RESEARCH_COST)
        proceeds = pos["Shares"] * net_exit
        cash += proceeds
        closed.append({
            **pos,
            "NetExitPrice": net_exit,
            "Proceeds": proceeds,
            "PnL": proceeds - pos["EntryCash"],
        })

    return pd.DataFrame(closed), float(cash)

surface_specs = [
    {"ATRMul": a, "Target2Pct": t, "MaxHoldDays": h}
    for a in STOP_GRID for t in TARGET_GRID for h in HOLD_GRID
]
print(f"Candidate stop/exit policies: {len(surface_specs)}")
print(f"Signal-date rows with ATR/RecentLow5: {len(surface_model_df):,}")

dates = np.array(sorted(surface_model_df["SignalDate"].map(thai_trading_day).unique()))
blocks = np.array_split(dates, 7)
fold_specs = [(0, 2, 3, 4), (0, 3, 4, 5), (0, 4, 5, 6)]

fold_results = []
selected_specs = []

for fold_no, (tr0, tr1, val_block, test_block) in enumerate(fold_specs, 1):
    train_dates = np.concatenate(blocks[tr0:tr1 + 1])
    val_dates = blocks[val_block]
    test_dates = blocks[test_block]

    train = surface_model_df[surface_model_df["SignalDate"].isin(train_dates)].copy()
    val_df = surface_model_df[surface_model_df["SignalDate"].isin(val_dates)].copy()
    test_df = surface_model_df[surface_model_df["SignalDate"].isin(test_dates)].copy()

    assert not train.empty and not val_df.empty and not test_df.empty
    assert train["SignalDate"].max() < val_df["SignalDate"].min()
    assert val_df["SignalDate"].max() < test_df["SignalDate"].min()

    scores = []
    for spec in surface_specs:
        vt, _ = run_stop_exit_policy(
            val_df,
            spec["ATRMul"],
            spec["Target2Pct"],
            spec["MaxHoldDays"],
        )
        vm = portfolio_metrics(vt)
        scores.append({
            **spec,
            "ValPnL": vm["PnL"],
            "ValPF": vm["PF"],
            "ValTrades": vm["Trades"],
        })

    score_table = pd.DataFrame(scores)
    eligible = score_table[score_table["ValTrades"] >= MIN_VALIDATION_TRADES].copy()
    profitable = eligible[eligible["ValPF"] > 1.0].copy()
    pool = profitable if not profitable.empty else eligible

    if pool.empty:
        selected = {
            "ATRMul": PRODUCTION_ATR_MULT,
            "Target2Pct": PRODUCTION_TARGET2_PCT,
            "MaxHoldDays": PRODUCTION_MAX_HOLDING_DAYS,
            "ValPnL": np.nan,
            "ValPF": np.nan,
            "ValTrades": 0,
        }
    else:
        selected = pool.sort_values(
            ["ValPnL", "ValPF", "ValTrades"],
            ascending=[False, False, False],
        ).iloc[0].to_dict()

    selected_spec = {
        "ATRMul": float(selected["ATRMul"]),
        "Target2Pct": float(selected["Target2Pct"]),
        "MaxHoldDays": int(selected["MaxHoldDays"]),
    }
    selected_specs.append({"Fold": fold_no, **selected_spec})

    base_test, _ = run_global_research(test_df)
    base_m = portfolio_metrics(base_test)

    selected_test, _ = run_stop_exit_policy(
        test_df,
        selected_spec["ATRMul"],
        selected_spec["Target2Pct"],
        selected_spec["MaxHoldDays"],
    )
    selected_m = portfolio_metrics(selected_test)

    fold_results.append({
        "Fold": fold_no,
        "TrainRows": len(train),
        "ValRows": len(val_df),
        "TestRows": len(test_df),
        "SelectedATRMul": selected_spec["ATRMul"],
        "SelectedTarget2Pct": selected_spec["Target2Pct"],
        "SelectedMaxHoldDays": selected_spec["MaxHoldDays"],
        "ValPnL": float(selected["ValPnL"]) if pd.notna(selected["ValPnL"]) else np.nan,
        "ValPF": float(selected["ValPF"]) if pd.notna(selected["ValPF"]) else np.nan,
        "ValTrades": int(selected["ValTrades"]),
        "BasePnL": base_m["PnL"],
        "SelectedPnL": selected_m["PnL"],
        "DeltaPnL": selected_m["PnL"] - base_m["PnL"],
        "BasePF": base_m["PF"],
        "SelectedPF": selected_m["PF"],
        "SelectedTrades": selected_m["Trades"],
        "PositiveDelta": selected_m["PnL"] > base_m["PnL"],
        "PFgt1": selected_m["PF"] > 1.0,
    })

research_stop_exit = pd.DataFrame(fold_results)
display(research_stop_exit)

passed = int((research_stop_exit["PositiveDelta"] & research_stop_exit["PFgt1"]).sum())
print(f"Promotion gate: {passed}/3 folds satisfy BOTH improvement and PF > 1.")
print("STOP_EXIT_SURFACE_GATE_PASS" if passed >= 2 else "STOP_EXIT_SURFACE_GATE_FAIL")

print("\nSELECTED STOP / EXIT POLICIES")
display(pd.DataFrame(selected_specs))

print("\nRESEARCH COST MODEL")
print(f"Cost/side={RESEARCH_COST:.4f}, Slippage/side={RESEARCH_SLIPPAGE:.4f}, GapShock={RESEARCH_GAP:.4f}")
print("\nPRODUCTION_UNCHANGED")
print("CELL 13.5 PASSED")



🧪 CAUSAL STOP / EXIT SURFACE NESTED OOS


Signal-date lookup misses: 0
Signal-date invalid stop-input rows: 0
Symbols missing from canonical_data: 0
Valid causal signal-date rows: 8,185
Candidate stop/exit policies: 54
Signal-date rows with ATR/RecentLow5: 8,185


,Fold,TrainRows,ValRows,TestRows,SelectedATRMul,SelectedTarget2Pct,SelectedMaxHoldDays,ValPnL,ValPF,ValTrades,BasePnL,SelectedPnL,DeltaPnL,BasePF,SelectedPF,SelectedTrades,PositiveDelta,PFgt1
0,1,3204,1112,788,0.8,0.2,30,-16951.854170,0.616423,79,-35584.089242,-25280.844663,10303.244579,0.384778,0.460394,95,True,False
1,2,4316,788,1253,1.8,0.2,30,-16677.318423,0.638381,85,-10359.139806,-11889.770057,-1530.630251,0.709959,0.611194,66,False,False
2,3,5104,1253,1828,0.8,0.2,30,8085.784931,1.357870,70,-6916.194752,-3429.945093,3486.249659,0.828951,0.871627,69,True,False


Promotion gate: 0/3 folds satisfy BOTH improvement and PF > 1.
STOP_EXIT_SURFACE_GATE_FAIL

SELECTED STOP / EXIT POLICIES


,Fold,ATRMul,Target2Pct,MaxHoldDays
0,1,0.8,0.2,30
1,2,1.8,0.2,30
2,3,0.8,0.2,30



RESEARCH COST MODEL
Cost/side=0.0015, Slippage/side=0.0010, GapShock=0.0025

PRODUCTION_UNCHANGED
CELL 13.5 PASSED
